# DaT Parkinson's Challenge — run 8: ensemble-corrected members (nested-OOF soft targets, mistake weights, extra seeds) + Master v3 gate
Hemisphere members trained on soft targets y* = (1-0.4)·y + 0.4·p_nested (Master v2's nested out-of-fold probability, embedded below), a mistake-weighted variant (w = 1 + 2.0·nested loss), two extra plain hemisphere seeds, crop-hist on soft targets, full-data soft hemispheres; then the run-7 super-stack over every arm family found in the checkpoint dataset and the gate table for Master v3.
Inputs: raw competition dataset (private), `dat-parkinsons-cache-v4`, `dat-run3-checkpoints` (resume + upload every 4 jobs via the KAGGLE_USERNAME/KAGGLE_KEY secrets).

In [ ]:
!mkdir -p datsc scripts artifacts_v3 out
import os, torch, subprocess
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count(), 'cpus', os.cpu_count())
RAW = '/kaggle/input/datasets/harshdipsaha/dat-parkinsons-challenge-data'; CACHE_DS = '/kaggle/input/datasets/harshdipsaha/dat-parkinsons-cache-v4'
print('raw:', os.listdir(RAW) if os.path.isdir(RAW) else 'MISSING')
print('cache ds:', os.listdir(CACHE_DS) if os.path.isdir(CACHE_DS) else 'MISSING (will build cache in-session)')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile datsc/__init__.py
"""DaT-SPECT classifier: shared preprocessing, models, calibration, inference."""
__version__ = "0.1.0"


In [ ]:
%%writefile datsc/augment.py
"""Augmentations applied to [B,1,X,Y,Z] tensors on the training device (GPU-side on Kaggle).

Physically motivated for multi-centre SPECT: blur (resolution differences), noise (counts),
quantisation (the ultra-low-count regime), reference-scale jitter (p75 normalisation error),
small affine (positioning), L/R mirror (the label is symmetric under mirroring).
"""
from __future__ import annotations

import math

import torch
import torch.nn.functional as F

from .config import SPACING_MM


def _gauss1d(sigma: float, device):
    r = max(1, int(math.ceil(3 * sigma)))
    t = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    k = torch.exp(-0.5 * (t / max(sigma, 1e-3)) ** 2)
    return k / k.sum()


def blur(x: torch.Tensor, sigma_vox: float) -> torch.Tensor:
    if sigma_vox < 0.05:
        return x
    k = _gauss1d(sigma_vox, x.device)
    n = k.numel(); pad = n // 2
    for dim in (2, 3, 4):
        shape = [1, 1, 1, 1, 1]; shape[dim] = n
        kk = k.view(*shape)
        p = [0] * 6; p[2 * (4 - dim)] = pad; p[2 * (4 - dim) + 1] = pad
        x = F.conv3d(F.pad(x, p, mode="replicate"), kk)
    return x


def random_blur(x, max_sigma_mm: float = 4.0):
    out = torch.empty_like(x)
    for i in range(x.shape[0]):
        s = float(torch.rand(1)) * max_sigma_mm / SPACING_MM
        out[i:i + 1] = blur(x[i:i + 1], s)
    return out


def random_noise(x, max_frac: float = 0.15):
    std = x.flatten(1).std(dim=1).view(-1, 1, 1, 1, 1)
    frac = torch.rand(x.shape[0], 1, 1, 1, 1, device=x.device) * max_frac
    return (x + torch.randn_like(x) * std * frac).clamp_min(0)


def quantize(x, k: int):
    m = x.flatten(1).amax(dim=1).view(-1, 1, 1, 1, 1).clamp_min(1e-6)
    return torch.round(x / m * (k - 1)) / (k - 1) * m


def random_quantize(x, levels=(4, 6, 8, 16, 64, 256), p: float = 0.5):
    out = x.clone()
    for i in range(x.shape[0]):
        if float(torch.rand(1)) < p:
            out[i:i + 1] = quantize(x[i:i + 1], int(levels[int(torch.randint(len(levels), (1,)))]))
    return out


def random_scale_jitter(x, frac: float = 0.2):
    s = 1.0 + (torch.rand(x.shape[0], 1, 1, 1, 1, device=x.device) * 2 - 1) * frac
    return x * s


def random_affine(x, max_deg: float = 10.0, max_scale: float = 0.10, max_shift_vox: float = 3.0):
    B = x.shape[0]
    theta = torch.zeros(B, 3, 4, device=x.device)
    for i in range(B):
        a = torch.deg2rad((torch.rand(3) * 2 - 1) * max_deg)
        cx, cy, cz = torch.cos(a); sx, sy, sz = torch.sin(a)
        Rx = torch.tensor([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
        Ry = torch.tensor([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
        Rz = torch.tensor([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
        S = torch.diag(1.0 + (torch.rand(3) * 2 - 1) * max_scale)
        M = Rz @ Ry @ Rx @ S
        shift = (torch.rand(3) * 2 - 1) * max_shift_vox * 2.0 / torch.tensor(x.shape[2:], dtype=torch.float32)
        theta[i, :, :3] = M.to(x.device); theta[i, :, 3] = shift.to(x.device)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


def random_flip(x, p: float = 0.5):
    out = x.clone()
    for i in range(x.shape[0]):
        if float(torch.rand(1)) < p:
            out[i] = x[i].flip(dims=[1])   # dim 1 of [1,X,Y,Z] is X
    return out


PRESETS = {
    # run-3 defaults (physically plausible multi-centre perturbations)
    "base":   dict(blur_mm=4.0, noise=0.15, deg=10.0, scale=0.10, shift=3.0, quant_p=0.5, scale_jitter=0.2),
    # deliberately unrealistic (Buddenkotte & Buchert 2024): pushes the net to ignore the camera's resolution/count statistics
    "strong": dict(blur_mm=8.0, noise=0.30, deg=15.0, scale=0.15, shift=4.0, quant_p=0.6, scale_jitter=0.3),
}


def train_augment(x: torch.Tensor, strength: str = "base") -> torch.Tensor:
    c = PRESETS[strength]
    x = random_flip(x)
    x = random_affine(x, c["deg"], c["scale"], c["shift"])
    x = random_blur(x, c["blur_mm"])
    x = random_scale_jitter(x, c["scale_jitter"])
    x = random_quantize(x, p=c["quant_p"])
    x = random_noise(x, c["noise"])
    return x


In [ ]:
%%writefile datsc/calibrate.py
"""Exactly three fitted numbers: blend weight w, temperature T, prior-mix lambda (plus the prior pi)."""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
from scipy.optimize import minimize_scalar

EPS = 1e-4


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def logloss(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def fit_blend_weight(p_cnn, p_lr, y, grid=np.linspace(0, 1, 21)):
    return float(min(grid, key=lambda w: logloss(y, w * p_cnn + (1 - w) * p_lr)))


def fit_temperature(z, y):
    res = minimize_scalar(lambda T: logloss(y, sigmoid(z / T)), bounds=(0.25, 10.0), method="bounded")
    return float(res.x)


@dataclass(frozen=True)
class Calibration:
    w: float
    T: float
    lam: float
    pi: float

    def apply(self, p_cnn, p_lr):
        p_cnn = np.asarray(p_cnn, float); p_lr = np.asarray(p_lr, float)
        blend = self.w * p_cnn + (1 - self.w) * p_lr
        cal = sigmoid(logit(blend) / self.T)
        return np.clip((1 - self.lam) * cal + self.lam * self.pi, EPS, 1 - EPS)

    def save(self, path):
        Path(path).write_text(json.dumps(asdict(self)))

    @classmethod
    def load(cls, path):
        return cls(**json.loads(Path(path).read_text()))


def fit(p_cnn, p_lr, y, lam: float, pi: float) -> Calibration:
    w = fit_blend_weight(p_cnn, p_lr, y)
    T = fit_temperature(logit(w * p_cnn + (1 - w) * p_lr), y)
    return Calibration(w=w, T=T, lam=lam, pi=pi)


In [ ]:
%%writefile datsc/config.py
"""Constants and data-location discovery shared by training and inference."""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

# --- geometry -------------------------------------------------------------
SPACING_MM = 3.0                   # isotropic resampling target
CROP_SHAPE = (72, 72, 48)          # [x, y, z] voxels at SPACING_MM (216x216x144 mm)
SMOOTH_MM = 6.0                    # gaussian sigma used for mask + anchor
ANCHOR_THRESH = 0.7                # fraction of (interior) smoothed max defining the hotspot
ANCHOR_INTERIOR_MM = 20.0          # hotspot search restricted to voxels this deep inside the head mask
ANCHOR_MIN_ML, ANCHOR_MAX_ML = 1.0, 30.0   # informational flag bounds (anchor is used regardless)
ANCHOR_MAX_DIST_MM = 50.0
FALLBACK_SUPERIOR_OFFSET_MM = 15.0 # CoM fallback only when the interior region is empty (degenerate mask)
STRIATAL_HALF_MM = (15.0, 15.0, 9.0)  # half-extent of the striatal box for radiomics

# --- scalars (order is a contract between preprocess, LR, CNN head) --------
SCALAR_NAMES = (
    "log_p99_over_p75", "mask_ml", "masked_skew", "mirror_ncc",
    "z_extent_mm", "native_spacing_mm", "log_n_levels", "anchor_fallback",
)
N_SCALARS = len(SCALAR_NAMES)

# --- runtime paths (DrivenData container) ----------------------------------
RUNTIME_DATA_ROOT = Path(os.environ.get("DATSC_DATA_ROOT", "/code_execution/data"))
RUNTIME_NIFTI_DIR = RUNTIME_DATA_ROOT / "niftis"
RUNTIME_SUBMISSION_FORMAT = RUNTIME_DATA_ROOT / "submission_format.csv"
RUNTIME_OUTPUT_CSV = Path(os.environ.get("DATSC_OUTPUT_CSV", "submission.csv"))

# --- training-time locations ---------------------------------------------
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/harshdipsaha/dat-parkinsons-challenge-data")
LOCAL_DATA_DIR = Path(__file__).resolve().parent.parent / "data"
DEFAULT_PRIOR = 747 / 1362         # training prevalence of is_pathologic == 1


@dataclass(frozen=True)
class DataDir:
    root: Path
    labels_csv: Path
    niftis_dir: Path


def _count_niftis(d: Path) -> int:
    return sum(1 for p in d.iterdir() if p.name.endswith((".nii", ".nii.gz"))) if d.is_dir() else 0


def find_data_dir() -> DataDir:
    """Locate labels csv + nifti folder. Order: $DATSC_DATA_DIR, Kaggle mount, ./data."""
    candidates = [Path(os.environ["DATSC_DATA_DIR"])] if "DATSC_DATA_DIR" in os.environ else []
    candidates += [KAGGLE_DATA_DIR, LOCAL_DATA_DIR]
    for root in candidates:
        if not root.is_dir():
            continue
        csvs = sorted(root.glob("train_labels*.csv"))
        if not csvs:
            continue
        best, best_n = None, 0
        for d in [root] + [p for p in root.iterdir() if p.is_dir()]:
            n = _count_niftis(d)
            if n > best_n:
                best, best_n = d, n
        if best is None:
            continue
        return DataDir(root=root, labels_csv=csvs[0], niftis_dir=best)
    raise FileNotFoundError("No data dir found; set DATSC_DATA_DIR")


def find_nifti_path(niftis_dir: Path, uid: str) -> Path | None:
    for ext in (".nii.gz", ".nii"):
        p = niftis_dir / f"{uid}{ext}"
        if p.exists():
            return p
    return None


In [ ]:
%%writefile datsc/data_b.py
"""Run-6b inputs: sagittal slab stacks and the tri-planar view tensor.

The registered box is [B,1,X,Y,Z] (X = left/right, Y = posterior->anterior, Z = inferior->superior, 2 mm). The existing
`to_slabs` yields axial (z) and coronal (y) stacks whose last image axis is X, so an L/R mirror is a flip on that axis.
Sagittal stacks are taken along X around each hemisphere's striatal centroid; their in-plane axes are (Z rows, Y cols), so
the L/R mirror of the volume maps the left stack onto the right stack and vice versa (a swap, not an in-plane flip).
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F

from .volume import VOL_SPACING_MM


def to_slabs_sagittal(x: torch.Tensor, n_slabs: int = 3, slab_mm: float = 8.0, hw: int = 160, x_center: float = 36.8,
                      spacing_mm: float = VOL_SPACING_MM, mode: str = "mean", crop: int | None = None) -> torch.Tensor:
    """[B,1,X,Y,Z] -> [B,n_slabs,hw,hw]: consecutive slabs along X centred on voxel `x_center`; image rows = Z, cols = Y."""
    B, _, X, Y, Z = x.shape
    t = max(1, int(round(slab_mm / spacing_mm)))
    xc = int(round(x_center)); x0 = xc - (n_slabs * t) // 2
    chans = []
    for k in range(n_slabs):
        a, b = max(0, x0 + k * t), min(X, x0 + (k + 1) * t)
        sl = x[:, 0, a:b, :, :]                                        # [B, t, Y, Z]
        chans.append(sl.amax(dim=1) if mode == "max" else sl.mean(dim=1))
    img = torch.stack(chans, dim=1).transpose(2, 3)                    # [B, n, Z, Y]
    if crop:
        H, W = img.shape[-2:]; c0 = max(0, (H - crop) // 2); c1 = max(0, (W - crop) // 2)
        img = img[:, :, c0:c0 + crop, c1:c1 + crop]
    if hw != img.shape[-1] or hw != img.shape[-2]:
        img = F.interpolate(img, size=(hw, hw), mode="bilinear", align_corners=False)
    return img


def hemisphere_centres(striatal_mask: np.ndarray, mid_x: float) -> tuple[float, float]:
    """x centroids (voxels) of the left and right halves of the template striatal mask."""
    xs = np.arange(striatal_mask.shape[0])[:, None, None]
    L = np.argwhere(striatal_mask & (xs < mid_x)); R = np.argwhere(striatal_mask & (xs >= mid_x))
    if len(L) == 0 or len(R) == 0:
        return float(mid_x) - 11.0, float(mid_x) + 11.0
    return float(L[:, 0].mean()), float(R[:, 0].mean())


In [ ]:
%%writefile datsc/data_v3.py
"""Run-3 data access: memmap-backed whole-brain cache + GPU-side derivation of model inputs.

Cache layout (scripts/cache_volumes.py): vols.npy fp16 [N,96,96,72], masks.npy u8, scalars.npy [N,8],
stats.npy [N,15], meta.csv (row order == array order). Inputs derived on the training device:
  * 3D: trilinear resize of the 2 mm box to `size` (64x64x48 == 3 mm effective, 96x96x72 == 2 mm)
  * 2.5D: `n_slabs` axial mean-projections of `slab_mm` each, centred on the box centre (== head CoM),
    stacked as channels and bilinearly resized to `hw` pixels for an ImageNet-pretrained backbone.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from .volume import VOL_SHAPE, VOL_SPACING_MM, renormalise


class VolumeCache:
    def __init__(self, root):
        self.root = Path(root)
        self.vols = np.load(self.root / "vols.npy", mmap_mode="r")
        self.scalars = np.load(self.root / "scalars.npy")
        self.stats = np.load(self.root / "stats.npy")
        self.meta = pd.read_csv(self.root / "meta.csv", dtype={"uid": str})
        self.index = {u: i for i, u in enumerate(self.meta["uid"])}
        assert self.vols.shape[1:] == VOL_SHAPE, self.vols.shape

    def __len__(self):
        return len(self.meta)


class VolDataset(Dataset):
    """Yields (vol fp32 [1,X,Y,Z] in the requested intensity normalisation, scalars fp32 [8], y, uid)."""

    def __init__(self, cache: VolumeCache, uids, labels, norm: str = "p75", template=None):
        self.c, self.uids, self.norm, self.template = cache, list(uids), norm, template
        self.labels = np.asarray(labels, np.float32)

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        k = self.c.index[self.uids[i]]
        v = renormalise(np.asarray(self.c.vols[k]), self.c.stats[k], self.norm, self.template)   # "ref" == as stored in a registered cache
        return (torch.from_numpy(np.ascontiguousarray(v))[None], torch.from_numpy(self.c.scalars[k].copy()),
                torch.tensor(self.labels[i]), self.uids[i])


def load_site_ids(cache_dir, uids, min_n: int = 25):
    """{uid: pseudo-site id} from <cache>/sites.csv; sites with fewer than `min_n` of the given uids merge into id 0 ("other").
    Returns (mapping, n_sites); without a sites.csv every uid maps to 0 and n_sites == 1."""
    p = Path(cache_dir) / "sites.csv"
    uids = list(uids)
    if not p.exists():
        return {u: 0 for u in uids}, 1
    s = pd.read_csv(p, dtype={"uid": str}).set_index("uid")["site"].reindex(uids).fillna("other")
    counts = s.value_counts()
    keep = sorted(k for k in counts.index if counts[k] >= min_n and k != "other")
    ids = {k: i + 1 for i, k in enumerate(keep)}
    return {u: int(ids.get(v, 0)) for u, v in s.items()}, len(ids) + 1


def to_3d_input(x: torch.Tensor, size=(64, 64, 48)) -> torch.Tensor:
    """[B,1,96,96,72] -> [B,1,*size] (trilinear). size == VOL_SHAPE is a no-op."""
    if tuple(x.shape[2:]) == tuple(size):
        return x
    return F.interpolate(x, size=size, mode="trilinear", align_corners=False)


def to_3d_crop(x: torch.Tensor, crop=(48, 48, 32), center=None) -> torch.Tensor:
    """[B,1,X,Y,Z] -> [B,1,*crop]: axis-aligned crop centred on `center` (voxel coords; default box centre), clamped
    to the box. In the registered frame pass the template striatal centre: full 2 mm resolution on the striatum."""
    B, _, X, Y, Z = x.shape
    c = [(X - 1) / 2, (Y - 1) / 2, (Z - 1) / 2] if center is None else list(center)
    sl = []
    for n, k, cc in zip((X, Y, Z), crop, c):
        a = int(round(cc - k / 2 + 0.5)); a = max(0, min(a, n - k)); sl.append(slice(a, a + k))
    return x[:, :, sl[0], sl[1], sl[2]]


def to_slabs(x: torch.Tensor, n_slabs: int = 3, slab_mm: float = 12.0, hw: int = 192,
             spacing_mm: float = VOL_SPACING_MM, mode: str = "mean", z_center: float | None = None,
             plane: str = "axial", crop: int | None = None) -> torch.Tensor:
    """[B,1,X,Y,Z] -> [B,n_slabs,hw,hw]: consecutive slabs centred on voxel `z_center` along the slab axis
    (axial: z, default box centre; coronal: y). In the registered frame pass the template's striatal centre.
    Each channel is the mean (or max) over `slab_mm`. The image's last axis is always X (L/R), so a flip on
    dim 3 is the label-preserving mirror for both planes."""
    B, _, X, Y, Z = x.shape
    t = max(1, int(round(slab_mm / spacing_mm)))
    if plane == "coronal":
        x = x.transpose(3, 4)                              # [B,1,X,Z,Y]: slab axis (y) last, rows will be z
        Z = Y
    zc = Z // 2 if z_center is None else int(round(z_center))
    z0 = zc - (n_slabs * t) // 2
    chans = []
    for k in range(n_slabs):
        a, b = max(0, z0 + k * t), min(Z, z0 + (k + 1) * t)
        sl = x[:, 0, :, :, a:b]
        chans.append(sl.amax(dim=-1) if mode == "max" else sl.mean(dim=-1))
    img = torch.stack(chans, dim=1)                       # [B, n, X, rows]
    img = img.transpose(2, 3)                             # [B, n, rows, X]
    if crop:                                              # central crop (e.g. 72 voxels = 144 mm at 2 mm) before resizing
        H, W = img.shape[-2:]; c0 = max(0, (H - crop) // 2); c1 = max(0, (W - crop) // 2)
        img = img[:, :, c0:c0 + crop, c1:c1 + crop]
    if hw != img.shape[-1] or hw != img.shape[-2]:
        img = F.interpolate(img, size=(hw, hw), mode="bilinear", align_corners=False)
    return img


In [ ]:
%%writefile datsc/featmodel.py
"""PCA (striatal patch) + standardised hand-crafted features -> L2 logistic regression; numpy-only inference, npz on disk."""
from __future__ import annotations

from pathlib import Path

import numpy as np


class FeatModel:
    def __init__(self, n_comp: int = 40, C: float = 0.1, seed: int = 0):
        self.n_comp, self.C, self.seed = int(n_comp), float(C), int(seed)
        self.patch_mean = self.components = None          # PCA of the striatal patch
        self.mean = self.scale = self.coef = None           # standardiser + logistic weights over [features, pca scores]
        self.intercept = 0.0
        self.score_mean = self.score_cov_inv = None         # typicality: Mahalanobis statistics of the training PCA scores

    # ---- fitting (sklearn only here) ----
    def fit(self, F: np.ndarray, P: np.ndarray, y: np.ndarray):
        from sklearn.decomposition import PCA
        from sklearn.linear_model import LogisticRegression
        F, P = np.asarray(F, np.float64), np.asarray(P, np.float32)
        pca = PCA(n_components=self.n_comp, svd_solver="randomized", random_state=self.seed).fit(P)
        self.patch_mean, self.components = pca.mean_.astype(np.float32), pca.components_.astype(np.float32)
        sc = (P - self.patch_mean) @ self.components.T
        self.score_mean = sc.mean(0).astype(np.float64)
        cov = np.cov(sc.astype(np.float64), rowvar=False) + 1e-3 * np.eye(sc.shape[1])
        self.score_cov_inv = np.linalg.inv(cov)
        X = self._design(F, P)
        self.mean = X.mean(0); self.scale = X.std(0) + 1e-6
        clf = LogisticRegression(C=self.C, max_iter=5000).fit((X - self.mean) / self.scale, y)
        self.coef = clf.coef_.ravel(); self.intercept = float(clf.intercept_[0])
        return self

    def _design(self, F, P):
        scores = (np.asarray(P, np.float32) - self.patch_mean) @ self.components.T
        return np.concatenate([np.asarray(F, np.float64), scores.astype(np.float64)], axis=1)

    def decision(self, F, P):
        Z = (self._design(F, P) - self.mean) / self.scale
        return Z @ self.coef + self.intercept

    def predict_proba(self, F, P):
        return 1.0 / (1.0 + np.exp(-self.decision(F, P)))

    def typicality(self, P):
        """Mahalanobis distance of each scan's PCA scores to the training distribution, divided by sqrt(n_comp)
        (so a typical training scan is ~1). Training statistics only: per-sample, never transductive."""
        if self.score_mean is None:
            return np.full(len(P), np.nan)
        sc = (np.asarray(P, np.float32) - self.patch_mean) @ self.components.T - self.score_mean
        d2 = np.einsum("ij,jk,ik->i", sc, self.score_cov_inv, sc)
        return np.sqrt(np.maximum(d2, 0) / sc.shape[1])

    # ---- persistence ----
    def save(self, path):
        np.savez_compressed(path, n_comp=self.n_comp, C=self.C, seed=self.seed, patch_mean=self.patch_mean, components=self.components,
                            mean=self.mean, scale=self.scale, coef=self.coef, intercept=self.intercept,
                            **({} if self.score_mean is None else {"score_mean": self.score_mean, "score_cov_inv": self.score_cov_inv}))

    @classmethod
    def load(cls, path):
        d = np.load(Path(path))
        m = cls(int(d["n_comp"]), float(d["C"]), int(d["seed"]))
        m.patch_mean, m.components = d["patch_mean"], d["components"]
        m.mean, m.scale, m.coef, m.intercept = d["mean"], d["scale"], d["coef"], float(d["intercept"])
        if "score_mean" in d.files:
            m.score_mean, m.score_cov_inv = d["score_mean"], d["score_cov_inv"]
        return m


In [ ]:
%%writefile datsc/featmodel_c.py
"""Approach C feature model: standardised descriptors + striatal-patch PCA scores -> a logistic on [linear features + random
Fourier features] (RBF-kernel logistic with a linear pass-through) or a small MLP, both serialised to npz and evaluated with numpy only (no pickle in the zip).
Keeps the typicality (Mahalanobis) statistics of the training PCA scores like the run-4 feature model."""
from __future__ import annotations

from pathlib import Path

import numpy as np


class FeatModelC:
    def __init__(self, kind: str = "rff", n_comp: int = 40, C: float = 0.3, gamma: float = 0.5, n_rff: int = 512,
                 hidden=(64, 32), alpha: float = 1e-2, seed: int = 0):
        assert kind in ("rff", "mlp", "linear")
        self.kind, self.n_comp, self.C, self.gamma, self.n_rff = kind, int(n_comp), float(C), float(gamma), int(n_rff)
        self.hidden, self.alpha, self.seed = tuple(int(h) for h in hidden), float(alpha), int(seed)
        self.patch_mean = self.components = None            # PCA of the striatal patch
        self.mean = self.scale = None                        # standardiser over [features, pca scores]
        self.W = self.b = None                               # rff projection
        self.coef = None; self.intercept = 0.0               # logistic weights (rff / linear)
        self.layers = None                                   # mlp: list of (W, b)
        self.score_mean = self.score_cov_inv = None          # typicality statistics

    # ---- fitting (sklearn only here) ----
    def _pca(self, P):
        from sklearn.decomposition import PCA
        pca = PCA(n_components=self.n_comp, svd_solver="randomized", random_state=self.seed).fit(P)
        self.patch_mean, self.components = pca.mean_.astype(np.float32), pca.components_.astype(np.float32)
        sc = (P - self.patch_mean) @ self.components.T
        self.score_mean = sc.mean(0).astype(np.float64)
        self.score_cov_inv = np.linalg.inv(np.cov(sc.astype(np.float64), rowvar=False) + 1e-3 * np.eye(sc.shape[1]))

    def _design(self, F, P):
        sc = (np.asarray(P, np.float32) - self.patch_mean) @ self.components.T
        return np.concatenate([np.asarray(F, np.float64), sc.astype(np.float64)], axis=1)

    def _phi(self, Z):
        if self.kind == "rff":                            # linear pass-through + random Fourier features (RBF kernel approx.)
            return np.concatenate([Z, np.sqrt(2.0 / self.n_rff) * np.cos(Z @ self.W + self.b)], axis=1)
        return Z

    def fit(self, F: np.ndarray, P: np.ndarray, y: np.ndarray):
        from sklearn.linear_model import LogisticRegression
        F, P = np.asarray(F, np.float64), np.asarray(P, np.float32)
        self._pca(P)
        X = self._design(F, P)
        self.mean = X.mean(0); self.scale = X.std(0) + 1e-6
        Z = (X - self.mean) / self.scale
        rng = np.random.default_rng(self.seed)
        if self.kind == "rff":
            self.W = (rng.normal(0, 1, (Z.shape[1], self.n_rff)) * np.sqrt(2 * self.gamma / Z.shape[1])).astype(np.float64)
            self.b = rng.uniform(0, 2 * np.pi, self.n_rff)
        if self.kind in ("rff", "linear"):
            clf = LogisticRegression(C=self.C, max_iter=5000).fit(self._phi(Z), y)
            self.coef = clf.coef_.ravel(); self.intercept = float(clf.intercept_[0])
        else:
            from sklearn.neural_network import MLPClassifier
            clf = MLPClassifier(hidden_layer_sizes=self.hidden, activation="tanh", alpha=self.alpha, max_iter=600,
                                random_state=self.seed, early_stopping=False).fit(Z, y)
            self.layers = [(w.astype(np.float64), b.astype(np.float64)) for w, b in zip(clf.coefs_, clf.intercepts_)]
        return self

    # ---- numpy inference ----
    def decision(self, F, P):
        Z = (self._design(F, P) - self.mean) / self.scale
        if self.kind == "mlp":
            h = Z
            for k, (w, b) in enumerate(self.layers):
                h = h @ w + b
                if k < len(self.layers) - 1:
                    h = np.tanh(h)
            return h.ravel()
        return self._phi(Z) @ self.coef + self.intercept

    def predict_proba(self, F, P):
        return 1.0 / (1.0 + np.exp(-self.decision(F, P)))

    def typicality(self, P):
        sc = (np.asarray(P, np.float32) - self.patch_mean) @ self.components.T - self.score_mean
        d2 = np.einsum("ij,jk,ik->i", sc, self.score_cov_inv, sc)
        return np.sqrt(np.maximum(d2, 0) / sc.shape[1])

    # ---- persistence ----
    def save(self, path):
        d = dict(kind=np.array(self.kind), n_comp=self.n_comp, C=self.C, gamma=self.gamma, n_rff=self.n_rff, hidden=np.array(self.hidden), alpha=self.alpha, seed=self.seed,
                 patch_mean=self.patch_mean, components=self.components, mean=self.mean, scale=self.scale,
                 score_mean=self.score_mean, score_cov_inv=self.score_cov_inv)
        if self.kind == "mlp":
            for k, (w, b) in enumerate(self.layers):
                d[f"W{k}"] = w; d[f"b{k}"] = b
            d["n_layers"] = len(self.layers)
        else:
            d["coef"] = self.coef; d["intercept"] = self.intercept
            if self.kind == "rff":
                d["W"] = self.W; d["b"] = self.b
        np.savez_compressed(path, **d)

    @classmethod
    def load(cls, path):
        d = np.load(Path(path))
        m = cls(str(d["kind"]), int(d["n_comp"]), float(d["C"]), float(d["gamma"]), int(d["n_rff"]), tuple(int(h) for h in d["hidden"]), float(d["alpha"]), int(d["seed"]))
        m.patch_mean, m.components, m.mean, m.scale = d["patch_mean"], d["components"], d["mean"], d["scale"]
        m.score_mean, m.score_cov_inv = d["score_mean"], d["score_cov_inv"]
        if m.kind == "mlp":
            m.layers = [(d[f"W{k}"], d[f"b{k}"]) for k in range(int(d["n_layers"]))]
        else:
            m.coef, m.intercept = d["coef"], float(d["intercept"])
            if m.kind == "rff":
                m.W, m.b = d["W"], d["b"]
        return m


In [ ]:
%%writefile datsc/features.py
"""Hand-crafted striatal features computed in the anchored crop frame (LR arm + alignment gate)."""
from __future__ import annotations

import numpy as np

from .config import CROP_SHAPE, SPACING_MM, STRIATAL_HALF_MM

RADIOMICS_NAMES = (
    "left_mean", "right_mean", "left_max", "right_max", "left_p90", "right_p90",
    "asym_mean", "asym_p90", "min_side_mean", "min_side_p90", "max_side_p90",
    "bg_ring_mean", "striatal_to_bg", "hot_frac_gt2",
)
N_RADIOMICS = len(RADIOMICS_NAMES)


def _boxes(shape=CROP_SHAPE, spacing_mm=SPACING_MM):
    cx, cy, cz = [s // 2 for s in shape]
    hx, hy, hz = [int(round(h / spacing_mm)) for h in STRIATAL_HALF_MM]
    left = (slice(cx - hx, cx), slice(cy - hy, cy + hy), slice(cz - hz, cz + hz))
    right = (slice(cx, cx + hx), slice(cy - hy, cy + hy), slice(cz - hz, cz + hz))
    return left, right, (cx, cy, cz), (hx, hy, hz)


def _ring_mask(shape, center, inner_vox, outer_vox):
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing="ij")
    d = np.sqrt((x - center[0]) ** 2 + (y - center[1]) ** 2)
    ring2d = (d >= inner_vox) & (d < outer_vox)
    m = np.zeros(shape, bool)
    z0, z1 = max(center[2] - 3, 0), min(center[2] + 3, shape[2])
    m[:, :, z0:z1] = ring2d[:, :, None]
    return m


def hotspot_radiomics(crop: np.ndarray) -> np.ndarray:
    """crop: [x,y,z] float32 in p75-normalised units, anchored so the striatum is at the centre."""
    c = crop.astype(np.float32)
    left, right, center, (hx, hy, hz) = _boxes(c.shape)
    L, R = c[left], c[right]
    lm, rm = float(L.mean()), float(R.mean())
    lx, rx = float(L.max()), float(R.max())
    lp, rp = float(np.percentile(L, 90)), float(np.percentile(R, 90))
    ring = _ring_mask(c.shape, center, inner_vox=max(hx, hy) + 3, outer_vox=max(hx, hy) + 9)
    bg = float(c[ring].mean()) if ring.any() else 1.0
    stri = 0.5 * (lp + rp)
    eps = 1e-6
    f = np.array([
        lm, rm, lx, rx, lp, rp,
        abs(lm - rm) / (lm + rm + eps),
        abs(lp - rp) / (lp + rp + eps),
        min(lm, rm), min(lp, rp), max(lp, rp),
        bg,
        stri / (bg + eps),
        float(((c[left] > 2.0).mean() + (c[right] > 2.0).mean()) / 2),
    ], dtype=np.float32)
    assert f.shape == (N_RADIOMICS,)
    return np.nan_to_num(f)


In [ ]:
%%writefile datsc/features_c.py
"""Approach C feature arm ("featc"): radiomic + shape descriptors of the registered striatum on top of the run-3/4 blocks.

Everything is computed in template space from the reference-normalised registered box (SBR+1 units), per hemisphere,
label-blind, with no acquisition/site scalars:

  * principal-axis profile: the striatal mask of each hemisphere is projected on its own geometric long axis (PCA of the
    mask voxel coordinates, oriented posterior -> anterior) and the mean SBR is taken in 6 bins along it (caudate head ..
    posterior putamen) with the slope; medial / lateral halves of the mask (caudate vs putamen proxy) and their ratio;
  * hot-spot shape: intensity-weighted second moments of the uptake above half of the hemisphere maximum inside a dilated
    ROI -> elongation, flatness, orientation of the major axis w.r.t. the anterior-posterior axis, centroid offsets;
  * radial profile: mean SBR in concentric shells (0-3-6-9-12-16 mm) around the hemisphere hot-spot centroid;
  * texture: grey-level co-occurrence (contrast, homogeneity, energy, correlation) of the 48x48x32 striatal box, averaged
    over the three axis offsets (numpy only, no scikit-image dependency);
  * asymmetry (|L-R| / (|L|+|R|)) of the profiles, moments and radial shells (asymmetric onset is the PD signature).

Radiomics of the striatum (texture + shape) has repeatedly matched or beaten semi-quantitative SBR indices for PD vs.
control on DAT-SPECT (EJNMMI Research 2022 radiomics signature; EJNMMI Physics 2024 radiomics + deep features).
Mirror orientation is a label-preserving augmentation: the model is fitted on both orientations and averaged at test time.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage as ndi

from .features_v3 import FEATURE_NAMES as V3_NAMES, _split_lr, striatal_features
from .features_v4 import CROP, ENG_NAMES, engineered_features, mirror, striatal_patch  # noqa: F401  (re-exported)

N_AXIS_BINS = 6
SHELLS_MM = (0.0, 3.0, 6.0, 9.0, 12.0, 16.0)
GLCM_LEVELS = 12
GLCM_RANGE = (0.5, 4.0)          # SBR+1 units covered by the grey levels (below/above are clipped)
GLCM_NAMES = ("contrast", "homogeneity", "energy", "correlation")
SPACING_MM = 2.0

_side = ("L", "R")
C_NAMES = tuple(
    [f"{s}_ax{b}" for s in _side for b in range(N_AXIS_BINS)] + [f"{s}_ax_slope" for s in _side]
    + [f"{s}_medial" for s in _side] + [f"{s}_lateral" for s in _side] + [f"{s}_med_lat_ratio" for s in _side]
    + [f"{s}_elong" for s in _side] + [f"{s}_flat" for s in _side] + [f"{s}_orient" for s in _side]
    + [f"{s}_mom_dy" for s in _side] + [f"{s}_mom_dz" for s in _side] + [f"{s}_mom_vol" for s in _side]
    + [f"{s}_shell{k}" for s in _side for k in range(len(SHELLS_MM) - 1)]
    + [f"glcm_{n}" for n in GLCM_NAMES]
    + [f"asym_ax{b}" for b in range(N_AXIS_BINS)] + ["asym_ax_slope", "asym_medial", "asym_lateral", "asym_elong", "asym_mom_vol"]
    + [f"asym_shell{k}" for k in range(len(SHELLS_MM) - 1)]
)
FEATURE_NAMES_C = tuple(V3_NAMES) + tuple(ENG_NAMES) + C_NAMES
N_FEATURES_C = len(FEATURE_NAMES_C)


def _asym(a: float, b: float) -> float:
    return abs(a - b) / max(abs(a) + abs(b), 1e-3)


def _axis_profile(v: np.ndarray, m: np.ndarray, n_bins: int = N_AXIS_BINS):
    """Mean SBR in n_bins along the mask's geometric long axis (oriented posterior -> anterior), plus the slope."""
    idx = np.argwhere(m)
    if len(idx) < 8:
        return np.zeros(n_bins, np.float32), 0.0
    c = idx.mean(0)
    cov = np.cov((idx - c).T)
    w, vec = np.linalg.eigh(cov)
    axis = vec[:, np.argmax(w)]
    if axis[1] < 0:                                    # y grows posterior -> anterior in this frame
        axis = -axis
    proj = (idx - c) @ axis
    edges = np.linspace(proj.min(), proj.max() + 1e-6, n_bins + 1)
    vals = v[m]
    out = []
    for b in range(n_bins):
        sel = (proj >= edges[b]) & (proj < edges[b + 1])
        out.append(float(vals[sel].mean()) if sel.any() else (out[-1] if out else 1.0))
    out = np.asarray(out, np.float32)
    slope = float(np.polyfit(np.arange(n_bins, dtype=np.float32), out, 1)[0])
    return out, slope


def _medial_lateral(v: np.ndarray, m: np.ndarray, mid_x: float):
    """Mean SBR of the medial and lateral halves of one hemisphere's striatal mask (caudate-ish vs putamen-ish)."""
    idx = np.argwhere(m)
    if len(idx) < 8:
        return 1.0, 1.0
    dist = np.abs(idx[:, 0] - mid_x)
    med = dist <= np.median(dist)
    vals = v[m]
    a, b = float(vals[med].mean()), float(vals[~med].mean()) if (~med).any() else float(vals.mean())
    return a, b


def _moments(v: np.ndarray, roi: np.ndarray, centre_yz):
    """Intensity-weighted second moments of the uptake above half the ROI maximum: elongation, flatness, orientation
    (|cos| of the major axis vs the A-P axis), centroid offset (y, z) from the template centre, and the hot volume."""
    vals = v[roi]
    if vals.size == 0:
        return (1.0, 1.0, 0.0, 0.0, 0.0, 0.0)
    thr = 1.0 + 0.5 * max(float(vals.max()) - 1.0, 1e-3)
    hot = roi & (v > thr)
    idx = np.argwhere(hot)
    if len(idx) < 6:
        return (1.0, 1.0, 0.0, 0.0, 0.0, float(len(idx)))
    wgt = np.clip(v[hot] - 1.0, 1e-3, None)
    c = (idx * wgt[:, None]).sum(0) / wgt.sum()
    d = idx - c
    cov = (d * wgt[:, None]).T @ d / wgt.sum()
    ev, vec = np.linalg.eigh(cov)
    ev = np.clip(ev, 1e-6, None)                      # ascending
    major = vec[:, 2]
    return (float(ev[2] / ev[1]), float(ev[0] / ev[2]), float(abs(major[1])), float(c[1] - centre_yz[0]), float(c[2] - centre_yz[1]), float(len(idx)))


def _radial(v: np.ndarray, roi: np.ndarray, brain: np.ndarray):
    """Mean SBR in concentric shells around the intensity-weighted centroid of the ROI's uptake."""
    vals = v[roi]
    if vals.size == 0:
        return np.ones(len(SHELLS_MM) - 1, np.float32)
    idx = np.argwhere(roi); wgt = np.clip(v[roi] - 1.0, 1e-3, None)
    c = (idx * wgt[:, None]).sum(0) / wgt.sum()
    zz, yy, xx = np.ogrid[:v.shape[2], :v.shape[1], :v.shape[0]]
    r = np.sqrt((xx - c[0]) ** 2 + (yy - c[1]) ** 2 + (zz - c[2]) ** 2).transpose(2, 1, 0) * SPACING_MM
    out = []
    for k in range(len(SHELLS_MM) - 1):
        sel = (r >= SHELLS_MM[k]) & (r < SHELLS_MM[k + 1]) & brain
        out.append(float(v[sel].mean()) if sel.any() else (out[-1] if out else 1.0))
    return np.asarray(out, np.float32)


def glcm_features(patch: np.ndarray, levels: int = GLCM_LEVELS, rng=GLCM_RANGE) -> np.ndarray:
    """Symmetric grey-level co-occurrence over the three unit axis offsets, averaged: contrast, homogeneity, energy, correlation."""
    q = np.clip((patch - rng[0]) / (rng[1] - rng[0]), 0, 1 - 1e-6)
    q = (q * levels).astype(np.int64)
    feats = np.zeros(4, np.float64)
    i_idx, j_idx = np.meshgrid(np.arange(levels), np.arange(levels), indexing="ij")
    for ax in range(3):
        a = np.moveaxis(q, ax, 0)
        p = np.zeros((levels, levels), np.float64)
        pairs = a[:-1].ravel() * levels + a[1:].ravel()
        np.add.at(p.ravel(), pairs, 1.0)
        p = p + p.T
        p /= max(p.sum(), 1.0)
        mu_i = (i_idx * p).sum(); mu_j = (j_idx * p).sum()
        sd_i = np.sqrt(((i_idx - mu_i) ** 2 * p).sum()); sd_j = np.sqrt(((j_idx - mu_j) ** 2 * p).sum())
        contrast = ((i_idx - j_idx) ** 2 * p).sum()
        homog = (p / (1 + (i_idx - j_idx) ** 2)).sum()
        energy = (p ** 2).sum()
        corr = (((i_idx - mu_i) * (j_idx - mu_j) * p).sum() / max(sd_i * sd_j, 1e-6))
        feats += np.array([contrast, homog, energy, corr])
    return (feats / 3).astype(np.float32)


def c_block(vol: np.ndarray, template) -> np.ndarray:
    v = vol.astype(np.float32)
    mid = float(template.striatal_center[0])
    L, R = _split_lr(template.striatal_mask, mid)
    if L.sum() == 0 or R.sum() == 0:
        L = R = template.striatal_mask
    cyz = (float(template.striatal_center[1]), float(template.striatal_center[2]))
    prof, slopes, medlat, moms, rads = [], [], [], [], []
    for m in (L, R):
        p, s = _axis_profile(v, m); prof.append(p); slopes.append(s)
        medlat.append(_medial_lateral(v, m, mid))
        roi = ndi.binary_dilation(m, iterations=3) & template.brain_mask
        moms.append(_moments(v, roi, cyz)); rads.append(_radial(v, roi, template.brain_mask))
    prof = np.stack(prof); medlat = np.asarray(medlat, np.float32); moms = np.asarray(moms, np.float32); rads = np.stack(rads)
    glcm = glcm_features(v[CROP])
    feats = np.concatenate([
        prof.ravel(), np.asarray(slopes, np.float32),
        medlat[:, 0], medlat[:, 1], medlat[:, 0] / np.maximum(medlat[:, 1], 1e-3),
        moms[:, 0], moms[:, 1], moms[:, 2], moms[:, 3], moms[:, 4], moms[:, 5],
        rads.ravel(), glcm,
        [_asym(prof[0, b], prof[1, b]) for b in range(N_AXIS_BINS)],
        [_asym(slopes[0], slopes[1]), _asym(medlat[0, 0], medlat[1, 0]), _asym(medlat[0, 1], medlat[1, 1]), _asym(moms[0, 0], moms[1, 0]), _asym(moms[0, 5], moms[1, 5])],
        [_asym(rads[0, k], rads[1, k]) for k in range(len(SHELLS_MM) - 1)],
    ]).astype(np.float32)
    assert feats.shape == (len(C_NAMES),), feats.shape
    return np.nan_to_num(feats)


def feat_vector_c(vol: np.ndarray, template) -> np.ndarray:
    """All hand-crafted features of approach C: run-3 (17) + run-4 engineered (44) + the C block."""
    return np.concatenate([striatal_features(vol, template), engineered_features(vol, template), c_block(vol, template)]).astype(np.float32)


def run_featc_arm(arm_cfg: dict, root, vols, st, ok):
    """Inference for arm type "featc": descriptors + striatal-patch PCA -> FeatModelC, averaged over both mirror orientations."""
    from pathlib import Path
    from .featmodel_c import FeatModelC
    from .register import load_template
    from .volume import renormalise
    tpl = load_template(Path(root) / "template.npz")
    if tpl is None:
        return None
    m = FeatModelC.load(Path(root) / Path(arm_cfg["checkpoints"][0]).name)
    idx = np.where(ok)[0]
    z = np.zeros(len(idx))
    for j, i in enumerate(idx):
        v = renormalise(vols[i], st[i], arm_cfg.get("norm", "ref")).astype(np.float32)
        zz = [float(m.decision(feat_vector_c(vv, tpl)[None], striatal_patch(vv)[None])[0]) for vv in (v, mirror(v))]
        z[j] = 0.5 * (zz[0] + zz[1])
    p = np.full(len(ok), np.nan); p[idx] = 1.0 / (1.0 + np.exp(-z))
    print(f"arm {arm_cfg['name']}: featc model ({m.kind}) on {len(idx)} volumes", flush=True)
    return p


In [ ]:
%%writefile datsc/features_v3.py
"""Registered-ROI striatal features (run-3 LR arm and the registration gate G1).

All ROIs are fixed masks in template space (datsc.register.Template), so the features are only as good as the
registration -- which is exactly what the gate measures. Input volumes are registered boxes in reference units
(SBR+1), so `mean - 1` is a specific binding ratio.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage as ndi

FEATURE_NAMES = (
    "left_mean", "right_mean", "left_p90", "right_p90", "left_max", "right_max", "min_side_mean", "max_side_mean",
    "asym_mean", "asym_p90", "left_post_ratio", "right_post_ratio", "min_post_ratio", "frac_gt15", "frac_gt20",
    "striatal_to_ref", "striatal_bg_ring",
)
N_FEATURES = len(FEATURE_NAMES)


def _split_lr(striatal_mask: np.ndarray, mid_x: float):
    xs = np.arange(striatal_mask.shape[0])[:, None, None]
    return striatal_mask & (xs < mid_x), striatal_mask & (xs >= mid_x)


def _post_ratio(v: np.ndarray, m: np.ndarray) -> float:
    """posterior-third mean / anterior-third mean inside one striatum (putamen tail loss shows here). y is P->A."""
    if m.sum() < 8:
        return 1.0
    ys = np.where(m.any(axis=(0, 2)))[0]
    y_lo, y_hi = ys.min(), ys.max()
    t = max(1, (y_hi - y_lo + 1) // 3)
    post = m.copy(); post[:, y_lo + t:, :] = False
    ant = m.copy(); ant[:, : y_hi - t + 1, :] = False
    a = float(v[ant].mean()) if ant.any() else 1.0
    return float(v[post].mean() / max(a, 1e-3))


def striatal_features(vol: np.ndarray, template) -> np.ndarray:
    v = vol.astype(np.float32)
    mid = float(template.striatal_center[0])
    L, R = _split_lr(template.striatal_mask, mid)
    if L.sum() == 0 or R.sum() == 0:
        L = R = template.striatal_mask
    lv, rv = v[L], v[R]
    lm, rm = float(lv.mean()), float(rv.mean())
    lp, rp = float(np.percentile(lv, 90)), float(np.percentile(rv, 90))
    ring = ndi.binary_dilation(template.striatal_mask, iterations=6) & ~ndi.binary_dilation(template.striatal_mask, iterations=2) & template.brain_mask
    ring_mean = float(v[ring].mean()) if ring.any() else 1.0
    s_all = v[template.striatal_mask]
    feats = np.array([
        lm, rm, lp, rp, float(lv.max()), float(rv.max()), min(lm, rm), max(lm, rm),
        abs(lm - rm) / max(lm + rm, 1e-3), abs(lp - rp) / max(lp + rp, 1e-3),
        _post_ratio(v, L), _post_ratio(v, R), min(_post_ratio(v, L), _post_ratio(v, R)),
        float((s_all > 1.5).mean()), float((s_all > 2.0).mean()),
        float(s_all.mean()), float(s_all.mean() / max(ring_mean, 1e-3)),
    ], dtype=np.float32)
    assert feats.shape == (N_FEATURES,)
    return np.nan_to_num(feats)


In [ ]:
%%writefile datsc/features_v4.py
"""Feature-engineered arm ("feat"): registered-space striatal descriptors + PCA of the striatal box -> L2 logistic.

Why a second hand-crafted arm next to the CNNs: the stack rewards members whose errors are uncorrelated with the
convolutional ones, and the public-test loss is a site shift. Everything here is anatomy in template space
(no acquisition or site scalars, which fingerprinted the camera in run 2):

  * anterior->posterior uptake profile per striatum (4 bins + slope): posterior putamen loses first in PD;
  * threshold-volume ladder (fraction of the striatal mask above 1.25 ... 3.0 SBR) per side: comma vs dot shape;
  * hot-spot volume above half the side maximum and its A/P, S/I shift from the template centroid;
  * left/right asymmetry of each of the above (asymmetric onset);
  * PCA scores of the 48x48x32 striatal box block-averaged to 24x24x16 (the classical PCA + linear classifier
    for registered DaT-SPECT), fitted on the training folds only.

Volumes are registered boxes in reference units (SBR+1). Mirror orientation is a label-preserving augmentation:
the model is fitted on both orientations and predictions are averaged over both.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage as ndi

from .features_v3 import _split_lr, striatal_features, FEATURE_NAMES as V3_NAMES

N_BINS = 4
LADDER = (1.25, 1.5, 2.0, 2.5, 3.0)
CROP = (slice(24, 72), slice(34, 82), slice(18, 50))       # 48x48x32 around the template striatal centre (47.5, 57.3, 33.5)
POOL = 2                                                  # block-average -> 24x24x16 = 9216 values
N_PATCH = (48 // POOL) * (48 // POOL) * (32 // POOL)

_side = ("L", "R")
ENG_NAMES = tuple(
    [f"{s}_bin{b}" for s in _side for b in range(N_BINS)] + [f"{s}_slope" for s in _side]
    + [f"{s}_gt{t}" for s in _side for t in LADDER]
    + [f"{s}_hotvol" for s in _side] + [f"{s}_hot_dy" for s in _side] + [f"{s}_hot_dz" for s in _side]
    + [f"asym_bin{b}" for b in range(N_BINS)] + [f"asym_gt{t}" for t in LADDER] + ["asym_hotvol", "asym_slope"]
)
FEATURE_NAMES = tuple(V3_NAMES) + ENG_NAMES
N_FEATURES = len(FEATURE_NAMES)


def _profile(v: np.ndarray, m: np.ndarray, n_bins: int = N_BINS):
    """Mean SBR in n_bins slabs along y (posterior -> anterior) inside one striatum, plus the linear slope."""
    if m.sum() < 8:
        return np.zeros(n_bins, np.float32), 0.0
    ys = np.where(m.any(axis=(0, 2)))[0]
    edges = np.linspace(ys.min(), ys.max() + 1, n_bins + 1)
    yy = np.arange(m.shape[1])[None, :, None]
    out = []
    for b in range(n_bins):
        sel = m & (yy >= edges[b]) & (yy < edges[b + 1])
        out.append(float(v[sel].mean()) if sel.any() else (out[-1] if out else 1.0))
    out = np.asarray(out, np.float32)
    x = np.arange(n_bins, dtype=np.float32)
    slope = float(np.polyfit(x, out, 1)[0]) if n_bins > 1 else 0.0
    return out, slope


def _hot(v: np.ndarray, m_dil: np.ndarray, centre_yz):
    """Volume (voxels) above half of the side maximum inside a dilated striatal ROI and the centroid offset (y, z)."""
    vals = v[m_dil]
    if vals.size == 0:
        return 0.0, 0.0, 0.0
    thr = 1.0 + 0.5 * max(float(vals.max()) - 1.0, 1e-3)
    hot = m_dil & (v > thr)
    n = float(hot.sum())
    if n == 0:
        return 0.0, 0.0, 0.0
    cy, cz = ndi.center_of_mass(hot)[1:]
    return n, float(cy - centre_yz[0]), float(cz - centre_yz[1])


def engineered_features(vol: np.ndarray, template) -> np.ndarray:
    v = vol.astype(np.float32)
    mid = float(template.striatal_center[0])
    L, R = _split_lr(template.striatal_mask, mid)
    if L.sum() == 0 or R.sum() == 0:
        L = R = template.striatal_mask
    cyz = (float(template.striatal_center[1]), float(template.striatal_center[2]))
    prof, slopes, ladder, hots = [], [], [], []
    for m in (L, R):
        p, s = _profile(v, m); prof.append(p); slopes.append(s)
        sv = v[m]; ladder.append([float((sv > t).mean()) for t in LADDER])
        hots.append(_hot(v, ndi.binary_dilation(m, iterations=3) & template.brain_mask, cyz))
    prof = np.stack(prof); ladder = np.asarray(ladder, np.float32); hots = np.asarray(hots, np.float32)
    asym = lambda a, b: abs(a - b) / max(abs(a) + abs(b), 1e-3)
    feats = np.concatenate([
        prof.ravel(), np.asarray(slopes, np.float32), ladder.ravel(),
        hots[:, 0], hots[:, 1], hots[:, 2],
        [asym(prof[0, b], prof[1, b]) for b in range(N_BINS)], [asym(ladder[0, i], ladder[1, i]) for i in range(len(LADDER))],
        [asym(hots[0, 0], hots[1, 0]), asym(slopes[0], slopes[1])],
    ]).astype(np.float32)
    assert feats.shape == (len(ENG_NAMES),), feats.shape
    return np.nan_to_num(feats)


def feat_vector(vol: np.ndarray, template) -> np.ndarray:
    """All hand-crafted features: the 17 run-3 ones + the engineered block."""
    return np.concatenate([striatal_features(vol, template), engineered_features(vol, template)]).astype(np.float32)


def striatal_patch(vol: np.ndarray) -> np.ndarray:
    """Block-averaged striatal box in SBR units (vol - 1), flattened: the PCA input."""
    v = vol[CROP].astype(np.float32) - 1.0
    a, b, c = v.shape
    v = v.reshape(a // POOL, POOL, b // POOL, POOL, c // POOL, POOL).mean(axis=(1, 3, 5))
    return v.ravel()


def mirror(vol: np.ndarray) -> np.ndarray:
    """L/R mirror in the registered frame (axis 0 is X; the template masks are symmetrised)."""
    return vol[::-1]


In [ ]:
%%writefile datsc/folds.py
"""Repeat-patient audit via coarse embeddings, then stratified group K-fold."""
from __future__ import annotations

import numpy as np
from scipy.ndimage import zoom
from sklearn.model_selection import StratifiedGroupKFold


def embed(crops: np.ndarray, size: int = 16) -> np.ndarray:
    """crops [N, X, Y, Z] -> L2-normalised [N, size^3] embeddings."""
    out = []
    for c in crops:
        f = [size / s for s in c.shape]
        e = zoom(c.astype(np.float32), f, order=1).ravel()
        e = e - e.mean()
        out.append(e / (np.linalg.norm(e) + 1e-9))
    return np.stack(out)


def group_near_duplicates(emb: np.ndarray, thresh: float = 0.98) -> np.ndarray:
    n = len(emb)
    parent = np.arange(n)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    sim = emb @ emb.T
    ii, jj = np.where(np.triu(sim, k=1) > thresh)
    for i, j in zip(ii, jj):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[rj] = ri
    roots = np.array([find(i) for i in range(n)])
    _, groups = np.unique(roots, return_inverse=True)
    return groups


def assign_folds(y: np.ndarray, groups: np.ndarray, n_splits: int = 5, seed: int = 0) -> np.ndarray:
    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.full(len(y), -1, dtype=int)
    for k, (_, val_idx) in enumerate(skf.split(np.zeros(len(y)), y, groups)):
        folds[val_idx] = k
    assert (folds >= 0).all()
    return folds


In [ ]:
%%writefile datsc/infer.py
"""Inference stages for main.py. Stage-wise: prior -> LR -> CNN ensemble, each overwriting submission.csv."""
from __future__ import annotations

import os
import time
from concurrent.futures import ProcessPoolExecutor, TimeoutError as FutTimeout
from pathlib import Path

import numpy as np
import pandas as pd

from .calibrate import EPS
from .config import CROP_SHAPE, N_SCALARS, find_nifti_path
from .features import N_RADIOMICS, hotspot_radiomics
from .lrmodel import LRModel
from .preprocess import preprocess

PER_VOLUME_TIMEOUT_S = 90
LADDER_SKIP_CNN_S = 130 * 60      # past this, stop preprocessing / loading more CNN checkpoints; keep what is written


def _worker_init():
    for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"):
        os.environ[k] = "1"


def _prep(path):
    r = preprocess(path)
    return r["crop"], r["crop_com"], r["scalars"], hotspot_radiomics(r["crop"].astype(np.float32))


def write_submission(uids, probs, out_csv):
    pd.DataFrame({"uid": uids, "is_pathologic": np.clip(probs, EPS, 1 - EPS)}).to_csv(out_csv, index=False)


def preprocess_all(uids, nifti_dir: Path, n_workers: int, deadline_s: float, t0: float):
    """Returns (crops[N,X,Y,Z] fp16, crops_com[N,X,Y,Z] fp16, scalars[N,8], radiomics[N,14], ok[N] bool)."""
    N = len(uids)
    crops = np.zeros((N,) + CROP_SHAPE, np.float16); crops_com = np.zeros((N,) + CROP_SHAPE, np.float16)
    sc = np.zeros((N, N_SCALARS), np.float32)
    rad = np.zeros((N, N_RADIOMICS), np.float32); ok = np.zeros(N, bool)
    paths = [find_nifti_path(nifti_dir, u) for u in uids]
    ex = ProcessPoolExecutor(max_workers=n_workers, initializer=_worker_init)
    futs = {ex.submit(_prep, p): i for i, p in enumerate(paths) if p is not None}
    for fut, i in futs.items():
        remaining = deadline_s - (time.time() - t0)
        if remaining <= 0:
            print("ladder: preprocessing deadline hit; cancelling remaining volumes")
            break
        try:
            c, cc, s, r = fut.result(timeout=min(PER_VOLUME_TIMEOUT_S, remaining))
            crops[i], crops_com[i], sc[i], rad[i], ok[i] = c, cc, s, r, True
        except (Exception, FutTimeout):  # noqa: BLE001
            ok[i] = False
    # never block on a hung worker: drop queued work, then hard-terminate every worker process.
    # (Idle workers would otherwise outlive us holding the stdout pipe, which hangs the caller.)
    procs = list((getattr(ex, "_processes", None) or {}).values())   # snapshot: shutdown() nulls this
    ex.shutdown(wait=False, cancel_futures=True)
    for proc in procs:
        try:
            proc.terminate()
        except Exception:  # noqa: BLE001
            pass
    for proc in procs:
        try:
            proc.join(timeout=5)
        except Exception:  # noqa: BLE001
            pass
    print(f"preprocessed {int(ok.sum())}/{N} volumes in {time.time()-t0:.0f}s")
    return crops, crops_com, sc, rad, ok


def run_lr(lr_json: Path, rad, sc, ok, pi):
    m = LRModel.load(lr_json)
    p = np.full(len(ok), pi)
    if ok.any():
        p[ok] = m.predict_proba(np.concatenate([rad[ok], sc[ok]], axis=1))
    return p


def run_cnn(weight_groups: list, sc, ok, t0: float, batch_size: int = 32):
    """weight_groups: list of (list_of_weight_paths, crops_array) — e.g. hotspot checkpoints on the
    hotspot crop and `_com` checkpoints on the CoM crop. All checkpoints are averaged with equal weight."""
    import torch
    from .model import SmallResNet3D
    device = "cuda" if torch.cuda.is_available() else "cpu"
    idx = np.where(ok)[0]
    acc = np.zeros(len(idx)); used = 0
    total = sum(len(w) for w, _ in weight_groups)
    s_all = torch.from_numpy(sc[idx])
    for weights, crops in weight_groups:
        if not weights:
            continue
        x_all = torch.from_numpy(crops[idx].astype(np.float32))[:, None]
        for wpath in weights:
            if time.time() - t0 > LADDER_SKIP_CNN_S:
                print("ladder: skipping remaining checkpoints"); break
            model = SmallResNet3D(n_scalars=N_SCALARS).to(device)
            model.load_state_dict(torch.load(wpath, map_location=device)); model.eval()
            out = []
            with torch.no_grad():
                for b in range(0, len(idx), batch_size):
                    x = x_all[b:b + batch_size].to(device); s = s_all[b:b + batch_size].to(device)
                    z = 0.5 * (model(x, s) + model(x.flip(dims=[2]), s))
                    out.append(torch.sigmoid(z).float().cpu().numpy())
            acc += np.concatenate(out); used += 1
    print(f"cnn checkpoints used: {used}/{total}")
    if used == 0:
        return None
    p = np.full(len(ok), np.nan); p[idx] = acc / used
    return p


In [ ]:
%%writefile datsc/infer_v3.py
"""Run-3 inference: manifest-driven multi-arm ensemble over the whole-brain cache representation.

manifest.json = {"pi": float, "T": float, "lam": float, "weights": {arm_name: w}, "arms": [
    {"name": str, "arm": "3d"|"2d", "spec": InputSpec dict, "model_kw": dict, "norm": str, "checkpoints": [relative paths]}]}
Stages (each overwrites submission.csv): prior -> per-arm averages combined by the fitted weights/T/lam.
"""
from __future__ import annotations

import json
import os
import time
from concurrent.futures import ProcessPoolExecutor, TimeoutError as FutTimeout
from pathlib import Path

import numpy as np
import pandas as pd

from .calibrate import EPS, logit, sigmoid
from .config import N_SCALARS, find_nifti_path
from .register import N_REG_STATS, load_template
from .volume import N_STATS, VOL_SHAPE, prepare_volume, renormalise

PER_VOLUME_TIMEOUT_S = 120
LADDER_STOP_S = 150 * 60           # stop starting new checkpoints after this (3-h limit)
_TEMPLATE = None


def _worker_init(template_path=None):
    global _TEMPLATE
    for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"):
        os.environ[k] = "1"
    _TEMPLATE = load_template(template_path)


def _prep(path):
    r = prepare_volume(path, template=_TEMPLATE)
    return r["vol"], r["scalars"], r["stats"]


def write_submission(uids, probs, out_csv):
    pd.DataFrame({"uid": uids, "is_pathologic": np.clip(probs, EPS, 1 - EPS)}).to_csv(out_csv, index=False)


def preprocess_all(uids, nifti_dir: Path, n_workers: int, deadline_s: float, t0: float, template_path=None):
    N = len(uids)
    n_st = N_STATS + (N_REG_STATS if template_path and Path(template_path).exists() else 0)
    vols = np.zeros((N,) + VOL_SHAPE, np.float16); sc = np.zeros((N, N_SCALARS), np.float32)
    st = np.zeros((N, n_st), np.float32); ok = np.zeros(N, bool)
    paths = [find_nifti_path(nifti_dir, u) for u in uids]
    ex = ProcessPoolExecutor(max_workers=n_workers, initializer=_worker_init, initargs=(str(template_path) if template_path else None,))
    futs = {ex.submit(_prep, p): i for i, p in enumerate(paths) if p is not None}
    for fut, i in futs.items():
        remaining = deadline_s - (time.time() - t0)
        if remaining <= 0:
            print("ladder: preprocessing deadline hit; cancelling remaining volumes"); break
        try:
            v, s, t = fut.result(timeout=min(PER_VOLUME_TIMEOUT_S, remaining))
            vols[i], sc[i], st[i], ok[i] = v, s, t, True
        except (Exception, FutTimeout):  # noqa: BLE001
            ok[i] = False
    procs = list((getattr(ex, "_processes", None) or {}).values())
    ex.shutdown(wait=False, cancel_futures=True)
    for proc in procs:
        try: proc.terminate()
        except Exception: pass  # noqa: BLE001
    for proc in procs:
        try: proc.join(timeout=5)
        except Exception: pass  # noqa: BLE001
    print(f"preprocessed {int(ok.sum())}/{N} volumes in {time.time()-t0:.0f}s", flush=True)
    return vols, sc, st, ok


def run_lr_arm(arm_cfg: dict, root: Path, vols, st, ok):
    """Registered-ROI striatal features -> logistic regression (arm type "lr"). Needs artifacts/template.npz."""
    from .features_v3 import striatal_features
    from .lrmodel import LRModel
    tpl = load_template(root / "template.npz")
    if tpl is None:
        return None
    m = LRModel.load(root / Path(arm_cfg["checkpoints"][0]).name)
    idx = np.where(ok)[0]
    X = np.stack([striatal_features(renormalise(vols[i], st[i], arm_cfg.get("norm", "ref")), tpl) for i in idx])
    p = np.full(len(ok), np.nan); p[idx] = m.predict_proba(X)
    print(f"arm {arm_cfg['name']}: LR on {len(idx)} volumes", flush=True)
    return p


def run_feat_arm(arm_cfg: dict, root: Path, vols, st, ok):
    """Feature-engineered arm (arm type "feat"): registered-space descriptors + striatal-patch PCA -> logistic, averaged
    over both mirror orientations. Needs artifacts/template.npz and the npz model."""
    from .featmodel import FeatModel
    from .features_v4 import feat_vector, mirror, striatal_patch
    tpl = load_template(root / "template.npz")
    if tpl is None:
        return None
    m = FeatModel.load(root / Path(arm_cfg["checkpoints"][0]).name)
    idx = np.where(ok)[0]
    z = np.zeros(len(idx))
    for j, i in enumerate(idx):
        v = renormalise(vols[i], st[i], arm_cfg.get("norm", "ref")).astype(np.float32)
        zz = [float(m.decision(feat_vector(vv, tpl)[None], striatal_patch(vv)[None])[0]) for vv in (v, mirror(v))]
        z[j] = 0.5 * (zz[0] + zz[1])
    p = np.full(len(ok), np.nan); p[idx] = 1.0 / (1.0 + np.exp(-z))
    print(f"arm {arm_cfg['name']}: feat model on {len(idx)} volumes", flush=True)
    return p


def typicality_all(arm_cfg: dict, root: Path, vols, st, ok):
    """Per-scan typicality (Mahalanobis distance in the feature arm's PCA space) for the ok volumes; nan elsewhere."""
    from .featmodel import FeatModel
    from .features_v4 import striatal_patch
    m = FeatModel.load(root / Path(arm_cfg["checkpoints"][0]).name)
    idx = np.where(ok)[0]
    d = np.full(len(ok), np.nan)
    if len(idx):
        P = np.stack([striatal_patch(renormalise(vols[i], st[i], arm_cfg.get("norm", "ref")).astype(np.float32)) for i in idx])
        d[idx] = m.typicality(P)
    return d


def run_arm(arm_cfg: dict, root: Path, vols, sc, st, ok, t0: float, batch_size: int = 16):
    """Average of all checkpoints of one arm over the ok volumes. Returns probs[N] (nan where not ok) or None."""
    if arm_cfg["arm"] == "lr":
        return run_lr_arm(arm_cfg, root, vols, st, ok)
    if arm_cfg["arm"] == "feat":
        return run_feat_arm(arm_cfg, root, vols, st, ok)
    if arm_cfg["arm"] == "featc":                       # approach C: richer descriptors + kernel/MLP model
        from .features_c import run_featc_arm
        return run_featc_arm(arm_cfg, root, vols, st, ok)
    import torch
    from .models_v3 import build_model
    from .train_b import make_spec
    device = "cuda" if torch.cuda.is_available() else "cpu"
    spec = make_spec(arm_cfg["spec"])                       # InputSpec for 2d/3d, InputSpecB for the tri-planar arm
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return None
    norm = arm_cfg.get("norm", "p75")
    tpl = load_template(root / "template.npz") if norm == "hist" else None
    x_all = np.stack([renormalise(vols[i], st[i], norm, tpl) for i in idx]).astype(np.float32)[:, None]
    x_all = torch.from_numpy(x_all); s_all = torch.from_numpy(sc[idx])
    acc = np.zeros(len(idx)); used = 0
    for ck in arm_cfg["checkpoints"]:
        if time.time() - t0 > LADDER_STOP_S:
            print("ladder: skipping remaining checkpoints"); break
        try:
            # weights_path is a training-time convenience (ImageNet init); the checkpoint holds the whole model and the
            # file does not exist inside the submission container (run 3 v2: every 2D arm failed with FileNotFoundError)
            model_kw = {k: v for k, v in arm_cfg.get("model_kw", {}).items() if k != "weights_path"}
            model = build_model(arm_cfg["arm"], **model_kw).to(device)
            sd = torch.load(str(root / Path(ck).name), map_location=device, weights_only=True)   # root == the artifacts dir
            model.load_state_dict(sd); model.eval()
            out = []
            with torch.no_grad(), torch.autocast(device_type="cuda", enabled=device.startswith("cuda")):
                for b in range(0, len(idx), batch_size):
                    x = x_all[b:b + batch_size].to(device); s = s_all[b:b + batch_size].to(device)
                    z = torch.stack([model(inp, s).float() for inp in spec.variants(x)]).mean(0)
                    out.append(torch.sigmoid(z).cpu().numpy())
            acc += np.concatenate(out); used += 1
        except Exception as e:  # noqa: BLE001
            print(f"checkpoint failed ({Path(ck).name}): {type(e).__name__}: {str(e)[:160]}", flush=True)
    print(f"arm {arm_cfg['name']}: checkpoints used {used}/{len(arm_cfg['checkpoints'])} at {time.time()-t0:.0f}s", flush=True)
    if used == 0:
        return None
    p = np.full(len(ok), np.nan); p[idx] = acc / used
    return p


def combine(arm_probs: dict, weights: dict, T: float, lam: float, pi: float, clip=None, hedge=None, typ=None):
    """Weighted logit average over the arms that produced predictions, temperature, prior mix.
    clip: [lo, hi] final probability bounds (winners' log-loss playbook), default [EPS, 1-EPS].
    hedge: {"thresh": t, "lam": l}: scans whose arm logits disagree by more than t are mixed with the prior at
    strength l instead of lam (uncertainty hedging; only meaningful with >= 2 arms).
    typ: {"kappa": k, "d0": d0, "d": per-scan typicality}: temperature T_i = T * (1 + k * max(0, d_i - d0)) so scans far
    from the training distribution are shrunk toward the prior (per-sample; nan d -> T)."""
    names = [n for n in weights if n in arm_probs and arm_probs[n] is not None]
    if not names:
        return None
    w = np.array([weights[n] for n in names], float); w = w / w.sum()
    Z = np.stack([logit(arm_probs[n]) for n in names])          # [A, N] (nan where not ok)
    z = np.nansum(Z * w[:, None], axis=0) / np.nansum(np.where(np.isnan(Z), 0, w[:, None]), axis=0)
    if typ and typ.get("kappa", 0) > 0 and typ.get("d") is not None:
        d = np.asarray(typ["d"], float)
        Ti = T * (1 + float(typ["kappa"]) * np.maximum(0.0, np.nan_to_num(d, nan=float(typ["d0"])) - float(typ["d0"])))
    else:
        Ti = T
    p = sigmoid(z / Ti)
    lam_i = np.full(p.shape, float(lam))
    if hedge and len(names) >= 2:
        spread = np.nanmax(Z, axis=0) - np.nanmin(Z, axis=0)
        lam_i = np.where(spread > float(hedge["thresh"]), float(hedge["lam"]), lam_i)
    p = (1 - lam_i) * p + lam_i * pi
    lo, hi = (clip if clip else (EPS, 1 - EPS))
    return np.clip(p, max(lo, EPS), min(hi, 1 - EPS))


def load_manifest(path: Path) -> dict:
    return json.loads(Path(path).read_text())


In [ ]:
%%writefile datsc/io.py
"""NIfTI loading: canonical RAS orientation and isotropic resampling."""
from __future__ import annotations

from pathlib import Path

import nibabel as nib
import numpy as np
from scipy.ndimage import zoom

from .config import SPACING_MM


def load_canonical(path: str | Path):
    """Load .nii/.nii.gz, reorient to RAS. Returns (vol[x,y,z] float32, spacing_mm, native_shape)."""
    img = nib.load(str(path))
    img = nib.as_closest_canonical(img)
    vol = np.asarray(img.dataobj, dtype=np.float32)
    if vol.ndim > 3:                      # defensive: squeeze any trailing singleton dims
        vol = vol.reshape(vol.shape[:3])
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    spacing = tuple(float(abs(s)) for s in img.header.get_zooms()[:3])
    return vol, spacing, tuple(int(s) for s in vol.shape)


def resample_iso(vol: np.ndarray, spacing, target_mm: float = SPACING_MM) -> np.ndarray:
    factors = [s / target_mm for s in spacing]
    out = zoom(vol, factors, order=1, mode="nearest", prefilter=False)
    return np.ascontiguousarray(out, dtype=np.float32)


def load_iso(path: str | Path, target_mm: float = SPACING_MM):
    """Convenience: load + resample. Returns (vol_iso, native_spacing)."""
    vol, spacing, _ = load_canonical(path)
    return resample_iso(vol, spacing, target_mm), spacing


In [ ]:
%%writefile datsc/lrmodel.py
"""L2 logistic regression with standardisation, serialised to JSON (no pickle in the submission)."""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression

from .config import SCALAR_NAMES
from .features import RADIOMICS_NAMES

DEFAULT_NAMES = list(RADIOMICS_NAMES) + list(SCALAR_NAMES)


class LRModel:
    def __init__(self, C: float = 1.0, names=None):
        self.C = C
        self.names = list(names) if names is not None else DEFAULT_NAMES
        self.mean = self.scale = self.coef = None
        self.intercept = 0.0

    def fit(self, X: np.ndarray, y: np.ndarray):
        X = np.asarray(X, np.float64)
        self.mean = X.mean(0); self.scale = X.std(0) + 1e-6
        clf = LogisticRegression(C=self.C, max_iter=2000).fit((X - self.mean) / self.scale, y)
        self.coef = clf.coef_.ravel(); self.intercept = float(clf.intercept_[0])
        return self

    def decision(self, X: np.ndarray) -> np.ndarray:
        Z = (np.asarray(X, np.float64) - self.mean) / self.scale
        return Z @ self.coef + self.intercept

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-self.decision(X)))

    def save(self, path):
        Path(path).write_text(json.dumps({"C": self.C, "names": self.names, "mean": self.mean.tolist(),
                                          "scale": self.scale.tolist(), "coef": self.coef.tolist(),
                                          "intercept": self.intercept}))

    @classmethod
    def load(cls, path):
        d = json.loads(Path(path).read_text())
        m = cls(C=d["C"], names=d["names"])
        m.mean, m.scale = np.array(d["mean"]), np.array(d["scale"])
        m.coef, m.intercept = np.array(d["coef"]), float(d["intercept"])
        return m


def feature_matrix(cache_dir: Path, uids) -> np.ndarray:
    rows = []
    for u in uids:
        z = np.load(Path(cache_dir) / f"{u}.npz")
        rows.append(np.concatenate([z["radiomics"], z["scalars"]]))
    return np.stack(rows).astype(np.float32)


In [ ]:
%%writefile datsc/model.py
"""Compact 3D ResNet (~0.9M params) with a scalar-feature head."""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv3d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm3d(cout)
        self.c2 = nn.Conv3d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm3d(cout)
        self.skip = None
        if stride != 1 or cin != cout:
            self.skip = nn.Sequential(nn.Conv3d(cin, cout, 1, stride, bias=False), nn.BatchNorm3d(cout))

    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return F.relu(y + (x if self.skip is None else self.skip(x)))


class SmallResNet3D(nn.Module):
    def __init__(self, n_scalars: int = 8, widths=(16, 32, 64, 128), dropout: float = 0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv3d(1, widths[0], 3, 1, 1, bias=False), nn.BatchNorm3d(widths[0]), nn.ReLU())
        layers, cin = [], widths[0]
        for i, w in enumerate(widths):
            layers.append(Block(cin, w, stride=1 if i == 0 else 2)); cin = w
        self.layers = nn.Sequential(*layers)
        self.scalar_bn = nn.BatchNorm1d(n_scalars)
        self.head = nn.Sequential(nn.Linear(cin + n_scalars, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x, scalars):
        h = self.layers(self.stem(x))
        h = F.adaptive_avg_pool3d(h, 1).flatten(1)
        h = torch.cat([h, self.scalar_bn(scalars)], dim=1)
        return self.head(h).squeeze(1)


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())


In [ ]:
%%writefile datsc/models_a.py
"""Approach A: a stronger striatal-crop 3D family (run 6a).

Why: the 48x48x32 striatal-crop ResNet3D (2.0M params, 1 block per stage) is the best single member of the run-4a stack
(OOF 0.2568 / AUROC 0.960) and the least correlated with the 2.5D arms (logit r 0.93 vs 0.97 between the 2D arms), yet it
is under-trained (validation loss still falling between epochs 40 and 50) and shallow. This module adds a configurable
family used through `datsc.models_v3.build_model(arm="3d", arch="se", ...)`:

  * `SEResNet3D`: GroupNorm/SiLU basic blocks with channel squeeze-and-excitation (Hu et al. 2018) and stochastic depth
    (Huang et al. 2016) per residual branch, `blocks` residual blocks per stage, widths configurable. With the default
    (32, 64, 128, 256) x 1 block it is ~3.5x the FLOPs of the run-4a net at the same 48x48x32 input.
  * Inference needs nothing new: the manifest carries `model_kw` (arch/widths/blocks/se/drop_path) and `build_model`
    dispatches on `arch`; stochastic depth is identity in eval mode.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def _gn(c: int, groups: int = 8) -> nn.GroupNorm:
    return nn.GroupNorm(min(groups, c), c)


class SE3D(nn.Module):
    """Squeeze-and-excitation channel gate (global average pool -> 2-layer MLP -> sigmoid)."""

    def __init__(self, c: int, r: int = 8):
        super().__init__()
        h = max(4, c // r)
        self.fc1 = nn.Linear(c, h); self.fc2 = nn.Linear(h, c)

    def forward(self, x):
        s = F.adaptive_avg_pool3d(x, 1).flatten(1)
        s = torch.sigmoid(self.fc2(F.silu(self.fc1(s))))
        return x * s.view(s.shape[0], s.shape[1], 1, 1, 1)


class DropPath(nn.Module):
    """Stochastic depth on the residual branch (per sample); identity in eval mode."""

    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if not self.training or self.p <= 0:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep


class SEBlock3D(nn.Module):
    def __init__(self, cin: int, cout: int, stride: int, se: bool = True, drop_path: float = 0.0):
        super().__init__()
        self.c1 = nn.Conv3d(cin, cout, 3, stride, 1, bias=False); self.n1 = _gn(cout)
        self.c2 = nn.Conv3d(cout, cout, 3, 1, 1, bias=False); self.n2 = _gn(cout)
        self.se = SE3D(cout) if se else nn.Identity()
        self.dp = DropPath(drop_path)
        self.skip = None
        if stride != 1 or cin != cout:
            self.skip = nn.Sequential(nn.Conv3d(cin, cout, 1, stride, bias=False), _gn(cout))

    def forward(self, x):
        y = F.silu(self.n1(self.c1(x)))
        y = self.se(self.n2(self.c2(y)))
        return F.silu(self.dp(y) + (x if self.skip is None else self.skip(x)))


class SEResNet3D(nn.Module):
    """Striatal-crop 3D classifier: stem -> 4 stages (stride 1, 2, 2, 2) of `blocks` SE residual blocks -> GAP -> head.
    Exposes the same interface as ResNet3D (features(), feat_dim, head, scalars) so the trainer's DANN/DRO path works."""

    def __init__(self, n_scalars: int = 0, widths=(32, 64, 128, 256), blocks: int = 1, se: bool = True,
                 drop_path: float = 0.1, dropout: float = 0.3, stem_stride: int = 1):
        super().__init__()
        from .models_v3 import ScalarHead
        widths = tuple(int(w) for w in widths); n_blocks = int(blocks)
        self.stem = nn.Sequential(nn.Conv3d(1, widths[0], 3, stem_stride, 1, bias=False), _gn(widths[0]), nn.SiLU())
        layers, cin = [], widths[0]
        total = len(widths) * n_blocks; k = 0
        for i, w in enumerate(widths):
            for b in range(n_blocks):
                dp = drop_path * k / max(1, total - 1); k += 1            # linearly increasing stochastic depth
                layers.append(SEBlock3D(cin, w, stride=(2 if (i > 0 and b == 0) else 1), se=se, drop_path=dp)); cin = w
        self.layers = nn.Sequential(*layers)
        self.scalars = ScalarHead(n_scalars)
        self.feat_dim = cin
        self.head = nn.Sequential(nn.Linear(cin + n_scalars, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))

    def features(self, x):
        return F.adaptive_avg_pool3d(self.layers(self.stem(x)), 1).flatten(1)

    def forward(self, x, scalars=None):
        return self.head(self.scalars(self.features(x), scalars)).squeeze(1)


def build_3d_a(n_scalars: int = 0, **kw) -> nn.Module:
    """Factory used by datsc.models_v3.build_model when model_kw['arch'] == 'se'."""
    keys = ("widths", "blocks", "se", "drop_path", "dropout", "stem_stride")
    return SEResNet3D(n_scalars=n_scalars, **{k: v for k, v in kw.items() if k in keys})


In [ ]:
%%writefile datsc/models_b.py
"""Run-6b model: tri-planar late-fusion network on a shared ImageNet backbone.

Input [B, V, C, H, W]: V view images (axial / coronal / sagittal-left / sagittal-right slab stacks of the registered box),
each C = n_slabs channels. All views go through ONE backbone (weight sharing = 4x the training signal per parameter and
the same texture features across planes), get a learned view embedding, and are fused by softmax attention pooling
(or concatenation) before the head. `features()` / `head` / `scalars` / `feat_dim` follow the run-3 model contract so the
existing trainer (mixup, EMA-final, DANN site head, group-DRO) and inference work unchanged.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from .models_v3 import Net2D, ScalarHead


class TriPlanarNet(nn.Module):
    MEAN, STD = Net2D.MEAN, Net2D.STD

    def __init__(self, backbone: str = "convnext_tiny", weights_path=None, n_in: int = 3, n_views: int = 4,
                 fusion: str = "attn", dropout: float = 0.3, in_scale: float = 8.0, n_scalars: int = 0, hidden: int = 256):
        super().__init__()
        body = Net2D(backbone=backbone, weights_path=weights_path, n_in=n_in, dropout=dropout, in_scale=in_scale)
        self.backbone = body.backbone                      # named `backbone` so the trainer's backbone-lr param split applies
        dim = body.feat_dim
        self.in_scale, self.n_views, self.fusion = float(in_scale), int(n_views), fusion
        self.view_emb = nn.Parameter(torch.zeros(n_views, dim))
        if fusion == "attn":
            self.score = nn.Sequential(nn.Linear(dim, 128), nn.SiLU(), nn.Linear(128, 1))
            self.feat_dim = dim
        elif fusion == "concat":
            self.feat_dim = dim * n_views
        else:
            raise ValueError(fusion)
        self.scalars = ScalarHead(n_scalars)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.feat_dim + n_scalars, hidden), nn.SiLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def features(self, x: torch.Tensor) -> torch.Tensor:
        """[B,V,C,H,W] -> [B, feat_dim]."""
        B, V = x.shape[:2]
        img = (x.flatten(0, 1).clamp(0, self.in_scale) / self.in_scale - self.MEAN) / self.STD
        h = self.backbone(img).view(B, V, -1) + self.view_emb[:V]
        if self.fusion == "attn":
            w = torch.softmax(self.score(h).squeeze(-1), dim=1)          # [B, V]
            return (w.unsqueeze(-1) * h).sum(1)
        return h.flatten(1)

    def view_attention(self, x: torch.Tensor) -> torch.Tensor:
        """Softmax view weights [B, V] (attn fusion only) -- for reporting which planes the model trusts."""
        B, V = x.shape[:2]
        img = (x.flatten(0, 1).clamp(0, self.in_scale) / self.in_scale - self.MEAN) / self.STD
        h = self.backbone(img).view(B, V, -1) + self.view_emb[:V]
        return torch.softmax(self.score(h).squeeze(-1), dim=1)

    def forward(self, x, scalars=None):
        return self.head(self.scalars(self.features(x), scalars)).squeeze(1)


In [ ]:
%%writefile datsc/models_h.py
"""Run 7: hemisphere-level striatal member (arm "3d", arch "hemi").

The registered striatal crop [B,1,48,48,32] is split at the midline (x = 24) into a left and a right half-crop of
[B,1,24,48,32]; the right half is mirrored along x so both halves sit in the same orientation, and ONE shared encoder
(the run-3 GroupNorm/SiLU ResNet3D blocks) embeds each. The head is symmetric in the two hemispheres by construction:
  z = pair_head([f_L + f_R, |f_L - f_R|]) + max(side_head(f_L), side_head(f_R))
so a one-sided dopaminergic deficit (the typical onset) is an explicit signal, and mirroring the whole scan (the
label-preserving flip used in training augmentation and in TTA) leaves the logit unchanged.

Why: in every run so far the left/right asymmetry descriptors were among the strongest single features, and the
whole-crop nets have to discover asymmetry through convolutions that span the midline. The half-crop split with
mirroring doubles the effective sample count for the encoder (each scan contributes two hemispheres in one
orientation) and gives the head the asymmetry directly.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from .models_v3 import Block3D, ScalarHead, _gn


class HemiEncoder3D(nn.Module):
    def __init__(self, widths=(24, 48, 96, 192), stem_stride: int = 1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv3d(1, widths[0], 3, stem_stride, 1, bias=False), _gn(widths[0]), nn.SiLU())
        layers, cin = [], widths[0]
        for i, w in enumerate(widths):
            layers.append(Block3D(cin, w, stride=1 if i == 0 else 2)); cin = w
        self.layers = nn.Sequential(*layers)
        self.out_dim = cin

    def forward(self, x):
        return F.adaptive_avg_pool3d(self.layers(self.stem(x)), 1).flatten(1)


class HemiNet3D(nn.Module):
    """Shared encoder over the two mirrored striatal half-crops; symmetric pair + max-side head."""

    def __init__(self, n_scalars: int = 0, widths=(24, 48, 96, 192), dropout: float = 0.3, stem_stride: int = 1, split: int | None = None):
        super().__init__()
        self.encoder = HemiEncoder3D(widths, stem_stride)
        d = self.encoder.out_dim
        self.feat_dim = 2 * d
        self.split = split                                     # x index of the midline; None = half of the crop width
        self.scalars = ScalarHead(n_scalars)
        self.pair_head = nn.Sequential(nn.Linear(2 * d + n_scalars, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.side_head = nn.Sequential(nn.Linear(d, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def halves(self, x):
        """[B,1,X,Y,Z] -> (left [B,1,X/2,Y,Z], right mirrored to the left orientation)."""
        X = x.shape[2]; m = self.split if self.split is not None else X // 2
        left, right = x[:, :, :m], x[:, :, m:]
        right = right.flip(dims=[2])
        if left.shape[2] != right.shape[2]:                   # odd widths: trim to the common size
            k = min(left.shape[2], right.shape[2]); left, right = left[:, :, :k], right[:, :, :k]
        return left, right

    def features(self, x):
        """Symmetric pooled features [f_L + f_R, |f_L - f_R|] (used by the DANN site head)."""
        left, right = self.halves(x)
        B = x.shape[0]
        f = self.encoder(torch.cat([left, right], 0))          # one pass for both hemispheres
        fl, fr = f[:B], f[B:]
        return torch.cat([fl + fr, (fl - fr).abs()], 1), fl, fr

    def forward(self, x, scalars=None):
        h, fl, fr = self.features(x)
        z_pair = self.pair_head(self.scalars(h, scalars)).squeeze(1)
        z_side = torch.maximum(self.side_head(fl).squeeze(1), self.side_head(fr).squeeze(1))
        return z_pair + z_side


def build_3d_h(n_scalars: int = 0, **kw) -> nn.Module:
    return HemiNet3D(n_scalars=n_scalars, **{k: v for k, v in kw.items() if k in ("widths", "dropout", "stem_stride", "split")})


In [ ]:
%%writefile datsc/models_v3.py
"""Run-3 model zoo: a GroupNorm/SiLU 3D ResNet and torchvision ImageNet backbones for 2.5D slabs.

Both expose forward(x, scalars) -> logits[B]. By default NO scalar features are used (n_scalars=0): the run-2
scalars fingerprinted the acquisition site (97% site-ID accuracy) and biased predictions under site shift.
2D backbones are built with weights=None; pretrained state_dicts are loaded from a local file when given.
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


def _gn(c: int, groups: int = 8) -> nn.GroupNorm:
    return nn.GroupNorm(min(groups, c), c)


class Block3D(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv3d(cin, cout, 3, stride, 1, bias=False); self.n1 = _gn(cout)
        self.c2 = nn.Conv3d(cout, cout, 3, 1, 1, bias=False); self.n2 = _gn(cout)
        self.skip = None
        if stride != 1 or cin != cout:
            self.skip = nn.Sequential(nn.Conv3d(cin, cout, 1, stride, bias=False), _gn(cout))

    def forward(self, x):
        y = F.silu(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return F.silu(y + (x if self.skip is None else self.skip(x)))


class ScalarHead(nn.Module):
    """Optional image-only scalar side-input; disabled when n_scalars == 0."""

    def __init__(self, n_scalars: int):
        super().__init__()
        self.n = n_scalars
        self.norm = nn.LayerNorm(n_scalars) if n_scalars else None

    def forward(self, h, scalars):
        if not self.n:
            return h
        return torch.cat([h, self.norm(scalars[:, : self.n])], 1)


class ResNet3D(nn.Module):
    def __init__(self, n_scalars: int = 0, widths=(24, 48, 96, 192), dropout: float = 0.3, stem_stride: int = 1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv3d(1, widths[0], 3, stem_stride, 1, bias=False), _gn(widths[0]), nn.SiLU())
        layers, cin = [], widths[0]
        for i, w in enumerate(widths):
            layers.append(Block3D(cin, w, stride=1 if i == 0 else 2)); cin = w
        self.layers = nn.Sequential(*layers)
        self.scalars = ScalarHead(n_scalars)
        self.feat_dim = cin
        self.head = nn.Sequential(nn.Linear(cin + n_scalars, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))

    def features(self, x):
        """Pooled features before the head (used by the DANN site head)."""
        return F.adaptive_avg_pool3d(self.layers(self.stem(x)), 1).flatten(1)

    def forward(self, x, scalars=None):
        return self.head(self.scalars(self.features(x), scalars)).squeeze(1)


class TinyNet2D(nn.Module):
    """~200k-parameter ResNet-style 2D net trained from scratch (the Hamburg DaT-SPECT recipe: small nets on a
    single striatal slab generalise across cameras). GroupNorm, SiLU, global average pooling."""

    def __init__(self, n_in: int = 3, widths=(16, 32, 64, 128)):
        super().__init__()
        def block(cin, cout, stride):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.GroupNorm(min(8, cout), cout), nn.SiLU(),
                                 nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.GroupNorm(min(8, cout), cout), nn.SiLU())
        layers, cin = [], n_in
        for i, w in enumerate(widths):
            layers.append(block(cin, w, 1 if i == 0 else 2)); cin = w
        self.features = nn.Sequential(*layers)
        self.out_dim = cin

    def forward(self, x):
        return F.adaptive_avg_pool2d(self.features(x), 1).flatten(1)


def _tv_backbone(name: str, weights_path: str | Path | None):
    if name == "tiny":
        m = TinyNet2D(); return m, m.out_dim
    import torchvision.models as tvm
    m = getattr(tvm, name)(weights=None)
    if weights_path:
        sd = torch.load(str(weights_path), map_location="cpu", weights_only=True)
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert not unexpected and all(k.startswith(("fc.", "classifier.")) for k in missing), (missing, unexpected)
    if name.startswith("convnext"):
        m.classifier[2] = nn.Identity(); dim = m.classifier[0].normalized_shape[0]
    elif name.startswith("efficientnet"):
        dim = m.classifier[1].in_features; m.classifier[1] = nn.Identity()
    elif name.startswith("resnet"):
        dim = m.fc.in_features; m.fc = nn.Identity()
    else:
        raise ValueError(name)
    return m, dim


class Net2D(nn.Module):
    """ImageNet backbone on n_in-channel slab images (+ optional scalars). Channel-count mismatch (n_in != 3) is
    handled by averaging the pretrained stem filters. Inputs are in SBR+1 / p75 units; in_scale maps [0, in_scale]
    to [0, 1] before ImageNet-style standardisation (8 keeps the striatal range: p99 of the central box > 3 in 93%
    of normals)."""
    MEAN, STD = 0.45, 0.225

    def __init__(self, backbone: str = "efficientnet_b0", weights_path=None, n_scalars: int = 0, n_in: int = 3,
                 dropout: float = 0.3, in_scale: float = 8.0):
        super().__init__()
        if backbone == "tiny":
            self.backbone = TinyNet2D(n_in=n_in); dim = self.backbone.out_dim
        else:
            self.backbone, dim = _tv_backbone(backbone, weights_path)
            if n_in != 3:
                self._adapt_stem(n_in)
        self.in_scale = in_scale
        self.scalars = ScalarHead(n_scalars)
        self.feat_dim = dim
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim + n_scalars, 1))

    def _adapt_stem(self, n_in: int):
        conv = next(mod for mod in self.backbone.modules() if isinstance(mod, nn.Conv2d) and mod.in_channels == 3)
        w = conv.weight.data
        new = nn.Conv2d(n_in, conv.out_channels, conv.kernel_size, conv.stride, conv.padding, bias=conv.bias is not None)
        new.weight.data = w.mean(1, keepdim=True).repeat(1, n_in, 1, 1) * (3.0 / n_in)
        if conv.bias is not None:
            new.bias.data = conv.bias.data.clone()
        for parent in self.backbone.modules():
            for name, child in parent.named_children():
                if child is conv:
                    setattr(parent, name, new); return

    def features(self, img):
        """Pooled backbone features before the head (used by the DANN site head)."""
        img = (img.clamp(0, self.in_scale) / self.in_scale - self.MEAN) / self.STD
        return self.backbone(img)

    def forward(self, img, scalars=None):
        return self.head(self.scalars(self.features(img), scalars)).squeeze(1)


class GradReverse(torch.autograd.Function):
    """Identity forward, reversed (and scaled) gradient backward: the DANN trick."""

    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = float(lam)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None


class SiteHead(nn.Module):
    """Pseudo-site classifier on pooled features behind a gradient-reversal layer. Trained jointly, never saved with the
    model: the backbone is pushed toward features that do not identify the camera/protocol."""

    def __init__(self, dim: int, n_sites: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.SiLU(), nn.Linear(hidden, n_sites))

    def forward(self, h, lam: float = 1.0):
        return self.net(GradReverse.apply(h, lam))


def build_model(arm: str, n_scalars: int = 0, **kw) -> nn.Module:
    if arm == "3d":
        if kw.get("arch") == "se":                       # run 6a: SE / stochastic-depth striatal-crop family (datsc.models_a)
            from .models_a import build_3d_a
            return build_3d_a(n_scalars=n_scalars, **kw)
        if kw.get("arch") == "hemi":                     # run 7: hemisphere-level shared encoder (datsc.models_h)
            from .models_h import build_3d_h
            return build_3d_h(n_scalars=n_scalars, **kw)
        return ResNet3D(n_scalars=n_scalars, **{k: v for k, v in kw.items() if k in ("widths", "dropout", "stem_stride")})
    if arm == "2d":
        return Net2D(n_scalars=n_scalars, **{k: v for k, v in kw.items() if k in ("backbone", "weights_path", "n_in", "dropout", "in_scale")})
    if arm == "tri":                                        # run 6b: tri-planar late fusion (datsc.models_b)
        from .models_b import TriPlanarNet
        return TriPlanarNet(n_scalars=n_scalars, **{k: v for k, v in kw.items() if k in ("backbone", "weights_path", "n_in", "n_views", "fusion", "dropout", "in_scale", "hidden")})
    raise ValueError(arm)


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())


In [ ]:
%%writefile datsc/preprocess.py
"""Shared preprocessing: brain mask, intensity normalisation, hotspot anchor, crop, scalars.

This module is imported unchanged by training scripts and by main.py.
Volumes are float32 arrays indexed [x, y, z] at config.SPACING_MM isotropic.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage as ndi

from .config import (ANCHOR_INTERIOR_MM, ANCHOR_MAX_DIST_MM, ANCHOR_MAX_ML, ANCHOR_MIN_ML,
                     ANCHOR_THRESH, CROP_SHAPE, FALLBACK_SUPERIOR_OFFSET_MM, N_SCALARS,
                     SMOOTH_MM, SPACING_MM)


def _otsu(values: np.ndarray) -> float:
    """Otsu threshold on a 1-D array (numpy only, no skimage dependency)."""
    v = values[np.isfinite(values)]
    if v.size == 0 or v.max() == v.min():
        return float(v.max()) if v.size else 0.0
    hist, edges = np.histogram(v, bins=256)
    centers = (edges[:-1] + edges[1:]) / 2
    w1 = np.cumsum(hist).astype(float)
    w2 = w1[-1] - w1
    m1 = np.cumsum(hist * centers) / np.maximum(w1, 1e-9)
    m2 = (np.cumsum((hist * centers)[::-1])[::-1] / np.maximum(w2, 1e-9))
    between = w1[:-1] * w2[:-1] * (m1[:-1] - m2[1:]) ** 2
    return float(centers[int(np.argmax(between))])


def _largest_component(mask: np.ndarray) -> np.ndarray:
    lab, n = ndi.label(mask)
    if n <= 1:
        return mask.astype(bool)
    sizes = ndi.sum(mask, lab, index=np.arange(1, n + 1))
    return lab == (int(np.argmax(sizes)) + 1)


def brain_mask(vol: np.ndarray, spacing_mm: float = SPACING_MM) -> np.ndarray:
    sm = ndi.gaussian_filter(vol, sigma=SMOOTH_MM / spacing_mm)
    thr = _otsu(sm.ravel())
    m = sm > thr
    if m.sum() < 50:                      # degenerate: fall back to "anything above 10% of max"
        m = sm > 0.1 * sm.max()
    m = _largest_component(m)
    return ndi.binary_fill_holes(m)


def normalize_p75(vol: np.ndarray, mask: np.ndarray):
    """Divide by the 75th percentile of masked voxels (template-free reference scaling)."""
    vals = vol[mask]
    p75 = float(np.percentile(vals, 75)) if vals.size else 0.0
    if p75 <= 0:
        p75 = float(np.percentile(vals, 99)) if vals.size else 0.0
    if p75 <= 0:
        p75 = float(vol.max()) if vol.max() > 0 else 1.0
    return (vol / p75).astype(np.float32), p75


def find_anchor(norm: np.ndarray, mask: np.ndarray, spacing_mm: float = SPACING_MM):
    """Hotspot anchor: x from the brain-mask midline, y/z from the brightest smoothed blob found
    in the deep-brain interior (> ANCHOR_INTERIOR_MM from the head-mask boundary).

    Restricting the search to the interior removes scalp/edge uptake as a candidate, which is
    what made the plain "brightest blob" fail preferentially on low-uptake (abnormal) scans.
    The interior hotspot is used unconditionally; the returned flag only records whether the
    component looked unusual (size/distance) so it can be fed to the models as a scalar.
    Returns ((cx, cy, cz) in voxel coords, flag). A CoM fallback is used only if the interior is empty.
    """
    vox_ml = spacing_mm ** 3 / 1000.0
    brain_c = np.array(ndi.center_of_mass(mask), dtype=float)
    fallback_c = brain_c.copy()
    fallback_c[2] += FALLBACK_SUPERIOR_OFFSET_MM / spacing_mm

    depth = ndi.distance_transform_edt(mask, sampling=spacing_mm)
    inner = depth > ANCHOR_INTERIOR_MM
    if inner.sum() < 50:                       # tiny/degenerate mask: use the whole mask
        inner = mask
    sm = ndi.gaussian_filter(norm * mask, sigma=SMOOTH_MM / spacing_mm)
    sm_in = np.where(inner, sm, 0.0)
    peak = float(sm_in.max())
    if not np.isfinite(peak) or peak <= 0:
        return tuple(fallback_c), True
    comp = _largest_component(sm_in > ANCHOR_THRESH * peak)
    if comp.sum() == 0:
        return tuple(fallback_c), True
    hot_c = np.array(ndi.center_of_mass(sm_in, labels=comp.astype(int), index=1), dtype=float)
    ml = comp.sum() * vox_ml
    dist = float(np.linalg.norm((hot_c - brain_c) * spacing_mm))
    flag = bool(ml < ANCHOR_MIN_ML or ml > ANCHOR_MAX_ML or dist > ANCHOR_MAX_DIST_MM)
    # x from the mask midline so a one-sided (asymmetric) hotspot cannot shift the crop laterally
    return (float(brain_c[0]), float(hot_c[1]), float(hot_c[2])), flag


def crop_at(vol: np.ndarray, center, shape=CROP_SHAPE) -> np.ndarray:
    out = np.zeros(shape, dtype=np.float32)
    starts = [int(round(c)) - s // 2 for c, s in zip(center, shape)]
    src, dst = [], []
    for st, s, n in zip(starts, shape, vol.shape):
        lo, hi = max(st, 0), min(st + s, n)
        if hi <= lo:
            return out
        src.append(slice(lo, hi))
        dst.append(slice(lo - st, hi - st))
    out[tuple(dst)] = vol[tuple(src)]
    return out


def _mirror_ncc(norm: np.ndarray, mask: np.ndarray, cx: float) -> float:
    """Normalised cross-correlation between the volume and its L/R mirror about x=cx."""
    n = norm.shape[0]
    shift = int(round(2 * cx - (n - 1)))
    mirrored = np.flip(norm, axis=0)
    mirrored = np.roll(mirrored, shift, axis=0)
    a, b = norm[mask], mirrored[mask]
    a = a - a.mean(); b = b - b.mean()
    den = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / den) if den > 0 else 0.0


def compute_scalars(raw_iso, norm, mask, p75, anchor, fallback, native_spacing, spacing_mm=SPACING_MM):
    vals = raw_iso[mask]
    p99 = float(np.percentile(vals, 99)) if vals.size else p75
    nv = norm[mask]
    sk = float(((nv - nv.mean()) ** 3).mean() / (nv.std() ** 3 + 1e-9)) if nv.size > 2 else 0.0
    zs = np.where(mask.any(axis=(0, 1)))[0]
    z_extent = float((zs.max() - zs.min() + 1) * spacing_mm) if zs.size else 0.0
    n_levels = int(np.unique(np.round(raw_iso[mask], 3)).size) if vals.size else 1
    s = np.array([
        np.log((p99 + 1e-6) / (p75 + 1e-6)),
        mask.sum() * spacing_mm ** 3 / 1000.0,
        sk,
        _mirror_ncc(norm, mask, anchor[0]),
        z_extent,
        float(np.mean(native_spacing)),
        np.log(n_levels),
        float(fallback),
    ], dtype=np.float32)
    assert s.shape == (N_SCALARS,)
    return np.nan_to_num(s)


def preprocess(path, spacing_mm: float = SPACING_MM) -> dict:
    """The single shared entrypoint. Returns dict with crop (fp16), crop_com (fp16), scalars, anchor, fallback."""
    from .io import load_iso  # local import keeps preprocess importable without nibabel at module load
    raw_iso, native_spacing = load_iso(path, spacing_mm)
    mask = brain_mask(raw_iso, spacing_mm)
    norm, p75 = normalize_p75(raw_iso, mask)
    anchor, fallback = find_anchor(norm, mask, spacing_mm)
    com = np.array(ndi.center_of_mass(mask), dtype=float)
    return {
        "crop": crop_at(norm, anchor).astype(np.float16),
        "crop_com": crop_at(norm, tuple(com)).astype(np.float16),
        "scalars": compute_scalars(raw_iso, norm, mask, p75, anchor, fallback, native_spacing, spacing_mm),
        "anchor": np.array(anchor, dtype=np.float32),
        "fallback": bool(fallback),
        "p75": float(p75),
    }


In [ ]:
%%writefile datsc/register.py
"""Spatial normalisation of CoM boxes to a study template (SimpleITK, 7-DOF similarity + optional affine refinement),
QC by normalised cross-correlation, and reference-region intensity normalisation in template space.

Everything operates on the 2 mm CoM box produced by datsc.volume.prepare_volume, so the same code runs in the cache
builder (from cached boxes) and in main.py (from a freshly loaded scan). SimpleITK is in the DrivenData runtime.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
from scipy import ndimage as ndi

from .volume import VOL_SHAPE, VOL_SPACING_MM

REG_STAT_NAMES = ("ncc_init", "ncc_reg", "reg_ok", "reg_scale", "reg_rot_deg", "reg_trans_mm", "ref_mean", "bg_level")
N_REG_STATS = len(REG_STAT_NAMES)


def _sitk():
    import SimpleITK as sitk
    sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(1)
    return sitk


def _to_sitk(box: np.ndarray, spacing: float = VOL_SPACING_MM):
    sitk = _sitk()
    img = sitk.GetImageFromArray(np.ascontiguousarray(box.astype(np.float32).transpose(2, 1, 0)))   # [x,y,z] -> [z,y,x]
    img.SetSpacing((spacing, spacing, spacing))
    return img


def _from_sitk(img) -> np.ndarray:
    return _sitk().GetArrayFromImage(img).transpose(2, 1, 0)


def prep_for_registration(box: np.ndarray, spacing: float = VOL_SPACING_MM, smooth_mm: float = 3.0) -> np.ndarray:
    x = ndi.gaussian_filter(box.astype(np.float32), smooth_mm / spacing)
    hi = float(np.percentile(x, 99.5))
    return np.clip(x, 0, hi) / max(hi, 1e-6)


def ncc(a: np.ndarray, b: np.ndarray, mask: np.ndarray | None = None) -> float:
    if mask is not None:
        a, b = a[mask], b[mask]
    a = a.astype(np.float64).ravel() - a.mean(); b = b.astype(np.float64).ravel() - b.mean()
    den = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / den) if den > 0 else 0.0


def register(moving_box: np.ndarray, template_box: np.ndarray, spacing: float = VOL_SPACING_MM,
             iterations: int = 100, affine_refine: bool = True, seed: int = 0, sampling: float = 0.15):
    """Register `moving_box` to `template_box` (both [X,Y,Z] at `spacing`). Returns (registered_box, params dict).
    Similarity (rotation, translation, isotropic scale) with moments initialisation, multi-resolution NCC at 8 and
    4 mm (a 10 mm-FWHM modality gains nothing from the 2 mm level); optional 12-DOF affine refinement at 4 mm."""
    sitk = _sitk()
    fixed = _to_sitk(prep_for_registration(template_box, spacing), spacing)
    moving = _to_sitk(prep_for_registration(moving_box, spacing), spacing)
    init = sitk.CenteredTransformInitializer(fixed, moving, sitk.Similarity3DTransform(), sitk.CenteredTransformInitializerFilter.MOMENTS)

    def run(initial, shrink, sigmas, iters, lr):
        R = sitk.ImageRegistrationMethod()
        R.SetMetricAsCorrelation()
        R.SetMetricSamplingStrategy(R.RANDOM); R.SetMetricSamplingPercentage(sampling, seed)
        R.SetInterpolator(sitk.sitkLinear)
        R.SetOptimizerAsGradientDescent(learningRate=lr, numberOfIterations=iters, convergenceMinimumValue=1e-6,
                                        convergenceWindowSize=10, estimateLearningRate=R.EachIteration)
        R.SetOptimizerScalesFromPhysicalShift()
        R.SetShrinkFactorsPerLevel(shrink); R.SetSmoothingSigmasPerLevel(sigmas); R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        R.SetInitialTransform(initial, inPlace=True)
        tx = R.Execute(fixed, moving)
        return tx, float(R.GetMetricValue())

    sim, m_sim = run(init, [4, 2], [3.0, 1.5], iterations, 1.0)
    tx = sim
    if affine_refine:
        aff = sitk.AffineTransform(3)
        aff.SetCenter(sim.GetCenter())
        aff.SetMatrix(sim.GetMatrix()); aff.SetTranslation(sim.GetTranslation())
        try:
            aff2, m_aff = run(aff, [2], [1.5], iterations // 2, 1.0)
            if m_aff <= m_sim:            # correlation metric is negative; keep the refinement only if it helped
                tx = aff2
        except RuntimeError:
            pass
    out = sitk.Resample(_to_sitk(moving_box, spacing), fixed, tx, sitk.sitkLinear, 0.0, sitk.sitkFloat32)
    p = np.array(sim.GetParameters(), float)     # versor(3), translation(3), scale
    rot_deg = float(np.degrees(2 * np.arcsin(min(1.0, np.linalg.norm(p[:3])))))
    params = {"scale": float(p[6]), "rot_deg": rot_deg, "trans_mm": float(np.linalg.norm(p[3:6])),
              "transform": tx.GetParameters(), "fixed_params": tx.GetFixedParameters(), "kind": tx.GetName()}
    return _from_sitk(out), params


def apply_transform(box: np.ndarray, kind: str, transform_params, fixed_params, spacing: float = VOL_SPACING_MM,
                    interpolator: str = "linear") -> np.ndarray:
    sitk = _sitk()
    tx = sitk.AffineTransform(3) if kind.startswith("Affine") else sitk.Similarity3DTransform()
    tx.SetFixedParameters(tuple(fixed_params)); tx.SetParameters(tuple(transform_params))
    ref = _to_sitk(np.zeros(VOL_SHAPE, np.float32), spacing)
    interp = sitk.sitkNearestNeighbor if interpolator == "nearest" else sitk.sitkLinear
    return _from_sitk(sitk.Resample(_to_sitk(box, spacing), ref, tx, interp, 0.0, sitk.sitkFloat32))


HIST_PCTS = (1, 5, 10, 25, 50, 75, 90, 95, 99)


def hist_landmarks(vol: np.ndarray, brain_mask: np.ndarray) -> np.ndarray:
    """Intensity percentiles inside `mask` used as landmarks for histogram standardisation. Use the reference region
    (brain minus striatum): standardising the striatum itself would erase the label signal."""
    return np.percentile(np.asarray(vol, np.float32)[brain_mask], HIST_PCTS).astype(np.float32)


def hist_standardise(vol: np.ndarray, brain_mask: np.ndarray, src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    """Piecewise-linear map of the scan's landmarks `src` onto the template's `dst` (Nyul-style), applied to every voxel;
    below the first landmark the first segment's slope is used, above the last landmark a pure shift (slope 1) so that the
    striatum (which sits above every reference-region landmark) keeps its absolute contrast. Monotonic by construction."""
    src = np.maximum.accumulate(np.asarray(src, np.float64)); dst = np.asarray(dst, np.float64)
    x = np.asarray(vol, np.float32).ravel()
    out = np.interp(x, src, dst).astype(np.float32)
    lo, hi = x < src[0], x > src[-1]
    if src[1] > src[0]:
        out[lo] = dst[0] + (x[lo] - src[0]) * (dst[1] - dst[0]) / (src[1] - src[0])
    out[hi] = dst[-1] + (x[hi] - src[-1])            # above the top landmark: shift only (slope 1) -> the striatal contrast
    return out.reshape(vol.shape)                      # over the standardised background is kept in absolute units



class Template:
    """Study template + derived masks. Stored as artifacts/template.npz (label-blind mean of registered training boxes)."""

    def __init__(self, box, brain_mask, striatal_mask, ref_mask, striatal_center, ncc_thresh: float, spacing: float = VOL_SPACING_MM, hist=None):
        self.box = box.astype(np.float32); self.brain_mask = brain_mask.astype(bool)
        self.hist = None if hist is None else np.asarray(hist, np.float32)     # histogram-standardisation landmarks (optional)
        self.striatal_mask = striatal_mask.astype(bool); self.ref_mask = ref_mask.astype(bool)
        self.striatal_center = np.asarray(striatal_center, float); self.ncc_thresh = float(ncc_thresh); self.spacing = spacing
        self.qc_mask = ndi.binary_erosion(self.brain_mask, iterations=2)

    def save(self, path):
        extra = {} if self.hist is None else {"hist": self.hist}
        np.savez_compressed(path, box=self.box.astype(np.float16), brain_mask=self.brain_mask, striatal_mask=self.striatal_mask,
                            ref_mask=self.ref_mask, striatal_center=self.striatal_center, ncc_thresh=self.ncc_thresh, spacing=self.spacing, **extra)

    @classmethod
    def load(cls, path):
        z = np.load(path)
        return cls(z["box"], z["brain_mask"], z["striatal_mask"], z["ref_mask"], z["striatal_center"], float(z["ncc_thresh"]), float(z["spacing"]),
                   hist=(z["hist"] if "hist" in z.files else None))


def derive_masks(template_box: np.ndarray, spacing: float = VOL_SPACING_MM, erode_mm: float = 8.0, dilate_mm: float = 20.0):
    """Brain mask (Otsu on the smoothed template, largest component), striatal mask (brightest interior blob of the
    template, which is label-blind because the template is), reference mask = eroded brain minus dilated striatum."""
    from .preprocess import _largest_component
    sm = ndi.gaussian_filter(template_box.astype(np.float32), 4.0 / spacing)
    hi = float(np.percentile(sm, 99.5))
    # the template is a smooth mean in p75 units (cortex ~1, striatum ~3): a fixed fraction of the peak range gives
    # the head/brain envelope; Otsu would isolate the striatal blob instead
    brain = ndi.binary_fill_holes(_largest_component(sm > 0.12 * hi))
    depth = ndi.distance_transform_edt(brain, sampling=spacing)
    inner = depth > 15.0
    sm_in = np.where(inner, sm, 0.0)
    peak = float(sm_in.max())
    hot = sm_in > 0.7 * peak
    # both hemispheres: everything above threshold within 40 mm of the centroid of ALL hot voxels (on the midline for a
    # symmetric template), then mirrored so the mask is exactly L/R symmetric
    c = np.array(ndi.center_of_mass(hot), float)
    idx = np.indices(template_box.shape).astype(float)
    dist = np.sqrt(sum(((idx[i] - c[i]) * spacing) ** 2 for i in range(3)))
    striatal = (sm_in > 0.6 * peak) & (dist < 40.0)
    striatal = striatal | np.flip(striatal, axis=0)
    striatal_center = np.array(ndi.center_of_mass(striatal), float)
    striatal_center[0] = (template_box.shape[0] - 1) / 2.0   # the template is symmetrised about the box centre
    eroded = ndi.binary_erosion(brain, iterations=max(1, int(round(erode_mm / spacing))))
    dil = ndi.binary_dilation(striatal, iterations=max(1, int(round(dilate_mm / spacing))))
    ref = eroded & ~dil
    return brain, striatal, ref, striatal_center


def symmetrize(T: np.ndarray, max_shift: int = 12):
    """Find the mid-sagittal plane of a template by maximising NCC with its x-mirror over integer shifts, roll the
    template so that plane sits at the box centre ((X-1)/2), and average with its mirror. Returns (T_sym, x_mid, delta)."""
    best = (-2.0, 0)
    for sh in range(-max_shift, max_shift + 1):
        m = np.roll(np.flip(T, axis=0), sh, axis=0)
        v = ncc(T, m)
        if v > best[0]:
            best = (v, sh)
    sh = best[1]
    x_mid = ((T.shape[0] - 1) + sh) / 2.0
    delta = int(round((T.shape[0] - 1) / 2.0 - x_mid))
    Tr = np.roll(T, delta, axis=0)
    return 0.5 * (Tr + np.flip(Tr, axis=0)), x_mid, delta


def register_and_normalise(box_p75: np.ndarray, template: Template, spacing: float = VOL_SPACING_MM, do_register: bool = True):
    """The shared per-volume step: register the CoM box to the template, QC, reference-normalise.
    Returns (vol_out float32 in SBR+1 units, reg_stats[N_REG_STATS], params dict)."""
    box = box_p75.astype(np.float32)
    ncc_init = ncc(prep_for_registration(box, spacing), prep_for_registration(template.box, spacing), template.qc_mask)
    reg, params, ncc_reg = box, {"scale": 1.0, "rot_deg": 0.0, "trans_mm": 0.0}, ncc_init
    ok = False
    if do_register:
        try:
            reg_try, params_try = register(box, template.box, spacing)
            ncc_try = ncc(prep_for_registration(reg_try, spacing), prep_for_registration(template.box, spacing), template.qc_mask)
            plausible = 0.6 <= params_try["scale"] <= 1.6 and params_try["rot_deg"] <= 45 and params_try["trans_mm"] <= 60
            if plausible and ncc_try >= max(template.ncc_thresh, ncc_init - 0.05):
                reg, params, ncc_reg, ok = reg_try, params_try, ncc_try, True
        except Exception:  # noqa: BLE001 - any registration failure falls back to the CoM frame
            pass
    outside = ~ndi.binary_dilation(template.brain_mask, iterations=3)
    bg = float(np.median(reg[outside])) if outside.any() else 0.0
    ref_vals = reg[template.ref_mask]
    ref_mean = float(ref_vals.mean() - bg) if ref_vals.size else 1.0
    if not np.isfinite(ref_mean) or ref_mean <= 1e-3:
        ref_mean = max(float(np.percentile(reg[template.brain_mask], 75) - bg), 1e-3)
    vol = np.clip(reg - bg, 0, None) / ref_mean
    stats = np.array([ncc_init, ncc_reg, float(ok), params["scale"], params["rot_deg"], params["trans_mm"], ref_mean, bg], np.float32)
    return vol.astype(np.float32), np.nan_to_num(stats), params


def load_template(path: str | Path | None) -> Template | None:
    return Template.load(path) if path and Path(path).exists() else None


In [ ]:
%%writefile datsc/ssl_c.py
"""Approach C: self-supervised pretraining of the striatal-crop 3D encoder on the TRAINING scans only (labels unused).

VICReg-style objective (invariance + variance + covariance; no negatives, so it is stable at batch 32 on a T4) between two
views of the same scan: two independent draws of our physical augmentations (flip, small affine, blur, count noise,
quantisation, reference-scale jitter) plus a random crop-centre jitter of +-2 voxels around the template striatal centre.
The encoder is the run-3 GroupNorm/SiLU ResNet3D (`features()`), so the pretrained weights load straight into the
supervised crop-3D arm (`scripts/train_v3.py --init`). Test scans are never touched (competition rule: per-sample
inference, no transductive use of the test set).

Why: contrastive / non-contrastive pretraining is the best-established way to get more out of ~1k labelled medical volumes
(Wolf et al., Sci. Rep. 2023: contrastive and masked-autoencoder pretraining for small medical datasets; VoCo, CVPR 2024:
volume contrast exploits the consistent geometry of registered anatomy -- our template frame gives that for free).
"""
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from .augment import train_augment
from .data_v3 import VolDataset, VolumeCache, to_3d_crop
from .models_v3 import ResNet3D
from .train_v3 import _grad_scaler


class Projector(nn.Module):
    def __init__(self, dim: int, hidden: int = 512, out: int = 256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Linear(hidden, out))

    def forward(self, h):
        return self.net(h)


def vicreg_loss(z1: torch.Tensor, z2: torch.Tensor, lam: float = 25.0, mu: float = 25.0, nu: float = 1.0):
    """Invariance (MSE) + variance hinge (std >= 1 per dimension) + covariance (off-diagonal -> 0)."""
    inv = F.mse_loss(z1, z2)
    def var_cov(z):
        z = z - z.mean(0)
        std = torch.sqrt(z.var(0) + 1e-4)
        v = torch.relu(1 - std).mean()
        n, d = z.shape
        cov = (z.T @ z) / max(n - 1, 1)
        off = cov - torch.diag(torch.diag(cov))
        return v, (off ** 2).sum() / d
    v1, c1 = var_cov(z1); v2, c2 = var_cov(z2)
    return lam * inv + mu * (v1 + v2) / 2 + nu * (c1 + c2), {"inv": float(inv), "var": float((v1 + v2) / 2), "cov": float(c1 + c2)}


def two_views(x: torch.Tensor, crop=(48, 48, 32), center=(47.5, 57.3, 33.5), jitter: int = 2, aug: str = "base"):
    """Two augmented striatal crops of the same batch of boxes [B,1,96,96,72]."""
    views = []
    for _ in range(2):
        xa = train_augment(x, aug)
        c = tuple(float(ci) + float(np.random.randint(-jitter, jitter + 1)) for ci in center)
        views.append(to_3d_crop(xa, crop, c))
    return views


def pretrain(cache_dir, out_path, epochs: int = 200, batch_size: int = 32, lr: float = 1e-3, weight_decay: float = 1e-4,
             widths=(24, 48, 96, 192), crop=(48, 48, 32), center=(47.5, 57.3, 33.5), jitter: int = 2, aug: str = "base",
             num_workers: int = 2, device=None, max_steps: int | None = None, seed: int = 0, norm: str = "ref", log_every: int = 1) -> dict:
    """VICReg pretraining on every scan of the cache (labels ignored). Saves {'encoder': state_dict, ...} to out_path."""
    cache_dir, out_path = Path(cache_dir), Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu"); amp = device.startswith("cuda")
    torch.manual_seed(seed); np.random.seed(seed)
    cache = VolumeCache(cache_dir)
    uids = list(cache.meta["uid"])
    dl = DataLoader(VolDataset(cache, uids, np.zeros(len(uids)), norm), batch_size=batch_size, shuffle=True, drop_last=True,
                    num_workers=num_workers, pin_memory=amp, persistent_workers=num_workers > 0)
    enc = ResNet3D(widths=tuple(widths)).to(device); proj = Projector(enc.feat_dim).to(device)
    params = list(enc.parameters()) + list(proj.parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    total = epochs * max(1, len(dl)) if max_steps is None else max_steps
    warm = max(1, min(total // 10, 5 * max(1, len(dl))))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * min(t - warm, total - warm) / max(1, total - warm))))
    scaler = _grad_scaler(amp)
    hist, step, t0 = [], 0, time.time()
    print(f"[ssl_c] scans={len(uids)} steps/epoch={len(dl)} epochs={epochs} encoder params={sum(p.numel() for p in enc.parameters())/1e6:.2f}M device={device}", flush=True)
    for ep in range(epochs):
        enc.train(); proj.train(); tl, n, parts = 0.0, 0, {"inv": 0.0, "var": 0.0, "cov": 0.0}
        for x, _, _, _ in dl:
            x = x.to(device, non_blocking=True)
            with torch.no_grad():
                v1, v2 = two_views(x, crop, center, jitter, aug)
            with torch.autocast(device_type="cuda", enabled=amp):
                z1 = proj(enc.features(v1)); z2 = proj(enc.features(v2))
            loss, p = vicreg_loss(z1.float(), z2.float())
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss.detach()); n += 1; step += 1
            for k in parts: parts[k] += p[k]
            if max_steps is not None and step >= max_steps:
                break
        rec = {"epoch": ep, "loss": tl / max(n, 1), **{k: v / max(n, 1) for k, v in parts.items()}, "sec": time.time() - t0}
        hist.append(rec)
        if ep % log_every == 0 or ep == epochs - 1:
            print(f"[ssl_c] ep{ep:03d} loss={rec['loss']:.3f} inv={rec['inv']:.3f} var={rec['var']:.3f} cov={rec['cov']:.3f} ({rec['sec']:.0f}s)", flush=True)
        if max_steps is not None and step >= max_steps:
            break
    state = {k: v.detach().cpu() for k, v in enc.state_dict().items()}
    torch.save({"encoder": state, "widths": list(widths), "crop3d": list(crop), "center3d": list(center), "epochs": epochs, "steps": step,
                "history": hist, "aug": aug, "jitter": jitter}, out_path)
    print(f"[ssl_c] saved encoder -> {out_path} after {step} steps in {time.time()-t0:.0f}s", flush=True)
    return {"steps": step, "history": hist, "out": str(out_path)}


@torch.no_grad()
def extract_features(encoder_path, cache_dir, uids, batch_size: int = 32, device=None, norm: str = "ref", num_workers: int = 0) -> np.ndarray:
    """Frozen-encoder features (identity + mirror averaged) of the striatal crop for the given uids -> [N, feat_dim]."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(Path(encoder_path), map_location="cpu", weights_only=False)
    enc = ResNet3D(widths=tuple(ck["widths"])); enc.load_state_dict(ck["encoder"]); enc.to(device).eval()
    cache = VolumeCache(cache_dir)
    dl = DataLoader(VolDataset(cache, list(uids), np.zeros(len(uids)), norm), batch_size=batch_size, shuffle=False, num_workers=num_workers)
    out = []
    for x, _, _, _ in dl:
        x = x.to(device)
        c = to_3d_crop(x, tuple(ck["crop3d"]), tuple(ck["center3d"]))
        h = 0.5 * (enc.features(c) + enc.features(c.flip(dims=[2])))
        out.append(h.float().cpu().numpy())
    return np.concatenate(out)


def linear_probe_oof(features: np.ndarray, y: np.ndarray, folds: np.ndarray, C: float = 0.1) -> np.ndarray:
    """Logistic-regression probe on frozen features, out-of-fold (reporting only; not shipped)."""
    from sklearn.linear_model import LogisticRegression
    p = np.zeros(len(y))
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        mu, sd = features[tr].mean(0), features[tr].std(0) + 1e-6
        clf = LogisticRegression(C=C, max_iter=3000).fit((features[tr] - mu) / sd, y[tr])
        p[va] = clf.predict_proba((features[va] - mu) / sd)[:, 1]
    return p


In [ ]:
%%writefile datsc/train.py
"""Train one (fold, seed) of SmallResNet3D from the crop cache; save state_dict + OOF predictions."""
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from .augment import train_augment
from .config import N_SCALARS
from .model import SmallResNet3D


class CropDataset(Dataset):
    def __init__(self, cache: Path, uids, labels, key: str = "crop"):
        self.cache, self.uids, self.labels, self.key = Path(cache), list(uids), np.asarray(labels, np.float32), key

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        z = np.load(self.cache / f"{self.uids[i]}.npz")
        x = torch.from_numpy(z[self.key].astype(np.float32))[None]
        s = torch.from_numpy(z["scalars"].astype(np.float32))
        return x, s, torch.tensor(self.labels[i]), self.uids[i]


def _seed_all(seed: int):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(model, loader, device, tta_flip: bool = True):
    model.eval(); ps, uids = [], []
    for x, s, _, u in loader:
        x, s = x.to(device), s.to(device)
        logit = model(x, s)
        if tta_flip:
            logit = 0.5 * (logit + model(x.flip(dims=[2]), s))
        ps.append(torch.sigmoid(logit).float().cpu().numpy()); uids += list(u)
    return np.concatenate(ps), uids


def train_fold(cache, fold: int, seed: int, out_dir, epochs: int = 40, batch_size: int = 16, lr: float = 1e-3,
               weight_decay: float = 1e-4, device: str | None = None, num_workers: int = 2,
               control: bool = False, amp: bool | None = None) -> dict:
    cache, out_dir = Path(cache), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    amp = (device.startswith("cuda")) if amp is None else amp
    _seed_all(seed)
    f = pd.read_csv(cache / "folds.csv", dtype={"uid": str})
    key = "crop_com" if control else "crop"
    tr, va = f[f["fold"] != fold], f[f["fold"] == fold]
    dl_tr = DataLoader(CropDataset(cache, tr["uid"], tr["is_pathologic"], key), batch_size=batch_size, shuffle=True,
                       num_workers=num_workers, drop_last=len(tr) > batch_size, pin_memory=device.startswith("cuda"))
    dl_va = DataLoader(CropDataset(cache, va["uid"], va["is_pathologic"], key), batch_size=batch_size, shuffle=False, num_workers=num_workers)
    model = SmallResNet3D(n_scalars=N_SCALARS).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps = max(1, epochs * len(dl_tr))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: 0.5 * (1 + math.cos(math.pi * min(t, steps) / steps)))
    scaler = torch.cuda.amp.GradScaler(enabled=amp)
    tag = f"f{fold}_s{seed}" + ("_com" if control else "")
    best, best_state, hist, t0 = float("inf"), None, [], time.time()
    for ep in range(epochs):
        model.train(); tl, n = 0.0, 0
        for x, s, y, _ in dl_tr:
            x, s, y = x.to(device), s.to(device), y.to(device)
            with torch.no_grad():
                x = train_augment(x)
            with torch.autocast(device_type="cuda", enabled=amp):
                loss = F.binary_cross_entropy_with_logits(model(x, s), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss) * len(y); n += len(y)
        p, uids = predict(model, dl_va, device, tta_flip=False)
        yv = va.set_index("uid").loc[uids, "is_pathologic"].values
        vl = float(-np.mean(yv * np.log(np.clip(p, 1e-6, 1)) + (1 - yv) * np.log(np.clip(1 - p, 1e-6, 1))))
        hist.append({"epoch": ep, "train_loss": tl / max(n, 1), "val_logloss": vl, "sec": time.time() - t0})
        print(f"[{tag}] ep{ep:02d} train={tl/max(n,1):.4f} val_logloss={vl:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if vl < best:
            best, best_state = vl, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    torch.save(best_state, out_dir / f"cnn_{tag}.pt")
    p, uids = predict(model, dl_va, device, tta_flip=True)
    yv = va.set_index("uid").loc[uids, "is_pathologic"].values
    pd.DataFrame({"uid": uids, "y": yv, "p": p}).to_csv(out_dir / f"oof_{tag}.csv", index=False)
    meta = {"fold": fold, "seed": seed, "control": control, "epochs": epochs, "best_val_logloss": best,
            "history": hist, "device": device, "amp": amp}
    (out_dir / f"cnn_{tag}.json").write_text(json.dumps(meta))
    return meta


In [ ]:
%%writefile datsc/train_b.py
"""Run-6b input spec: tri-planar view tensor for `TriPlanarNet`, drop-in for the run-3 trainer/inference contract.

`InputSpecB(x, dz)` -> [B, V, n_slabs, hw, hw]; `variants(x)` yields (identity + affine boxes) x z-shift x mirror exactly like
`InputSpec`, and `flip()` is the anatomically correct mirror: axial/coronal images flip their X axis, the two sagittal stacks
swap (the right stack is the left window taken on the mirrored volume, so the swap is exact). `make_spec(cfg)` builds the right spec class from a manifest/meta dict (arm "tri" -> InputSpecB, else InputSpec).
"""
from __future__ import annotations

import torch

from .data_v3 import to_slabs
from .data_b import to_slabs_sagittal
from .train_v3 import InputSpec

DEFAULT_VIEWS = ("axial", "coronal", "sagL", "sagR")


class InputSpecB:
    arm = "tri"

    def __init__(self, arm: str = "tri", n_slabs: int = 3, slab_mm: float = 8.0, hw: int = 160, slab_mode: str = "mean",
                 z_center: float = 33.5, y_center: float = 57.3, x_left: float = 36.8, x_right: float = 58.2,
                 views=DEFAULT_VIEWS, tta_z=(0,), crop=None, tta_affine: int = 0, **_ignored):
        assert arm == "tri"
        self.n_slabs, self.slab_mm, self.hw, self.slab_mode = int(n_slabs), float(slab_mm), int(hw), slab_mode
        self.z_center, self.y_center, self.x_left, self.x_right = float(z_center), float(y_center), float(x_left), float(x_right)
        self.views, self.tta_z, self.crop, self.tta_affine = tuple(views), tuple(tta_z), crop, int(tta_affine or 0)
        self._aff = InputSpec("3d", tta_affine=self.tta_affine)          # reuse the shared reproducible affine boxes

    # ---- one view ----
    def _view(self, x, name: str, dz: int):
        k = dict(n_slabs=self.n_slabs, slab_mm=self.slab_mm, hw=self.hw, mode=self.slab_mode, crop=self.crop)
        if name == "axial":
            return to_slabs(x, z_center=self.z_center + dz, plane="axial", **k)
        if name == "coronal":
            return to_slabs(x, z_center=self.y_center + dz, plane="coronal", **k)
        if name == "sagL":
            return to_slabs_sagittal(x, x_center=self.x_left, **k)
        if name == "sagR":                                              # the left window applied to the mirrored volume:
            return to_slabs_sagittal(x.flip(dims=[2]), x_center=self.x_left, **k)   # exact mirror of sagL, so flip == swap
        raise ValueError(name)

    def __call__(self, x, dz: int = 0):
        return torch.stack([self._view(x, v, dz) for v in self.views], dim=1)       # [B, V, C, hw, hw]

    def variants(self, x):
        boxes = [x] + (list(self._aff._affine_boxes(x)) if self.tta_affine else [])
        for b in boxes:
            for dz in self.tta_z:
                inp = self(b, dz)
                yield inp
                yield self.flip(inp)

    def flip(self, inp):
        """Mirror: axial/coronal views flip their last axis (X); sagittal-left <-> sagittal-right swap."""
        out = []
        for i, v in enumerate(self.views):
            if v == "sagL" and "sagR" in self.views:
                out.append(inp[:, self.views.index("sagR")])
            elif v == "sagR" and "sagL" in self.views:
                out.append(inp[:, self.views.index("sagL")])
            elif v in ("sagL", "sagR"):
                out.append(inp[:, i])                                              # lone sagittal view: unchanged
            else:
                out.append(inp[:, i].flip(dims=[-1]))
        return torch.stack(out, dim=1)

    def to_dict(self):
        return dict(arm="tri", n_slabs=self.n_slabs, slab_mm=self.slab_mm, hw=self.hw, slab_mode=self.slab_mode,
                    z_center=self.z_center, y_center=self.y_center, x_left=self.x_left, x_right=self.x_right,
                    views=list(self.views), tta_z=list(self.tta_z), crop=self.crop, tta_affine=self.tta_affine)


def make_spec(cfg: dict):
    """Spec object from a meta/manifest dict: arm 'tri' -> InputSpecB, '2d'/'3d' -> InputSpec."""
    cfg = dict(cfg)
    if cfg.get("arm") == "tri":
        return InputSpecB(**cfg)
    return InputSpec(**cfg)


In [ ]:
%%writefile datsc/train_v3.py
"""Run-3 trainer: one (arm, fold, seed) run from the memmap cache.

Protocol (fixes the run-2 optimism): fixed epoch budget, EMA weights, OOF and checkpoint from the FINAL EMA state
(no best-epoch selection on the validation fold), linear warmup + cosine, AdamW with real weight decay, optional
mixup, torch.amp API, loss.detach(). Model input (3D resize or axial slabs) is derived on the GPU AFTER the shared
3D augmentation so both arms see identical physical perturbations. AUROC is logged next to log loss every epoch.
"""
from __future__ import annotations

import copy
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from .augment import train_augment
from .data_v3 import load_site_ids, VolDataset, VolumeCache, to_3d_input, to_slabs, to_3d_crop
from .models_v3 import SiteHead, build_model, count_params


def logloss_np(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def auroc_np(y, p):
    return float(roc_auc_score(y, p)) if 0 < y.mean() < 1 else float("nan")


class InputSpec:
    """How a [B,1,96,96,72] box becomes model input, and which test-time variants are averaged.
    tta_z: slab-centre offsets in voxels for the 2D arm (e.g. [-1, 0, 1]); every variant is also L/R-flipped."""

    def __init__(self, arm: str, size=(64, 64, 48), n_slabs: int = 3, slab_mm: float = 12.0, hw: int = 192,
                 slab_mode: str = "mean", z_center: float | None = None, plane: str = "axial", tta_z=(0,), crop=None,
                 crop3d=None, center3d=None, tta_affine: int = 0):
        self.arm, self.size, self.n_slabs, self.slab_mm, self.hw, self.slab_mode, self.z_center = \
            arm, tuple(size), n_slabs, slab_mm, hw, slab_mode, z_center
        self.plane, self.tta_z, self.crop = plane, tuple(tta_z), crop
        self.crop3d = tuple(int(v) for v in crop3d) if crop3d else None        # 3D: striatal crop before the resize
        self.center3d = tuple(float(v) for v in center3d) if center3d else None
        self.tta_affine = int(tta_affine or 0)          # K reproducible small affine perturbations of the box at test time

    def __call__(self, x, dz: int = 0):
        if self.arm == "3d":
            if self.crop3d:
                x = to_3d_crop(x, self.crop3d, self.center3d)
            return to_3d_input(x, self.size)
        zc = None if self.z_center is None else self.z_center + dz
        return to_slabs(x, self.n_slabs, self.slab_mm, self.hw, mode=self.slab_mode, z_center=zc, plane=self.plane, crop=self.crop)

    def _affine_boxes(self, x):
        """K reproducible small affine perturbations of the registered box (rotation <= 3 deg, scale <= 2 %, shift <= 1 voxel):
        the same K boxes every call (fixed generator seed), so OOF and inference average identical variants."""
        g = torch.Generator(device="cpu").manual_seed(1234)
        for _ in range(self.tta_affine):
            ang = torch.deg2rad((torch.rand(3, generator=g) * 2 - 1) * 3.0)
            sc = 1 + (torch.rand(3, generator=g) * 2 - 1) * 0.02
            sh = (torch.rand(3, generator=g) * 2 - 1) * 2.0 / torch.tensor(x.shape[2:], dtype=torch.float32)
            cx, cy, cz = torch.cos(ang); sx, sy, sz = torch.sin(ang)
            Rx = torch.tensor([[1, 0, 0], [0, cx, -sx], [0, sx, cx]]); Ry = torch.tensor([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]]); Rz = torch.tensor([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
            theta = torch.zeros(x.shape[0], 3, 4); theta[:, :, :3] = (Rz @ Ry @ Rx) @ torch.diag(sc); theta[:, :, 3] = sh
            grid = F.affine_grid(theta.to(x.device, x.dtype), list(x.shape), align_corners=False)
            yield F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

    def variants(self, x):
        """Yield every TTA input ((identity + affine boxes) x z-shift x flip). Training uses only variant 0."""
        boxes = [x] + (list(self._affine_boxes(x)) if self.tta_affine else [])
        for b in boxes:
            for dz in (self.tta_z if self.arm == "2d" else (0,)):
                inp = self(b, dz)
                yield inp
                yield self.flip(inp)

    def flip(self, inp):
        """Left-right mirror of a model input (label-preserving)."""
        return inp.flip(dims=[2]) if self.arm == "3d" else inp.flip(dims=[3])

    def to_dict(self):
        return dict(arm=self.arm, size=list(self.size), n_slabs=self.n_slabs, slab_mm=self.slab_mm, hw=self.hw,
                    slab_mode=self.slab_mode, z_center=self.z_center, plane=self.plane, tta_z=list(self.tta_z), crop=self.crop,
                    crop3d=(list(self.crop3d) if self.crop3d else None), center3d=(list(self.center3d) if self.center3d else None),
                    tta_affine=self.tta_affine)


def _seed_all(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(model, loader, spec: InputSpec, device, tta_flip=True):
    """tta_flip=False: single plain pass (per-epoch monitoring). tta_flip=True: logit-average over spec.variants()."""
    model.eval(); ps, uids = [], []
    for x, s, _, u in loader:
        x, s = x.to(device, non_blocking=True), s.to(device)
        with torch.autocast(device_type="cuda", enabled=device.startswith("cuda")):
            if tta_flip:
                zs = [torch.nan_to_num(model(inp, s).float(), nan=0.0, posinf=30.0, neginf=-30.0) for inp in spec.variants(x)]
                z = torch.stack(zs).mean(0)
            else:
                z = torch.nan_to_num(model(spec(x), s).float(), nan=0.0, posinf=30.0, neginf=-30.0)
        ps.append(torch.sigmoid(z).cpu().numpy()); uids += list(u)
    return np.concatenate(ps), uids


class EMA:
    """Exponential moving average of the weights with a step-adaptive decay min(decay, (1+t)/(10+t)).
    Keep the window SHORT (2-3 epochs: 0.99 for ~35 steps/epoch, 0.995 for ~70): a 0.998 window of ~15 epochs
    averaged weights from when the model was still poor and produced compressed, under-confident logits
    (measured: val log loss 0.58 at AUROC 0.87 on the 2.5D arm)."""

    def __init__(self, model, decay=0.995):
        self.m = copy.deepcopy(model).eval(); self.decay = decay; self.t = 0
        for p in self.m.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.t += 1
        d = min(self.decay, (1 + self.t) / (10 + self.t))
        msd = model.state_dict()
        for k, v in self.m.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


def _grad_scaler(amp: bool):
    if hasattr(torch.amp, "GradScaler"):                 # torch >= 2.3 (Kaggle / runtime)
        return torch.amp.GradScaler("cuda", enabled=amp)
    return torch.cuda.amp.GradScaler(enabled=amp)        # local torch 2.2 fallback


def soft_targets(soft_csv, uids, labels, soft_lam: float = 0.0, mistake_w: float = 0.0):
    """Run 8 (ensemble-corrected members). From a table uid,p[,loss] of the ensemble's NESTED out-of-fold probability (and loss) per
    training scan: soft target y* = (1 - lam) * y + lam * p for uids in the table (the hard label otherwise) and a per-sample weight
    w = 1 + kappa * loss normalised to mean 1 over the training uids (all 1 when kappa == 0 or the table is missing).
    Returns (soft_map, w_map, n_found); soft_map is empty when lam == 0 so the caller can skip the lookup."""
    uids = list(uids); labels = np.asarray(labels, float)
    if not soft_csv or (soft_lam <= 0 and mistake_w <= 0):
        return {}, {}, 0
    t = pd.read_csv(soft_csv, dtype={"uid": str}).drop_duplicates("uid").set_index("uid")
    p = t["p"].reindex(uids); l = t["loss"].reindex(uids) if "loss" in t.columns else pd.Series(np.nan, index=uids)
    found = p.notna().values
    soft = {u: float((1 - soft_lam) * y + soft_lam * pp) for u, y, pp, ok in zip(uids, labels, p.values, found) if ok} if soft_lam > 0 else {}
    w = 1.0 + mistake_w * np.nan_to_num(l.values.astype(float), nan=0.0) if mistake_w > 0 else np.ones(len(uids))
    w = w / max(float(w.mean()), 1e-8)
    return soft, ({u: float(x) for u, x in zip(uids, w)} if mistake_w > 0 else {}), int(found.sum())


def train_run(cache_dir, fold: int, seed: int, out_dir, spec: InputSpec, model_kw: dict, tag: str,
              epochs=60, batch_size=16, lr=5e-4, backbone_lr=None, weight_decay=1e-2, warmup_epochs=3,
              mixup=0.0, ema_decay=0.995, norm="ref", folds_csv="folds.csv", num_workers=2, device=None,
              augment=True, save_fp16=True, select: str = "final", aug: str = "base", z_jitter: int = 0,
              dann: float = 0.0, dro: float = 0.0, init_path=None, loss: str = "bce",
              soft_csv=None, soft_lam: float = 0.0, mistake_w: float = 0.0) -> dict:
    cache_dir, out_dir = Path(cache_dir), Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu"); amp = device.startswith("cuda")
    torch.backends.cudnn.benchmark = True
    _seed_all(seed)
    cache = VolumeCache(cache_dir)
    f = pd.read_csv(cache_dir / folds_csv, dtype={"uid": str})
    f = f[f["uid"].isin(cache.index)]
    full = fold < 0                                        # fold -1: train on everything, no validation split
    tr, va = (f, f.iloc[:0]) if full else (f[f["fold"] != fold], f[f["fold"] == fold])
    tpl = None
    if norm == "hist":
        from .register import load_template
        tpl = load_template(next((p for p in (Path("artifacts/template.npz"), cache_dir / "template.npz", Path("artifacts_v3/template.npz")) if p.exists()), None))
        assert tpl is not None and getattr(tpl, "hist", None) is not None, "norm=hist needs artifacts/template.npz with landmarks (scripts/template_landmarks.py)"
    dl_tr = DataLoader(VolDataset(cache, tr["uid"], tr["is_pathologic"], norm, tpl), batch_size=batch_size, shuffle=True,
                       num_workers=num_workers, drop_last=True, pin_memory=amp, persistent_workers=num_workers > 0)
    dl_va = None if full else DataLoader(VolDataset(cache, va["uid"], va["is_pathologic"], norm, tpl), batch_size=batch_size, shuffle=False,
                                         num_workers=num_workers, persistent_workers=num_workers > 0)
    model = build_model(spec.arm, **model_kw).to(device)
    init_loaded = 0
    if init_path:                                          # warm start from a pretrained encoder (e.g. SSL, approach C): only backbone tensors
        sd = torch.load(str(init_path), map_location="cpu", weights_only=False)
        sd = sd.get("encoder", sd) if isinstance(sd, dict) else sd
        own = model.state_dict()
        keep = {k: v for k, v in sd.items() if k.startswith(("stem.", "layers.", "backbone.")) and k in own and own[k].shape == v.shape}
        model.load_state_dict({**own, **keep}); init_loaded = len(keep)
        print(f"[{tag}] init from {init_path}: {init_loaded} tensors loaded", flush=True)
    if spec.arm == "2d" and backbone_lr is not None:
        bb = [p for n, p in model.named_parameters() if n.startswith("backbone.")]
        rest = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
        opt = torch.optim.AdamW([{"params": bb, "lr": backbone_lr}, {"params": rest, "lr": lr}], weight_decay=weight_decay)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    # site-invariant training: pseudo-site ids of the training uids; DANN head (gradient reversal) and/or group-DRO weights
    site_map, n_sites = load_site_ids(cache_dir, tr["uid"], min_n=25)
    soft_map, w_map, n_soft = soft_targets(soft_csv, tr["uid"], tr["is_pathologic"], soft_lam, mistake_w)   # run 8: ensemble-corrected targets/weights
    site_head = SiteHead(model.feat_dim, n_sites).to(device) if (dann > 0 and n_sites > 1) else None
    if site_head is not None:
        opt.add_param_group({"params": list(site_head.parameters()), "lr": lr, "weight_decay": weight_decay})
    q_dro = torch.ones(n_sites, device=device) / n_sites
    steps_per_epoch = max(1, len(dl_tr)); total = epochs * steps_per_epoch; warm = max(1, warmup_epochs * steps_per_epoch)

    def lr_lambda(t):
        if t < warm:
            return (t + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * min(t - warm, total - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = _grad_scaler(amp)
    ema = EMA(model, ema_decay) if ema_decay else None
    yv_lookup = va.set_index("uid")["is_pathologic"]
    best, best_state, hist, t0 = float("inf"), None, [], time.time()
    print(f"[{tag}] params={count_params(model)/1e6:.2f}M train={len(tr)} val={len(va)} norm={norm} spec={spec.to_dict()} "
          f"model_kw={ {k: (str(v) if isinstance(v, Path) else v) for k, v in model_kw.items()} }", flush=True)
    n_skipped = 0; step = 0; loss_kind = loss
    for ep in range(epochs):
        model.train(); tl, n = 0.0, 0
        for x, s, y, u in dl_tr:
            x, s, y = x.to(device, non_blocking=True), s.to(device), y.to(device)
            site = torch.tensor([site_map.get(uid, 0) for uid in u], device=device)
            if soft_map:
                y = torch.tensor([soft_map.get(uid, float(yy)) for uid, yy in zip(u, y.tolist())], device=device, dtype=y.dtype)
            w = torch.tensor([w_map.get(uid, 1.0) for uid in u], device=device, dtype=torch.float32)
            with torch.no_grad():
                if augment:
                    x = train_augment(x, aug)
                dz = int(np.random.randint(-z_jitter, z_jitter + 1)) if (z_jitter and spec.arm in ("2d", "tri")) else 0
                inp = spec(x, dz)                                     # random slab-centre shift == the TTA offsets
                if mixup > 0:
                    lam = float(np.random.beta(mixup, mixup)); perm = torch.randperm(inp.shape[0], device=device)
                    inp = lam * inp + (1 - lam) * inp[perm]; s = lam * s + (1 - lam) * s[perm]; y = lam * y + (1 - lam) * y[perm]; w = lam * w + (1 - lam) * w[perm]
            with torch.autocast(device_type="cuda", enabled=amp):
                if site_head is None and not (dro > 0 and n_sites > 1):
                    logit = model(inp, s).float()
                    y_t = (0.9 * y + 0.1 * torch.sigmoid(logit).detach()) if loss_kind == "bootstrap" else y   # soft bootstrapping (Reed 2015): ambiguous labels pull less
                    loss = (F.binary_cross_entropy_with_logits(logit, y_t, reduction="none") * w).mean()
                else:
                    h = model.features(inp)
                    logit = model.head(model.scalars(h, s)).squeeze(1).float()
                    y_t = (0.9 * y + 0.1 * torch.sigmoid(logit).detach()) if loss_kind == "bootstrap" else y
                    per = F.binary_cross_entropy_with_logits(logit, y_t, reduction="none") * w
                    if dro > 0 and n_sites > 1:                       # group-DRO: exponentially up-weight the worst pseudo-sites
                        Lg = torch.zeros(n_sites, device=device).index_add_(0, site, per)
                        cnt = torch.zeros(n_sites, device=device).index_add_(0, site, torch.ones_like(per))
                        Lg = Lg / cnt.clamp_min(1); present = (cnt > 0).float()
                        with torch.no_grad():
                            q_dro = q_dro * torch.exp(dro * Lg.detach() * present); q_dro = q_dro / q_dro.sum()
                        loss = (q_dro * Lg * present).sum() / (q_dro * present).sum().clamp_min(1e-6)
                    else:
                        loss = per.mean()
                    if site_head is not None:                          # DANN: reversed-gradient site classification, ramped over the first third
                        lam_dann = dann * min(1.0, step / max(1, total // 3))
                        loss = loss + F.cross_entropy(site_head(h.float(), lam_dann), site)
            step += 1
            if not torch.isfinite(loss):                      # diverged batch (fp16 overflow): skip it instead of poisoning the weights
                n_skipped += 1; opt.zero_grad(set_to_none=True); continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            tl += float(loss.detach()) * len(y); n += len(y)
        eval_model = ema.m if ema is not None else model
        if full:
            hist.append({"epoch": ep, "train_loss": tl / max(n, 1), "sec": time.time() - t0})
            print(f"[{tag}] ep{ep:02d} train={tl/max(n,1):.4f} ({time.time()-t0:.0f}s)", flush=True)
            continue
        p, uids = predict(eval_model, dl_va, spec, device, tta_flip=False)
        yv = yv_lookup.loc[uids].values.astype(float)
        vl, va_auc = logloss_np(yv, p), auroc_np(yv, p)
        raw = ""
        rec = {"epoch": ep, "train_loss": tl / max(n, 1), "val_logloss": vl, "val_auroc": va_auc, "sec": time.time() - t0}
        if ema is not None:                                   # raw-model metrics next to the EMA ones (lag / collapse detector)
            p_raw, u_raw = predict(model, dl_va, spec, device, tta_flip=False)
            y_raw = yv_lookup.loc[u_raw].values.astype(float)
            rec["val_raw_logloss"], rec["val_raw_auroc"] = logloss_np(y_raw, p_raw), auroc_np(y_raw, p_raw)
            raw = f" raw={rec['val_raw_logloss']:.4f}/{rec['val_raw_auroc']:.4f}"
        hist.append(rec)
        print(f"[{tag}] ep{ep:02d} train={tl/max(n,1):.4f} val_logloss={vl:.4f} val_auroc={va_auc:.4f}{raw} ({time.time()-t0:.0f}s)", flush=True)
        if vl < best:
            best = vl
            if select == "best":
                best_state = {k: v.detach().cpu().clone() for k, v in eval_model.state_dict().items()}
    eval_model = ema.m if ema is not None else model
    if select == "best" and best_state is not None:
        eval_model.load_state_dict(best_state)
    state = {k: v.detach().cpu().clone() for k, v in eval_model.state_dict().items()}
    if full:
        p, yv = np.zeros(0), np.zeros(0)
    else:
        p, uids = predict(eval_model, dl_va, spec, device, tta_flip=True)
        yv = yv_lookup.loc[uids].values.astype(float)
        pd.DataFrame({"uid": uids, "y": yv, "p": p}).to_csv(out_dir / f"oof_{tag}.csv", index=False)
    if save_fp16:
        state = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in state.items()}
    torch.save(state, out_dir / f"{tag}.pt")
    meta = {"tag": tag, "fold": fold, "seed": seed, "epochs": epochs, "select": select, "best_val_logloss": best,
            "val_logloss": (logloss_np(yv, p) if len(yv) else None), "val_auroc": (auroc_np(yv, p) if len(yv) else None),
            "spec": spec.to_dict(), "model_kw": {k: (str(v) if isinstance(v, Path) else v) for k, v in model_kw.items()},
            "norm": norm, "folds_csv": folds_csv, "mixup": mixup, "ema_decay": ema_decay, "lr": lr, "backbone_lr": backbone_lr,
            "aug": aug, "z_jitter": z_jitter, "skipped_batches": n_skipped, "dann": dann, "dro": dro, "n_sites": n_sites, "loss": loss_kind,
            "init": (str(init_path) if init_path else None), "init_loaded": init_loaded,
            "soft_csv": (str(soft_csv) if soft_csv else None), "soft_lam": soft_lam, "mistake_w": mistake_w, "n_soft": n_soft,
            "weight_decay": weight_decay, "history": hist, "params": count_params(model), "sec": time.time() - t0}
    (out_dir / f"{tag}.json").write_text(json.dumps(meta))
    if full:
        print(f"[{tag}] done (full-data model, no validation) in {time.time()-t0:.0f}s", flush=True)
    else:
        print(f"[{tag}] done final val_logloss={meta['val_logloss']:.4f} auroc={meta['val_auroc']:.4f} (best epoch {best:.4f}) in {time.time()-t0:.0f}s", flush=True)
    return meta


In [ ]:
%%writefile datsc/volume.py
"""Crop-agnostic whole-brain volume representation (run 3).

One `prepare_volume(path)` call yields a 2 mm isotropic, RAS-oriented, brain-CoM-centred box of the
p75-normalised volume plus the head mask and per-volume statistics. Every downstream input (tight
crop, whole-brain crop, axial slabs, alternative intensity normalisations) is derived from this box,
so the expensive NIfTI load + resample happens once per scan in training AND in main.py.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage as ndi

from .config import N_SCALARS
from .preprocess import brain_mask, compute_scalars, crop_at, normalize_p75

VOL_SPACING_MM = 2.0
VOL_SHAPE = (96, 96, 72)          # [x, y, z] voxels at 2 mm = 192 x 192 x 144 mm, centred on the head CoM
STAT_NAMES = ("p75", "p50", "p99", "mean", "std", "mask_ml", "com_x", "com_y", "com_z",
              "sp_x", "sp_y", "sp_z", "shape_x", "shape_y", "shape_z", "n_levels_native")
N_STATS = len(STAT_NAMES)


def prepare_volume(path, spacing_mm: float = VOL_SPACING_MM, shape=VOL_SHAPE, template=None) -> dict:
    """Returns dict(vol fp16 [X,Y,Z], mask bool [X,Y,Z], scalars[N_SCALARS], stats[N_STATS (+N_REG_STATS)]).
    Without a template: the CoM box in p75 units. With a datsc.register.Template: the box is registered to the
    template (QC'd, CoM fallback) and expressed in reference-region units; registration stats are appended."""
    from .io import load_canonical, resample_iso
    raw, native_spacing, native_shape = load_canonical(path)
    n_levels_native = int(np.unique(raw[raw > 0]).size) if raw.size and (raw > 0).any() else 1
    raw_iso = resample_iso(raw, native_spacing, spacing_mm)
    del raw
    mask = brain_mask(raw_iso, spacing_mm)
    norm, p75 = normalize_p75(raw_iso, mask)
    vals = raw_iso[mask]
    if vals.size == 0:
        vals = raw_iso.ravel()
    com = np.array(ndi.center_of_mass(mask), dtype=float) if mask.any() else np.array(raw_iso.shape, float) / 2
    scalars = compute_scalars(raw_iso, norm, mask, p75, (float(com[0]), float(com[1]), float(com[2])), False,
                              native_spacing, spacing_mm)
    stats = np.array([
        p75, float(np.percentile(vals, 50)), float(np.percentile(vals, 99)), float(vals.mean()), float(vals.std()),
        float(mask.sum()) * spacing_mm ** 3 / 1000.0, com[0], com[1], com[2],
        native_spacing[0], native_spacing[1], native_spacing[2], native_shape[0], native_shape[1], native_shape[2],
    ], dtype=np.float32)
    box = crop_at(norm, tuple(com), shape)
    mbox = crop_at(mask.astype(np.float32), tuple(com), shape) > 0.5
    stats = np.concatenate([np.nan_to_num(stats), [n_levels_native]]).astype(np.float32)
    if template is None:
        return {"vol": box.astype(np.float16), "mask": mbox, "scalars": scalars.astype(np.float32), "stats": stats}
    from .register import register_and_normalise
    vol, reg_stats, params = register_and_normalise(box, template, spacing_mm)
    return {"vol": vol.astype(np.float16), "mask": mbox, "scalars": scalars.astype(np.float32),
            "stats": np.concatenate([stats, reg_stats]).astype(np.float32), "reg_params": params}


def renormalise(vol_p75: np.ndarray, stats: np.ndarray, mode: str = "p75", template=None) -> np.ndarray:
    """Re-derive alternative intensity normalisations from the p75-scaled box + stored raw statistics.
    mode "hist": reference-normalised box mapped onto the template's brain-mask intensity landmarks (needs `template`
    with `.brain_mask` and `.hist`, see datsc.register.hist_standardise)."""
    p75, p50, p99, mean, std = [float(stats[i]) for i in range(5)]
    x = vol_p75.astype(np.float32)
    if mode in ("p75", "ref"):           # "ref": the cache already stores reference-normalised registered boxes
        return x
    if mode == "hist":
        from .register import hist_landmarks, hist_standardise
        assert template is not None and getattr(template, "hist", None) is not None, "hist mode needs a template with landmarks"
        return hist_standardise(x, template.ref_mask, hist_landmarks(x, template.ref_mask), template.hist)   # landmarks from the reference region only: the striatal contrast is preserved
    if mode == "median":                 # background-reference (SBR-like) scaling
        return x * (p75 / max(p50, 1e-6))
    if mode == "zscore":                 # brain-mask z-score
        return (x * p75 - mean) / max(std, 1e-6)
    if mode == "p99":
        return x * (p75 / max(p99, 1e-6))
    raise ValueError(mode)


assert N_SCALARS == 8


In [ ]:
%%writefile scripts/alignment_gate.py
"""OOF logistic regression on hotspot radiomics + scalars. PASS iff OOF AUROC > 0.90.
Also writes artifacts/lr.json (fit on all data) and cache/oof_lr.csv."""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score

from datsc.lrmodel import LRModel, feature_matrix

GATE_AUROC = 0.90

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--artifacts", default="artifacts")
    ap.add_argument("--C", type=float, default=0.3)
    a = ap.parse_args()
    cache, art = Path(a.cache), Path(a.artifacts)
    art.mkdir(exist_ok=True)
    f = pd.read_csv(cache / "folds.csv", dtype={"uid": str})
    X = feature_matrix(cache, f["uid"]); y = f["is_pathologic"].values.astype(int)
    oof = np.zeros(len(y))
    for k in sorted(f["fold"].unique()):
        tr, va = (f["fold"] != k).values, (f["fold"] == k).values
        oof[va] = LRModel(C=a.C).fit(X[tr], y[tr]).predict_proba(X[va])
    auc, ll = roc_auc_score(y, oof), log_loss(y, np.clip(oof, 1e-4, 1 - 1e-4))
    pd.DataFrame({"uid": f["uid"], "y": y, "p_lr": oof}).to_csv(cache / "oof_lr.csv", index=False)
    LRModel(C=a.C).fit(X, y).save(art / "lr.json")
    print(f"OOF AUROC={auc:.4f}  OOF logloss={ll:.4f}  (prior-only logloss={log_loss(y, np.full(len(y), y.mean())):.4f})")
    print("GATE:", "PASS" if auc > GATE_AUROC else "FAIL - fix the anchor before training any CNN")


In [ ]:
%%writefile scripts/anchor_diag.py
"""Why does find_anchor fall back? Per-scan hotspot component stats on a random sample.

python scripts/anchor_diag.py --n 400 --workers 10
"""
from __future__ import annotations

import argparse
import os
import sys
from multiprocessing import get_context
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np
import pandas as pd
from scipy import ndimage as ndi

from datsc import config as C
from datsc.config import find_data_dir, find_nifti_path
from datsc.io import load_iso
from datsc.preprocess import _largest_component, brain_mask, normalize_p75


def _one(args):
    uid, path, y = args
    try:
        raw, sp = load_iso(path)
        mask = brain_mask(raw)
        norm, p75 = normalize_p75(raw, mask)
        s = C.SPACING_MM
        brain_c = np.array(ndi.center_of_mass(mask))
        sm = ndi.gaussian_filter(norm * mask, sigma=C.SMOOTH_MM / s)
        peak = float(sm.max())
        res = {"uid": uid, "y": y, "spacing": float(np.mean(sp)), "peak": peak, "shape_z": raw.shape[2]}
        for thr in (0.5, 0.6, 0.7, 0.8):
            comp = _largest_component(sm > thr * peak)
            ml = comp.sum() * s ** 3 / 1000
            hc = np.array(ndi.center_of_mass(sm, labels=comp.astype(int), index=1))
            d = float(np.linalg.norm((hc - brain_c) * s))
            res[f"ml@{thr}"] = ml; res[f"dist@{thr}"] = d
            res[f"dz@{thr}"] = float((hc[2] - brain_c[2]) * s)
        return res
    except Exception as e:  # noqa: BLE001
        return {"uid": uid, "y": y, "err": repr(e)[:120]}


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=400)
    ap.add_argument("--workers", type=int, default=10)
    a = ap.parse_args()
    d = find_data_dir()
    labels = pd.read_csv(d.labels_csv, dtype={"uid": str}).sample(n=a.n, random_state=0)
    jobs = [(u, find_nifti_path(d.niftis_dir, u), y) for u, y in zip(labels["uid"], labels["is_pathologic"])]
    jobs = [j for j in jobs if j[1] is not None]
    with get_context("spawn").Pool(a.workers) as pool:
        rows = list(pool.imap_unordered(_one, jobs, chunksize=4))
    df = pd.DataFrame(rows)
    print("errors:", df["err"].notna().sum() if "err" in df else 0)
    df = df[df.get("err").isna()] if "err" in df else df
    pd.set_option("display.width", 200)
    for thr in (0.5, 0.6, 0.7, 0.8):
        ml, dist = df[f"ml@{thr}"], df[f"dist@{thr}"]
        rej = (ml < C.ANCHOR_MIN_ML) | (ml > C.ANCHOR_MAX_ML) | (dist > C.ANCHOR_MAX_DIST_MM)
        print(f"\nthr={thr}: ml p10/50/90 = {ml.quantile(.1):.1f}/{ml.median():.1f}/{ml.quantile(.9):.1f} mL | "
              f"dist p50/90 = {dist.median():.1f}/{dist.quantile(.9):.1f} mm | dz p50 = {df[f'dz@{thr}'].median():+.1f} mm | "
              f"reject(current rule)={rej.mean():.1%}  (too big {(ml > C.ANCHOR_MAX_ML).mean():.1%}, too small {(ml < C.ANCHOR_MIN_ML).mean():.1%}, too far {(dist > C.ANCHOR_MAX_DIST_MM).mean():.1%})")
        print("   by label:", df.assign(rej=rej).groupby("y")["rej"].mean().round(3).to_dict(),
              "| by spacing:", df.assign(rej=rej, sp=df.spacing.round(1)).groupby("sp")["rej"].mean().round(2).to_dict())
    df.to_csv("cache_anchor_diag.csv", index=False)


In [ ]:
%%writefile scripts/build_master_v2.py
"""Master v2: the 5 pillars with EQUAL weights (nested 0.2354 vs 0.2370 fitted), temperature from the leave-site-out pool
(T = 0.9, between the pool optimum 0.83 and the old floor 1.0), all hemisphere/SSL seeds and two V2-S seeds (the seed-0-only
pruning of master_final cost 0.011 on the V2-S arm). python scripts/build_master_v2.py [--T 0.9] [--lam 0.02] [--out data/submission_master_v2.zip]"""
import argparse, glob, json, zipfile
from pathlib import Path

ap = argparse.ArgumentParser(); ap.add_argument("--T", type=float, default=0.9); ap.add_argument("--lam", type=float, default=0.02)
ap.add_argument("--out", default="data/submission_master_v2.zip"); ap.add_argument("--v2s-seeds", default="01")
a = ap.parse_args()
ROOT = Path("."); master_art = Path("data/master_pool/artifacts_v3")
man = json.loads((master_art / "manifest.json").read_text())
keep = {"a3d_hemi": None, "a3d_ssl": None, "featc": None, "a2d_cnx_cor": None, "a2d_v2s": [f"_s{s}." for s in a.v2s_seeds]}
arms = []
for arm in man["arms"]:
    if arm["name"] not in keep:
        continue
    if keep[arm["name"]]:
        arm["checkpoints"] = [c for c in arm["checkpoints"] if any(k in c for k in keep[arm["name"]]) and "_fall_" not in c]
    else:
        arm["checkpoints"] = [c for c in arm["checkpoints"] if "_site" not in c]
    arms.append(arm)
man["arms"] = arms; man["weights"] = {x["name"]: 1.0 / len(arms) for x in arms}
man["T"], man["lam"], man["clip"], man["hedge"], man["typ"] = float(a.T), float(a.lam), None, None, None
man["note"] = "master v2: equal weights over 5 pillars; T from the leave-site-out pool; all hemi/SSL seeds; V2-S seeds " + a.v2s_seeds
sources = [Path("data/run6c/artifacts_v3"), Path("data/kaggle_output_v5/artifacts_v3"), Path("data/run6b/artifacts_v3"), Path("data/ckpt_ds_v4/artifacts_v3")]
zips = [zipfile.ZipFile(p) for p in ("data/submission_run7.zip", "data/submission_run5.zip") if Path(p).exists()]
def find(name):
    for d in sources:
        if (d / name).exists():
            return (d / name).read_bytes()
    for z in zips:
        m = [i for i in z.infolist() if i.filename.endswith("/" + name) or i.filename == name]
        if m:
            return z.read(m[0])
    raise FileNotFoundError(name)
out = Path(a.out); n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zo:
    zo.write(ROOT / "main_v3.py", "main.py"); zo.write(ROOT / "tests_synth.py", "tests_synth.py")
    for p in sorted((ROOT / "datsc").glob("*.py")):
        zo.write(p, f"datsc/{p.name}")
    zo.writestr("artifacts/template.npz", find("template.npz"))
    for arm in arms:
        for ck in arm["checkpoints"]:
            zo.writestr(ck, find(Path(ck).name)); n += 1
    zo.writestr("artifacts/manifest.json", json.dumps(man, indent=1))
print(f"wrote {out}: {n} checkpoint files across {len(arms)} arms; {out.stat().st_size/1e6:.0f} MB; weights {man['weights']}; T {man['T']} lam {man['lam']}")
for arm in arms:
    print(f"  {arm['name']:<12} {len(arm['checkpoints'])} ckpts")


In [ ]:
%%writefile scripts/build_master_v3.py
"""Master v3 = Master v2's five pillars + the run-8 arms that pass the nested gate, all seeds, equal weights, T 0.9.

python scripts/build_master_v3.py [--src data/run8/artifacts_v3 ...] [--smoke <smoke_dir>] [--zip data/submission_master_v3.zip]
                                  [--gate 0.003] [--T 0.9] [--lam 0.02] [--max-mb 850] [--no-e2e] [--mode equal|shrunk] [--allow-trim]

Prints the gate table (nested estimate, weights + T refit per fold): Master v2 as shipped, pillars with every local seed, pillars + each
run-8 arm, pillars + all run-8 arms, greedy best subset; builds the zip for the best set; runs scripts/local_e2e_v3.py on the smoke set.
The smoke set is never fitted on (ADR 0005)."""
import argparse, glob, json, subprocess, sys, zipfile
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path(__file__).resolve().parent.parent; sys.path.insert(0, str(ROOT))
src = open(ROOT / "scripts" / "fit_stack_v3.py", encoding="utf-8").read().split('if __name__ == "__main__":')[0]
ns = {"__name__": "fs", "__file__": str(ROOT / "scripts" / "fit_stack_v3.py")}; exec(src, ns)
logloss, sigmoid, logit, fit_temperature, compositions = ns["logloss"], ns["sigmoid"], ns["logit"], ns["fit_temperature"], ns["compositions"]

PILLARS = {"a3d_hemi": "r3d_hemi48_f?_s?", "a3d_ssl": "r3d_ssl_f?_s?", "featc": "featcarm", "a2d_cnx_cor": "r2d_convnext_tiny_cor_f?_s?", "a2d_v2s": "r2d_efficientnet_v2_s_f?_s[01]"}
V2_SHIPPED = {**PILLARS, "a3d_hemi": "r3d_hemi48_f?_s[012]"}
NEW = {"a3d_hemisoft": "r3d_hemisoft_f?_s?", "a3d_hemimw": "r3d_hemimw_f?_s?", "a3d_crophistsoft": "r3d_crophistsoft_f?_s?"}
TEMPLATE_ARM = {"a3d_hemisoft": "a3d_hemi", "a3d_hemimw": "a3d_hemi", "a3d_crophistsoft": "a3d_crophist"}   # manifest entry to copy
DROP_ORDER = [("a2d_v2s", "_s1."), ("a3d_hemimw", ""), ("a3d_crophistsoft", ""), ("a3d_hemi", "_s4."), ("a3d_hemi", "_s3.")]   # size trimming


def load_arm(srcs, pat):
    fs = sorted(set(f for d in srcs for f in glob.glob(str(Path(d) / f"oof_{pat}.csv")) if "_site" not in Path(f).name and "_fall" not in Path(f).name))
    if not fs:
        return None, []
    df = pd.concat([pd.read_csv(f, dtype={"uid": str}) for f in fs])
    return df.groupby("uid").agg(y=("y", "first"), p=("p", "mean")), fs


class Pool:
    def __init__(self, srcs, cache, T_floor, lam, pi=0.548, grid=8):
        self.srcs, self.T_floor, self.lam, self.pi, self.grid = srcs, T_floor, lam, pi, grid
        self.oofs, self.files = {}, {}
        for n, p in {**V2_SHIPPED, **{k + "@all": v for k, v in PILLARS.items()}, **NEW}.items():
            o, fs = load_arm(srcs, p)
            if o is not None and len(o) >= 1300:
                self.oofs[n], self.files[n] = o, fs
        self.uids = sorted(set.intersection(*[set(o.index) for o in self.oofs.values()]))
        self.y = next(iter(self.oofs.values())).loc[self.uids, "y"].values.astype(float)
        self.folds = pd.read_csv(Path(cache) / "folds.csv", dtype={"uid": str}).set_index("uid").loc[self.uids, "fold"].values

    def Z(self, names):
        return np.stack([logit(self.oofs[n].loc[self.uids, "p"].values) for n in names])

    def fit_w(self, Zs, ys):
        best = (9, None)
        for comb in compositions(self.grid, Zs.shape[0]):
            w = np.array(comb, float) / self.grid; ll = logloss(ys, sigmoid((w[:, None] * Zs).sum(0)))
            if ll < best[0]:
                best = (ll, w)
        return best[1]

    def nested(self, names, mode="equal"):
        Zs, y, folds = self.Z(names), self.y, self.folds; pn = np.zeros(len(y))
        for k in np.unique(folds):
            tr, va = folds != k, folds == k
            we = np.ones(len(names)) / len(names)
            w = we if mode == "equal" else (lambda wf: wf if mode == "fitted" else 0.5 * (wf + we))(self.fit_w(Zs[:, tr], y[tr]))
            T = max(fit_temperature((w[:, None] * Zs[:, tr]).sum(0), y[tr]), self.T_floor)
            pn[va] = (1 - self.lam) * sigmoid((w[:, None] * Zs[:, va]).sum(0) / T) + self.lam * self.pi
        return logloss(y, pn)


def gate_table(pool, gate, mode):
    have_new = [n for n in NEW if n in pool.oofs]; pil = [n + "@all" for n in PILLARS if n + "@all" in pool.oofs]
    rows = []
    def add(label, names):
        rows.append((label, names, pool.nested(names, "equal"), pool.nested(names, "shrunk"), pool.nested(names, "fitted")))
    add("Master v2 as shipped (hemi s0-2, v2s s0-1)", [n for n in V2_SHIPPED if n in pool.oofs])
    add("pillars, every local seed", pil)
    for n in have_new:
        add(f"pillars + {n}", pil + [n])
    if len(have_new) > 1:
        add("pillars + all run-8 arms", pil + have_new)
    # greedy forward selection from the pillars by equal-weight nested
    cur, best = list(pil), pool.nested(pil, "equal"); left = list(have_new)
    while left:
        cand = [(pool.nested(cur + [n], "equal"), n) for n in left]; ll, n = min(cand)
        if ll >= best - 1e-4:
            break
        cur.append(n); left.remove(n); best = ll
    add("greedy best subset", cur)
    print("\nGATE TABLE (nested estimate; equal / shrunk / fitted weights; T floor %.2f; lam %.2f)" % (pool.T_floor, pool.lam))
    for label, names, e, s, f in rows:
        print(f"  {label:<48} equal {e:.4f}  shrunk {s:.4f}  fitted {f:.4f}   [{', '.join(n.replace('@all', '') for n in names)}]")
    v2 = rows[0][2]; col = {"equal": 2, "shrunk": 3, "fitted": 4}[mode]
    best_row = min(rows[1:], key=lambda r: r[col]); gain = v2 - best_row[col]
    print(f"\nbest set by {mode} nested: {best_row[0]} = {best_row[col]:.4f}; Master v2 shipped = {v2:.4f}; gain {gain:+.4f} -> GATE: {'PASS' if gain >= gate else 'FAIL (keep Master v2 for slot 1)'}")
    return best_row, gain >= gate


def build_zip(pool, names, mode, T, lam, out, max_mb, note, allow_trim=False):
    master = json.loads((ROOT / "data" / "master_pool" / "artifacts_v3" / "manifest.json").read_text())
    by_name = {a["name"]: a for a in master["arms"]}
    if "a3d_crophist" not in by_name and Path("data/submission_run5.zip").exists():   # template for crophistsoft lives in the run-5 manifest
        z = zipfile.ZipFile("data/submission_run5.zip"); m5 = json.loads(z.read([n for n in z.namelist() if n.endswith("manifest.json")][0]))
        by_name.update({a["name"]: a for a in m5["arms"] if a["name"] == "a3d_crophist"})
    arms = []
    for n in names:
        base = n.replace("@all", ""); tpl = by_name[TEMPLATE_ARM.get(base, base)]
        arm = json.loads(json.dumps(tpl)); arm["name"] = base
        arm["checkpoints"] = sorted(set("artifacts/" + Path(f).name.replace("oof_", "").replace(".csv", ".pt") for f in pool.files[n])) if base != "featc" else tpl["checkpoints"]
        arms.append(arm)
    weights = {a["name"]: 1.0 / len(arms) for a in arms}
    if mode != "equal":
        wf = pool.fit_w(pool.Z(names), pool.y); we = np.ones(len(names)) / len(names); w = wf if mode == "fitted" else 0.5 * (wf + we)
        weights = {a["name"]: float(x) for a, x in zip(arms, w)}
    man = {**master, "arms": arms, "weights": weights, "T": float(T), "lam": float(lam), "clip": None, "hedge": None, "typ": None, "note": note}
    sources = [Path(d) for d in pool.srcs] + [Path(p) for p in ("data/run6c/artifacts_v3", "data/kaggle_output_v5/artifacts_v3", "data/run6b/artifacts_v3", "data/ckpt_ds_v4/artifacts_v3")]
    zips = [zipfile.ZipFile(p) for p in ("data/submission_run7.zip", "data/submission_run5.zip", "data/submission_master_v2.zip") if Path(p).exists()]
    def find(name):
        for d in sources:
            if (d / name).exists():
                return (d / name).read_bytes()
        for z in zips:
            m = [i for i in z.infolist() if i.filename.endswith("/" + name) or i.filename == name]
            if m:
                return z.read(m[0])
        raise FileNotFoundError(name)
    def write(man):
        n = 0
        with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zo:
            zo.write(ROOT / "main_v3.py", "main.py"); zo.write(ROOT / "tests_synth.py", "tests_synth.py")
            for p in sorted((ROOT / "datsc").glob("*.py")):
                zo.write(p, f"datsc/{p.name}")
            zo.writestr("artifacts/template.npz", find("template.npz"))
            for arm in man["arms"]:
                for ck in arm["checkpoints"]:
                    zo.writestr(ck, find(Path(ck).name)); n += 1
            zo.writestr("artifacts/manifest.json", json.dumps(man, indent=1))
        return n, Path(out).stat().st_size / 1e6
    n, mb = write(man)
    trimmed = []
    for arm_name, key in DROP_ORDER:
        if mb <= max_mb:
            break
        for arm in man["arms"]:
            if arm["name"] == arm_name:
                before = len(arm["checkpoints"]); arm["checkpoints"] = [c for c in arm["checkpoints"] if key and key not in c]
                if key == "":
                    man["arms"] = [a for a in man["arms"] if a["name"] != arm_name]; man["weights"].pop(arm_name, None)
                    s = sum(man["weights"].values()); man["weights"] = {k: v / s for k, v in man["weights"].items()}
                print(f"zip {mb:.0f} MB > {max_mb}: dropped {arm_name} {key or '(whole arm)'} ({before} -> {len(arm['checkpoints'])} ckpts)")
                trimmed.append(f"{arm_name} {key or '(whole arm)'}")
        n, mb = write(man)
    if mb > max_mb:
        raise SystemExit(f"FATAL: {out} is {mb:.0f} MB > --max-mb {max_mb} and DROP_ORDER is exhausted. "
                         f"Raise --max-mb or extend DROP_ORDER; do NOT submit this zip.")
    if trimmed and not allow_trim:
        raise SystemExit(f"FATAL: the zip fitted only after trimming {trimmed}. The gate table above scored the "
                         f"UNTRIMMED set, and dropping an arm also renormalises the weights, so what would ship is "
                         f"not what was measured. Raise --max-mb and rebuild, or pass --allow-trim to accept it "
                         f"and re-read the gate table for the trimmed set.")
    print(f"wrote {out}: {n} checkpoint files across {len(man['arms'])} arms, {mb:.0f} MB; weights {json.dumps({k: round(v, 3) for k, v in man['weights'].items()})}; T {man['T']} lam {man['lam']}")
    for arm in man["arms"]:
        print(f"  {arm['name']:<16} {len(arm['checkpoints'])} ckpts")
    return man


if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--src", action="append", default=None); ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--smoke", default=None); ap.add_argument("--zip", default="data/submission_master_v3.zip"); ap.add_argument("--gate", type=float, default=0.003)
    ap.add_argument("--T", type=float, default=0.9); ap.add_argument("--lam", type=float, default=0.02); ap.add_argument("--max-mb", type=float, default=850)
    ap.add_argument("--mode", default="equal", choices=["equal", "shrunk", "fitted"]); ap.add_argument("--no-e2e", action="store_true"); ap.add_argument("--table-only", action="store_true")
    ap.add_argument("--allow-trim", action="store_true", help="accept a zip that only fitted after DROP_ORDER trimming; by default that is fatal, because the gate table scored the untrimmed set")
    a = ap.parse_args()
    srcs = a.src or ["data/run8/artifacts_v3", "data/master_pool/artifacts_v3"]
    pool = Pool(srcs, a.cache, a.T, a.lam)
    print(f"pool: {len(pool.uids)} scans; arms with complete OOF: {sorted(pool.oofs)}")
    best_row, passed = gate_table(pool, a.gate, a.mode)
    if a.table_only:
        sys.exit(0)
    man = build_zip(pool, best_row[1], a.mode, a.T, a.lam, a.zip, a.max_mb, f"master v3: {best_row[0]}; nested {a.mode} {best_row[2 if a.mode == 'equal' else 3 if a.mode == 'shrunk' else 4]:.4f}; gate {'PASS' if passed else 'FAIL'}", a.allow_trim)
    if a.smoke and not a.no_e2e:
        print("container-style check on the smoke set ...", flush=True)
        r = subprocess.run([sys.executable, str(ROOT / "scripts" / "local_e2e_v3.py"), a.zip, a.smoke], capture_output=True, text=True)
        print(r.stdout[-1500:]); print(r.stderr[-500:]); sys.exit(r.returncode)


In [ ]:
%%writefile scripts/build_template.py
"""Build the label-blind study template from cached CoM boxes (cache_v3), iteratively:
  T0 = mean of all CoM boxes -> register every box to T_k -> T_{k+1} = mean of registered boxes  (x n_iter)
Then derive masks (brain / striatal / reference), set the NCC QC threshold from the training distribution and save
artifacts/template.npz. Also writes <cache>/template_ncc.csv for QC.

python scripts/build_template.py --cache cache_v3 --out artifacts/template.npz [--n-iter 2] [--subset 400] [--workers 8]
"""
from __future__ import annotations

import argparse
import os
import sys
import time
from multiprocessing import get_context
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd

from datsc.register import Template, derive_masks, ncc, prep_for_registration, register
from datsc.volume import VOL_SHAPE

_T = None


def _init(template_path):
    global _T
    _T = np.load(template_path)


def _reg_one(args):
    i, box = args
    try:
        reg, p = register(box, _T, affine_refine=False)
        return i, reg.astype(np.float16), p["scale"], p["rot_deg"], p["trans_mm"], ""
    except Exception as e:  # noqa: BLE001
        return i, box.astype(np.float16), 1.0, 0.0, 0.0, repr(e)[:100]


def register_all(vols, idx, template, workers, tmp):
    np.save(tmp, template.astype(np.float32))
    out = np.zeros((len(idx),) + VOL_SHAPE, np.float16); stats = np.zeros((len(idx), 3), np.float32); errs = 0
    t0 = time.time()
    with get_context("spawn").Pool(workers, initializer=_init, initargs=(tmp,)) as pool:
        for k, (i, reg, s, r, t, err) in enumerate(pool.imap_unordered(_reg_one, ((j, np.asarray(vols[i], np.float32)) for j, i in enumerate(idx)), chunksize=4)):
            out[i] = reg; stats[i] = (s, r, t); errs += bool(err)
            if (k + 1) % 200 == 0:
                print(f"  {k+1}/{len(idx)} registered in {time.time()-t0:.0f}s", flush=True)
    print(f"  registered {len(idx)} boxes in {time.time()-t0:.0f}s; errors {errs}; scale p10/50/90 = "
          f"{np.percentile(stats[:,0],[10,50,90]).round(3)} rot p50/90 = {np.percentile(stats[:,1],[50,90]).round(1)} deg", flush=True)
    return out, stats


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache_v3")
    ap.add_argument("--out", default="artifacts/template.npz")
    ap.add_argument("--n-iter", type=int, default=2)
    ap.add_argument("--subset", type=int, default=400, help="boxes used for template building (label-blind random subset)")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 2))
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    cache = Path(a.cache)
    vols = np.load(cache / "vols.npy", mmap_mode="r")
    meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
    rng = np.random.RandomState(a.seed)
    idx = np.sort(rng.choice(np.where(meta.ok.values)[0], size=min(a.subset, int(meta.ok.sum())), replace=False))
    print(f"template subset: {len(idx)} boxes (abnormal fraction {meta.is_pathologic.values[idx].mean():.3f})")
    T = np.mean([np.asarray(vols[i], np.float32) for i in idx], axis=0)
    tmp = str(cache / "_template_tmp.npy")
    for it in range(a.n_iter):
        print(f"iteration {it+1}/{a.n_iter}")
        reg, st = register_all(vols, idx, T, a.workers, tmp)
        T = reg.astype(np.float32).mean(axis=0)     # boxes are already CoM-centred; no re-centring (it drifted onto the striatum)
    from scipy import ndimage as ndi
    from datsc.register import symmetrize
    T, x_mid, delta = symmetrize(T)
    print(f"symmetrised: mirror midline found at x={x_mid:.1f} vox, rolled by {delta:+d} so the midline is the box centre")
    brain, striatal, ref, s_center = derive_masks(T)
    assert brain.sum() * 8 / 1000 > 1000 and striatal.sum() * 8 / 1000 > 3, (brain.sum(), striatal.sum())
    xs = np.arange(T.shape[0])[:, None, None]
    print(f"striatal voxels left/right of the midline: {(striatal & (xs < s_center[0])).sum()} / {(striatal & (xs > s_center[0])).sum()}")
    print(f"template masks: brain {brain.sum()*8/1000:.0f} mL, striatal {striatal.sum()*8/1000:.1f} mL, reference {ref.sum()*8/1000:.0f} mL, "
          f"striatal centre (vox) {np.round(s_center, 1)}")
    # QC threshold: register the subset once more to the final template and take the 2nd percentile of NCC
    reg, st = register_all(vols, idx, T, a.workers, tmp)
    qc = ~ndi.binary_erosion(~brain, iterations=1) & ndi.binary_erosion(brain, iterations=2)
    tp = prep_for_registration(T)
    nccs = np.array([ncc(prep_for_registration(np.asarray(reg[k], np.float32)), tp, qc) for k in range(len(idx))])
    ncc0 = np.array([ncc(prep_for_registration(np.asarray(vols[i], np.float32)), tp, qc) for i in idx])
    thresh = float(np.percentile(nccs, 2))
    print(f"NCC to template: unregistered p10/50/90 = {np.percentile(ncc0,[10,50,90]).round(3)}; registered p2/10/50/90 = "
          f"{np.percentile(nccs,[2,10,50,90]).round(3)} -> ncc_thresh {thresh:.3f}")
    Template(T, brain, striatal, ref, s_center, thresh).save(a.out)
    pd.DataFrame({"uid": meta.uid.values[idx], "ncc_init": ncc0, "ncc_reg": nccs, "scale": st[:, 0], "rot_deg": st[:, 1], "trans_mm": st[:, 2]}).to_csv(cache / "template_ncc.csv", index=False)
    os.remove(tmp)
    print(f"wrote {a.out}")


In [ ]:
%%writefile scripts/cache_crops.py
"""Preprocess every training volume once -> cache/{uid}.npz + cache/meta.csv.

Usage: python scripts/cache_crops.py [--out cache] [--workers 10] [--limit N]
"""
from __future__ import annotations

import argparse
import os
import sys
import time
from multiprocessing import get_context
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np
import pandas as pd

from datsc.config import find_data_dir, find_nifti_path
from datsc.features import hotspot_radiomics
from datsc.preprocess import preprocess


def _one(args):
    uid, path, out_dir = args
    try:
        r = preprocess(path)
        rad = hotspot_radiomics(r["crop"].astype(np.float32))
        np.savez_compressed(out_dir / f"{uid}.npz", crop=r["crop"], crop_com=r["crop_com"],
                            scalars=r["scalars"], radiomics=rad, anchor=r["anchor"])
        return {"uid": uid, "ok": True, "fallback": r["fallback"], "p75": r["p75"],
                "native_spacing_mm": float(r["scalars"][5]), "z_extent_mm": float(r["scalars"][4]),
                "n_levels": float(np.exp(r["scalars"][6])), "err": ""}
    except Exception as e:  # noqa: BLE001
        return {"uid": uid, "ok": False, "fallback": True, "p75": np.nan, "native_spacing_mm": np.nan,
                "z_extent_mm": np.nan, "n_levels": np.nan, "err": repr(e)[:200]}


def build_cache(labels_csv: Path, niftis_dir: Path, out_dir: Path, n_workers: int = 8, limit: int | None = None):
    out_dir.mkdir(parents=True, exist_ok=True)
    labels = pd.read_csv(labels_csv, dtype={"uid": str})
    if limit:
        labels = labels.head(limit)
    jobs = []
    for uid in labels["uid"]:
        p = find_nifti_path(niftis_dir, uid)
        if p is None:
            print(f"missing file for uid (skipping): {uid}")
            continue
        jobs.append((uid, p, out_dir))
    t0 = time.time()
    if n_workers <= 1:
        results = [_one(j) for j in jobs]
    else:
        with get_context("spawn").Pool(n_workers) as pool:
            results = list(pool.imap_unordered(_one, jobs, chunksize=4))
    meta = pd.DataFrame(results).merge(labels, on="uid", how="left")
    meta.to_csv(out_dir / "meta.csv", index=False)
    print(f"cached {meta['ok'].sum()}/{len(meta)} in {time.time()-t0:.0f}s; "
          f"fallback anchors: {int(meta['fallback'].sum())}; failures: {int((~meta['ok']).sum())}")
    return meta


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="cache")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 2))
    ap.add_argument("--limit", type=int, default=None)
    a = ap.parse_args()
    d = find_data_dir()
    print(f"data: {d.root}\nlabels: {d.labels_csv}\nniftis: {d.niftis_dir}")
    build_cache(d.labels_csv, d.niftis_dir, Path(a.out), a.workers, a.limit)


In [ ]:
%%writefile scripts/cache_volumes.py
"""Build the run-3 whole-brain cache: one memmap-able shard set instead of per-uid npz files.

Outputs in --out:
  vols.npy    float16 [N, 96, 96, 72]   p75-normalised 2 mm boxes (see datsc/volume.py)
  masks.npy   uint8   [N, 96, 96, 72]   head mask (0/1)
  scalars.npy float32 [N, 8]            the run-2 scalar contract (datsc.config.SCALAR_NAMES)
  stats.npy   float32 [N, 15]           raw-intensity + geometry statistics (datsc.volume.STAT_NAMES)
  meta.csv    uid, is_pathologic, ok, err, + stats columns   (row order == array order)

Usage: python scripts/cache_volumes.py [--out cache_v3] [--workers 8] [--limit N]
"""
from __future__ import annotations

import argparse
import os
import sys
import time
from multiprocessing import get_context
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np
import pandas as pd

from datsc.config import N_SCALARS, find_data_dir, find_nifti_path
from datsc.volume import N_STATS, STAT_NAMES, VOL_SHAPE, prepare_volume


def _one(args):
    i, uid, path = args
    try:
        r = prepare_volume(path)
        return i, uid, r, ""
    except Exception as e:  # noqa: BLE001
        return i, uid, None, repr(e)[:200]


def build(labels_csv: Path, niftis_dir: Path, out: Path, n_workers: int, limit: int | None):
    out.mkdir(parents=True, exist_ok=True)
    labels = pd.read_csv(labels_csv, dtype={"uid": str})
    if limit:
        labels = labels.head(limit)
    jobs = []
    for uid in labels["uid"]:
        p = find_nifti_path(niftis_dir, uid)
        if p is None:
            print(f"missing file for uid (skipping): {uid}")
            continue
        jobs.append((len(jobs), uid, p))
    N = len(jobs)
    vols = np.lib.format.open_memmap(out / "vols.npy", mode="w+", dtype=np.float16, shape=(N,) + VOL_SHAPE)
    masks = np.lib.format.open_memmap(out / "masks.npy", mode="w+", dtype=np.uint8, shape=(N,) + VOL_SHAPE)
    scalars = np.zeros((N, N_SCALARS), np.float32)
    stats = np.zeros((N, N_STATS), np.float32)
    ok = np.zeros(N, bool); errs = [""] * N; uids = [j[1] for j in jobs]
    t0 = time.time(); done = 0
    it = map(_one, jobs) if n_workers <= 1 else get_context("spawn").Pool(n_workers).imap_unordered(_one, jobs, chunksize=2)
    for i, uid, r, err in it:
        done += 1
        if r is not None:
            vols[i] = r["vol"]; masks[i] = r["mask"].astype(np.uint8)
            scalars[i] = r["scalars"]; stats[i] = r["stats"]; ok[i] = True
        else:
            errs[i] = err
        if done % 100 == 0 or done == N:
            print(f"{done}/{N} in {time.time()-t0:.0f}s", flush=True)
    vols.flush(); masks.flush()
    np.save(out / "scalars.npy", scalars); np.save(out / "stats.npy", stats)
    meta = pd.DataFrame({"uid": uids, "ok": ok, "err": errs})
    for k, name in enumerate(STAT_NAMES):
        meta[name] = stats[:, k]
    meta = meta.merge(labels, on="uid", how="left")
    meta.to_csv(out / "meta.csv", index=False)
    print(f"cached {int(ok.sum())}/{N} volumes in {time.time()-t0:.0f}s; failures: {int((~ok).sum())}")
    if (~ok).any():
        print(meta.loc[~ok, ["uid", "err"]].head(20).to_string(index=False))
    return meta


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="cache_v3")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 2))
    ap.add_argument("--limit", type=int, default=None)
    a = ap.parse_args()
    d = find_data_dir()
    print(f"data: {d.root}\nlabels: {d.labels_csv}\nniftis: {d.niftis_dir}")
    build(d.labels_csv, d.niftis_dir, Path(a.out), a.workers, a.limit)


In [ ]:
%%writefile scripts/diagnose_hard_core.py
"""Is the hard core irreducible label noise, or a shared preprocessing failure? One command decides.

    python scripts/diagnose_hard_core.py
    python scripts/diagnose_hard_core.py --pool kaggle/run8_distill/soft_master_v2.csv --cache cache_v4 --top 88

The endgame analysis found 88 scans (6.5 % of the pool) carrying 46.9 % of all loss and wrong under every
representation tried. That was read as irreducible label noise. But every arm shares the SAME upstream
preprocessing -- the registered 2 mm frame, the head box, the reference-region scaling -- so "wrong everywhere"
has a second explanation: one shared upstream defect. The published check only showed the errors do not cluster
by FOLD, and folds are random, so it could not have detected a site effect.

This script tests the alternative directly: do the hard-core scans concentrate in a pseudo-site, an intensity
regime, a voxel spacing, or the scans where registration fell back to the CoM frame? Enrichment is scored with
a label-permutation test, so no scipy is needed and the multiple-comparison problem is handled by permuting the
max statistic rather than testing each group separately.

Reads only cached artefacts. No GPU, no training, no submission. Needs numpy + pandas.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
RNG = np.random.default_rng(0)


def permutation_max_enrichment(flag, groups, n_perm=20000):
    """p-value for 'some group is enriched in flag', correcting for the number of groups by permuting the max."""
    codes, _ = pd.factorize(groups)
    k = codes.max() + 1
    n = np.bincount(codes, minlength=k).astype(float)
    obs_counts = np.bincount(codes, weights=flag, minlength=k)
    base = flag.mean()
    with np.errstate(divide="ignore", invalid="ignore"):
        obs = np.nanmax(np.where(n >= 10, obs_counts / n / base, np.nan))
    hits, f = 0, flag.copy()
    for _ in range(n_perm):
        RNG.shuffle(f)
        c = np.bincount(codes, weights=f, minlength=k)
        with np.errstate(divide="ignore", invalid="ignore"):
            m = np.nanmax(np.where(n >= 10, c / n / base, np.nan))
        hits += m >= obs
    return float(obs), (hits + 1) / (n_perm + 1)


def report_group(name, df, col, n_perm):
    if col not in df.columns or df[col].nunique() < 2:
        print(f"\n[{name}] column '{col}' absent or constant -- skipped")
        return None
    g = df[col].astype(str).fillna("?")
    base = df.hard.mean()
    tab = df.groupby(g).agg(scans=("hard", "size"), hard=("hard", "sum"), mean_loss=("loss", "mean"))
    tab["rate"] = tab.hard / tab.scans
    tab["enrich"] = tab["rate"] / base
    tab = tab.sort_values("enrich", ascending=False)
    print(f"\n[{name}]  overall hard-core rate {base:.3f}")
    print(f"  {'group':<22} {'scans':>6} {'hard':>5} {'rate':>7} {'enrich':>7} {'mean loss':>10}")
    for k, r in tab.iterrows():
        mark = "  <-- " if r.enrich >= 1.5 and r.scans >= 10 else ""
        print(f"  {k[:22]:<22} {int(r.scans):>6} {int(r.hard):>5} {r.rate:>7.3f} {r.enrich:>7.2f} {r.mean_loss:>10.4f}{mark}")
    obs, p = permutation_max_enrichment(df.hard.values.astype(float), g.values, n_perm)
    print(f"  max enrichment {obs:.2f}x over a group with >=10 scans; permutation p = {p:.4f}")
    return obs, p, tab


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pool", default="kaggle/run8_distill/soft_master_v2.csv", help="nested OOF with uid,y,p[,loss]")
    ap.add_argument("--cache", default="cache_v4", help="folder holding sites.csv / folds.csv")
    ap.add_argument("--top", type=int, default=88, help="hard core = this many worst-loss scans")
    ap.add_argument("--n-perm", type=int, default=20000)
    a = ap.parse_args()

    pool = Path(a.pool if Path(a.pool).is_absolute() else ROOT / a.pool)
    df = pd.read_csv(pool, dtype={"uid": str})
    if "loss" not in df.columns:
        p = np.clip(df.p.values, 1e-15, 1 - 1e-15)
        df["loss"] = -(df.y * np.log(p) + (1 - df.y) * np.log(1 - p))
    df = df.sort_values("loss", ascending=False).reset_index(drop=True)
    df["hard"] = (np.arange(len(df)) < a.top).astype(int)
    share = df.loc[df.hard == 1, "loss"].sum() / df.loss.sum()
    print(f"pool {pool.name}: {len(df)} scans; hard core = worst {a.top} "
          f"({100 * a.top / len(df):.1f} % of scans, {100 * share:.1f} % of total loss)")

    cache = Path(a.cache if Path(a.cache).is_absolute() else ROOT / a.cache)
    sites = cache / "sites.csv"
    if not sites.exists():
        print(f"\nFATAL: {sites} not found. This script must run on the training machine, where cache_v4/ exists.")
        print("Without it the site question cannot be answered and the 'irreducible noise' reading stays unverified.")
        raise SystemExit(2)
    s = pd.read_csv(sites, dtype={"uid": str})
    df = df.merge(s, on="uid", how="left")
    print(f"joined {sites.name}: columns {[c for c in s.columns if c != 'uid']}")

    results = {}
    for label, col in [("PSEUDO-SITE", "site"), ("INTENSITY REGIME", "regime"), ("VOXEL SPACING", "spacing"),
                       ("GRID SHAPE", "shape"), ("REGISTRATION FALLBACK", "fallback")]:
        for cand in (col, col + "_id", "pseudo_" + col, col.upper()):
            if cand in df.columns:
                r = report_group(label, df, cand, a.n_perm)
                if r:
                    results[label] = r
                break
        else:
            print(f"\n[{label}] no column matching '{col}' in sites.csv -- skipped")

    print("\n" + "=" * 78)
    sig = {k: v for k, v in results.items() if v[1] < 0.05 and v[0] >= 1.5}
    if sig:
        print("VERDICT: CONCENTRATED -- the hard core is NOT uniformly spread.")
        for k, (obs, p, _) in sig.items():
            print(f"  {k}: {obs:.2f}x enrichment, p = {p:.4f}")
        print("""
This is the good outcome. It means a slice of the data is being preprocessed badly rather than being
inherently ambiguous, and that is fixable. Next, in this order:
  1. Pull 10 hard-core scans from the most enriched group and 10 easy scans from the same group, and look at
     them with scripts/qc_grid.py. You are looking for a registration that put the striatum in the wrong
     place, a clipped field of view, or a reference region landing on skull or background.
  2. If registration is the fault: the per-scan fix is already in the codebase as the registration fallback.
     Check whether these scans took the fallback and whether the fallback is worse than the registration it
     replaced for this group.
  3. Only if 1-2 show a real defect, and only if it is fixable without touching datsc/ in a way that ships
     untested inference code, is this worth a rebuild before the deadline. Re-cache that group, re-run the
     affected arms, re-gate. Anything less than +0.003 nested is inside the pool's +/-0.013 noise.
  4. If the defect is real but too large to fix safely in the time left, write it up as the first item of the
     next run. It is the most valuable thing you have found.""")
    else:
        print("VERDICT: UNIFORM -- no group carries the hard core.")
        print("""
The 'irreducible label noise' reading survives. These scans are genuinely ambiguous, which matches the ~7.8 %
discrepant-expert-read rate for DaT-SPECT in the literature. Consequences:
  1. Stop looking for accuracy. More capacity, more seeds and more members will not move these scans.
  2. Stop looking at calibration too -- it was measured at 0.0002 of headroom, and both 'never be confident'
     and hedge-on-disagreement were shown to lose outright.
  3. Ship the best gated ensemble you already have and spend the remaining time on verification.
  4. The ceiling is real. Record it so the next run does not re-litigate it.""")
    print("=" * 78)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/feat_arm_c.py
"""Approach C: fit the richer feature arm ("featc") on the registered cache -- OOF on random folds and pseudo-site folds
over a small model grid (kernel-RFF logistic / MLP / linear), mirror-averaged, typicality kept -- then a full-data model.
Writes <out>/featc_v5.npz, oof_featcarm.csv, oof_featcarm_site.csv, featcarm.json (same contract as the run-4 feat arm).

python scripts/feat_arm_c.py --cache cache_v4 --template artifacts/template.npz --out artifacts_v3 [--grid] [--force]
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score

from datsc.featmodel_c import FeatModelC
from datsc.features_c import FEATURE_NAMES_C, C_NAMES, feat_vector_c
from datsc.features_v4 import N_PATCH, mirror, striatal_patch
from datsc.register import Template


def build(cache: Path, T, force=False):
    """F[N,2,n_feat] (index 1 of axis 1 = mirrored), P[N,2,n_patch]."""
    fp, pp = cache / "features_c.npy", cache / "patch_v4.npy"
    meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
    have_f = fp.exists() and not force and np.load(fp, mmap_mode="r").shape[:2] == (len(meta), 2) and np.load(fp, mmap_mode="r").shape[2] == len(FEATURE_NAMES_C)
    have_p = pp.exists() and np.load(pp, mmap_mode="r").shape[0] == len(meta)
    vols = np.load(cache / "vols.npy", mmap_mode="r")
    F = np.load(fp) if have_f else np.zeros((len(meta), 2, len(FEATURE_NAMES_C)), np.float32)
    P = np.load(pp) if have_p else np.zeros((len(meta), 2, N_PATCH), np.float32)
    if have_f and have_p:
        return F, P, meta
    t0 = time.time()
    for i in range(len(meta)):
        v = np.asarray(vols[i], np.float32)
        for o, vv in enumerate((v, mirror(v))):
            if not have_f:
                F[i, o] = feat_vector_c(vv, T)
            if not have_p:
                P[i, o] = striatal_patch(vv)
        if i % 200 == 0:
            print(f"features {i}/{len(meta)} ({time.time()-t0:.0f}s)", flush=True)
    np.save(fp, F)
    if not have_p:
        np.save(pp, P)
    return F, P, meta


def oof(F, P, y, folds, kw, with_d=False):
    p = np.zeros(len(y)); d = np.zeros(len(y))
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        m = FeatModelC(**kw).fit(np.concatenate([F[tr, 0], F[tr, 1]]), np.concatenate([P[tr, 0], P[tr, 1]]), np.concatenate([y[tr], y[tr]]))
        z = 0.5 * (m.decision(F[va, 0], P[va, 0]) + m.decision(F[va, 1], P[va, 1]))
        p[va] = 1 / (1 + np.exp(-z)); d[va] = m.typicality(P[va, 0])
    return (p, d) if with_d else p


def ll(y, p):
    return log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))


GRID = [
    dict(kind="linear", n_comp=40, C=0.3),
    dict(kind="linear", n_comp=40, C=0.1),
    dict(kind="rff", n_comp=40, C=0.3, gamma=0.25, n_rff=512),
    dict(kind="rff", n_comp=40, C=1.0, gamma=0.25, n_rff=512),
    dict(kind="rff", n_comp=40, C=0.3, gamma=0.5, n_rff=1024),
    dict(kind="rff", n_comp=20, C=0.3, gamma=0.5, n_rff=512),
    dict(kind="mlp", n_comp=40, hidden=(64, 32), alpha=1e-1),
    dict(kind="mlp", n_comp=40, hidden=(64, 32), alpha=1.0),
    dict(kind="mlp", n_comp=20, hidden=(32,), alpha=1.0),
]

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--template", default="artifacts/template.npz")
    ap.add_argument("--out", default="artifacts_v3")
    ap.add_argument("--grid", action="store_true", help="evaluate the model grid on the random folds first (else: linear C=0.3)")
    ap.add_argument("--kind", default=None, help="force a model kind: linear | rff | mlp (overrides --grid)")
    ap.add_argument("--force", action="store_true")
    a = ap.parse_args()
    cache, out = Path(a.cache), Path(a.out); out.mkdir(exist_ok=True)
    T = Template.load(a.template)
    F, P, meta = build(cache, T, a.force)
    y = meta["is_pathologic"].values.astype(float)
    folds = pd.read_csv(cache / "folds.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "fold"].values
    fs = pd.read_csv(cache / "folds_site.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "fold"].values if (cache / "folds_site.csv").exists() else None
    print(f"features: {F.shape[2]} ({len(C_NAMES)} new in the C block); patch {P.shape[2]}")
    kw = dict(kind="linear", n_comp=40, C=0.3)
    if a.grid:
        rows = []
        for g in GRID:
            if a.kind and g["kind"] != a.kind:
                continue
            t0 = time.time(); p = oof(F, P, y, folds, g); rows.append((ll(y, p), roc_auc_score(y, p), g))
            print(f"  {json.dumps(g):<70} OOF logloss={rows[-1][0]:.4f} AUROC={rows[-1][1]:.4f} ({time.time()-t0:.0f}s)", flush=True)
        best = min(rows, key=lambda r: r[0]); kw = best[2]
        print(f"grid best: {json.dumps(kw)} (logloss {best[0]:.4f})")
    elif a.kind:
        kw = next(g for g in GRID if g["kind"] == a.kind)
    p, d = oof(F, P, y, folds, kw, with_d=True)
    auc, l = roc_auc_score(y, p), ll(y, p)
    print(f"featc random folds: OOF AUROC={auc:.4f} logloss={l:.4f}  (run-4 feat arm: 0.9398 / 0.3120; LR arm 0.902 / 0.4015); d median {np.median(d):.2f} p90 {np.percentile(d, 90):.2f}")
    pd.DataFrame({"uid": meta.uid, "y": y, "p": p, "d": d}).to_csv(out / "oof_featcarm.csv", index=False)
    if fs is not None:
        ps, ds = oof(F, P, y, fs, kw, with_d=True)
        print(f"featc site folds  : OOF AUROC={roc_auc_score(y, ps):.4f} logloss={ll(y, ps):.4f}  (run-4 feat arm site folds: 0.9277 / 0.3548)")
        pd.DataFrame({"uid": meta.uid, "y": y, "p": ps, "d": ds}).to_csv(out / "oof_featcarm_site.csv", index=False)
        sites = pd.read_csv(cache / "sites.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "site"].values
        for s in pd.Series(sites).value_counts().index[:9]:
            m = sites == s
            print(f"  site {s:<24} n={m.sum():4d} prev={y[m].mean():.2f} mean_p={ps[m].mean():.2f} logloss={ll(y[m], ps[m]):.4f} "
                  f"AUROC={roc_auc_score(y[m], ps[m]) if 0 < y[m].mean() < 1 else float('nan'):.3f}")
    off = len(FEATURE_NAMES_C) - len(C_NAMES)
    uni = sorted([(round(max(roc_auc_score(y, F[:, 0, off + i]), 1 - roc_auc_score(y, F[:, 0, off + i])), 3), n) for i, n in enumerate(C_NAMES)], reverse=True)[:10]
    print("top C-block features (univariate AUROC):", uni)
    full = FeatModelC(**kw).fit(np.concatenate([F[:, 0], F[:, 1]]), np.concatenate([P[:, 0], P[:, 1]]), np.concatenate([y, y]))
    full.save(out / "featc_v5.npz")
    (out / "featcarm.json").write_text(json.dumps({"tag": "featcarm", "spec": {"arm": "featc"}, "model_kw": kw, "norm": "ref",
                                                   "checkpoints": ["artifacts/featc_v5.npz"], "val_auroc": float(auc), "val_logloss": float(l)}))
    print(f"saved {out/'featc_v5.npz'} ({(out/'featc_v5.npz').stat().st_size/1e6:.1f} MB) + oof_featcarm.csv (+ _site) + featcarm.json")


In [ ]:
%%writefile scripts/feat_arm_v4.py
"""Fit the feature-engineered arm ("feat") on the registered cache: OOF on random folds and pseudo-site folds, then a
full-data model. Writes artifacts_v3/feat_v4.npz, oof_featarm.csv, oof_featarm_site.csv, featarm.json (same contract as
the LR arm from gate_v3.py, so fit_stack_v3.py can take `--arm feat featarm --site-arm feat featarm_site`).

python scripts/feat_arm_v4.py --cache cache_v4 --template artifacts/template.npz --out artifacts_v3 [--grid]
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score

from datsc.featmodel import FeatModel
from datsc.features_v4 import FEATURE_NAMES, N_PATCH, feat_vector, mirror, striatal_patch
from datsc.register import Template


def build(cache: Path, T, force=False):
    """F[N,2,n_feat], P[N,2,n_patch]: index 1 of axis 1 is the mirrored orientation."""
    fp, pp = cache / "features_v4.npy", cache / "patch_v4.npy"
    meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
    if fp.exists() and pp.exists() and not force and np.load(fp, mmap_mode="r").shape[0] == len(meta):
        return np.load(fp), np.load(pp), meta
    vols = np.load(cache / "vols.npy", mmap_mode="r")
    F = np.zeros((len(meta), 2, len(FEATURE_NAMES)), np.float32); P = np.zeros((len(meta), 2, N_PATCH), np.float32)
    t0 = time.time()
    for i in range(len(meta)):
        v = np.asarray(vols[i], np.float32)
        for o, vv in enumerate((v, mirror(v))):
            F[i, o] = feat_vector(vv, T); P[i, o] = striatal_patch(vv)
        if i % 200 == 0:
            print(f"features {i}/{len(meta)} ({time.time()-t0:.0f}s)", flush=True)
    np.save(fp, F); np.save(pp, P)
    return F, P, meta


def oof(F, P, y, folds, n_comp, C, with_d=False):
    p = np.zeros(len(y)); d = np.zeros(len(y))
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        m = FeatModel(n_comp, C).fit(np.concatenate([F[tr, 0], F[tr, 1]]), np.concatenate([P[tr, 0], P[tr, 1]]), np.concatenate([y[tr], y[tr]]))
        z = 0.5 * (m.decision(F[va, 0], P[va, 0]) + m.decision(F[va, 1], P[va, 1]))      # average over both orientations
        p[va] = 1 / (1 + np.exp(-z)); d[va] = m.typicality(P[va, 0])
    return (p, d) if with_d else p


def ll(y, p):
    return log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--template", default="artifacts/template.npz")
    ap.add_argument("--out", default="artifacts_v3")
    ap.add_argument("--n-comp", type=int, default=40)
    ap.add_argument("--C", type=float, default=0.1)
    ap.add_argument("--grid", action="store_true", help="evaluate a small (n_comp, C) grid on the random folds first")
    ap.add_argument("--force", action="store_true")
    a = ap.parse_args()
    cache, out = Path(a.cache), Path(a.out); out.mkdir(exist_ok=True)
    T = Template.load(a.template)
    F, P, meta = build(cache, T, a.force)
    y = meta["is_pathologic"].values.astype(float)
    folds = pd.read_csv(cache / "folds.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "fold"].values
    fs = pd.read_csv(cache / "folds_site.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "fold"].values if (cache / "folds_site.csv").exists() else None
    n_comp, C = a.n_comp, a.C
    if a.grid:
        rows = []
        for nc in (20, 40, 80):
            for c in (0.03, 0.1, 0.3):
                p = oof(F, P, y, folds, nc, c); rows.append((ll(y, p), roc_auc_score(y, p), nc, c))
                print(f"  n_comp={nc:3d} C={c:<5} OOF logloss={rows[-1][0]:.4f} AUROC={rows[-1][1]:.4f}", flush=True)
        best = min(rows); n_comp, C = best[2], best[3]
        print(f"grid best: n_comp={n_comp} C={C} (logloss {best[0]:.4f})")
    p, d = oof(F, P, y, folds, n_comp, C, with_d=True)
    auc, l = roc_auc_score(y, p), ll(y, p)
    print(f"feat arm random folds: OOF AUROC={auc:.4f} logloss={l:.4f}  (run-3 LR arm: 0.902 / 0.4015); typicality d: median {np.median(d):.2f} p90 {np.percentile(d, 90):.2f} max {d.max():.2f}")
    pd.DataFrame({"uid": meta.uid, "y": y, "p": p, "d": d}).to_csv(out / "oof_featarm.csv", index=False)
    if fs is not None:
        ps, ds = oof(F, P, y, fs, n_comp, C, with_d=True)
        print(f"feat arm site folds  : OOF AUROC={roc_auc_score(y, ps):.4f} logloss={ll(y, ps):.4f}; typicality d (site folds): median {np.median(ds):.2f} p90 {np.percentile(ds, 90):.2f}")
        pd.DataFrame({"uid": meta.uid, "y": y, "p": ps, "d": ds}).to_csv(out / "oof_featarm_site.csv", index=False)
        sites = pd.read_csv(cache / "sites.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "site"].values
        for s in pd.Series(sites).value_counts().index[:8]:
            m = sites == s
            print(f"  site {s:<24} n={m.sum():4d} prev={y[m].mean():.2f} mean_p={ps[m].mean():.2f} logloss={ll(y[m], ps[m]):.4f} "
                  f"AUROC={roc_auc_score(y[m], ps[m]) if 0 < y[m].mean() < 1 else float('nan'):.3f}")
    # univariate view of the engineered block
    from datsc.features_v4 import ENG_NAMES
    off = len(FEATURE_NAMES) - len(ENG_NAMES)
    uni = sorted([(round(max(roc_auc_score(y, F[:, 0, off + i]), 1 - roc_auc_score(y, F[:, 0, off + i])), 3), n) for i, n in enumerate(ENG_NAMES)], reverse=True)[:8]
    print("top engineered features (AUROC):", uni)
    full = FeatModel(n_comp, C).fit(np.concatenate([F[:, 0], F[:, 1]]), np.concatenate([P[:, 0], P[:, 1]]), np.concatenate([y, y]))
    full.save(out / "feat_v4.npz")
    (out / "featarm.json").write_text(json.dumps({"tag": "featarm", "spec": {"arm": "feat"}, "model_kw": {"n_comp": n_comp, "C": C}, "norm": "ref",
                                                  "checkpoints": ["artifacts/feat_v4.npz"], "val_auroc": float(auc), "val_logloss": float(l)}))
    print(f"saved {out/'feat_v4.npz'} ({(out/'feat_v4.npz').stat().st_size/1e6:.1f} MB) + oof_featarm.csv (+ _site) + featarm.json")


In [ ]:
%%writefile scripts/fetch_pretrained.py
"""Download permissively-licensed ImageNet/Kinetics torchvision weights once, for bundling in the submission.

All torchvision weights below are BSD-3 (torchvision) and trained on ImageNet-1k / Kinetics-400 (CC-BY).
Saved as plain state_dicts so main.py never needs network access.
"""
from __future__ import annotations

import sys
from pathlib import Path

import torch
import torchvision.models as tvm
from torchvision.models import video as tvv

OUT = Path(sys.argv[1] if len(sys.argv) > 1 else "pretrained")
OUT.mkdir(parents=True, exist_ok=True)

SPECS = {
    "convnext_tiny_in1k.pt": lambda: tvm.convnext_tiny(weights=tvm.ConvNeXt_Tiny_Weights.IMAGENET1K_V1),
    "efficientnet_b0_in1k.pt": lambda: tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1),
    "efficientnet_v2_s_in1k.pt": lambda: tvm.efficientnet_v2_s(weights=tvm.EfficientNet_V2_S_Weights.IMAGENET1K_V1),
    "resnet18_in1k.pt": lambda: tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1),
    "resnet50_in1k_v2.pt": lambda: tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2),
    "r3d_18_kinetics.pt": lambda: tvv.r3d_18(weights=tvv.R3D_18_Weights.KINETICS400_V1),
}

for name, ctor in SPECS.items():
    p = OUT / name
    if p.exists():
        print("exists", name); continue
    m = ctor()
    torch.save(m.state_dict(), p)
    print(f"saved {name}: {p.stat().st_size/1e6:.0f} MB, {sum(x.numel() for x in m.parameters())/1e6:.1f}M params")


In [ ]:
%%writefile scripts/fit_calibration.py
"""Average CNN OOF over seeds, blend with LR OOF, fit w & T, write artifacts/calib.json.
lam and pi are inputs: lam from scripts/shift_diagnostic.py (default 0.02), pi from the prior-only LB (default 0.548)."""
from __future__ import annotations

import argparse
import glob
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd

from datsc import calibrate as cb
from datsc.config import DEFAULT_PRIOR

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--artifacts", default="artifacts")
    ap.add_argument("--lam", type=float, default=0.02)
    ap.add_argument("--pi", type=float, default=DEFAULT_PRIOR)
    a = ap.parse_args()
    art = Path(a.artifacts)
    files = sorted(glob.glob(str(art / "oof_f*_s*.csv")))   # hotspot AND _com checkpoints: inference averages all of them
    assert files, "no CNN OOF files found"
    cnn = pd.concat([pd.read_csv(f, dtype={"uid": str}) for f in files]).groupby("uid").agg(y=("y", "first"), p_cnn=("p", "mean")).reset_index()
    lr = pd.read_csv(Path(a.cache) / "oof_lr.csv", dtype={"uid": str})[["uid", "p_lr"]]
    df = cnn.merge(lr, on="uid")
    y = df["y"].values.astype(float)
    print(f"n={len(df)}  checkpoints={len(files)}  logloss cnn-avg={cb.logloss(y, df.p_cnn):.4f}  lr={cb.logloss(y, df.p_lr):.4f}")
    c = cb.fit(df["p_cnn"].values, df["p_lr"].values, y, lam=a.lam, pi=a.pi)
    final = c.apply(df["p_cnn"].values, df["p_lr"].values)
    print(f"w={c.w:.2f} T={c.T:.3f} lam={c.lam} pi={c.pi:.3f}  OOF logloss after calibration={cb.logloss(y, final):.4f}")
    c.save(art / "calib.json")


In [ ]:
%%writefile scripts/fit_stack_v3.py
"""Fit the run-3 ensemble combiner on OOF predictions and write artifacts_v3/manifest.json.

Each arm = a glob of run tags sharing one InputSpec/model_kw (read from the first run's json). Arm-level OOF =
mean over its checkpoints. Combiner = weighted logit average (weights on a simplex grid), temperature T, prior mix.
T is fitted on the random folds (--calib random) or on leave-pseudo-site-out predictions (--calib site) and can be
floored (--t-floor) so a shifted test set is never met with sharpened probabilities.

python scripts/fit_stack_v3.py --out artifacts_v3 --arm 3d_a "r3d_64x64x48_f*_s*" --arm 2d_b0 "r2d_efficientnet_b0_f*_s*" [--lam 0.02] [--pi 0.548] [--t-floor 1.0]
"""
from __future__ import annotations

import argparse
import glob
import itertools
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from datsc.calibrate import fit_temperature, logit, logloss, sigmoid
from datsc.config import DEFAULT_PRIOR


def arm_oof(art: Path, pattern: str):
    files = sorted(glob.glob(str(art / f"oof_{pattern}.csv")))
    assert files, f"no OOF files for {pattern}"
    df = pd.concat([pd.read_csv(f, dtype={"uid": str}) for f in files])
    agg = dict(y=("y", "first"), p=("p", "mean"))
    if "d" in df.columns:
        agg["d"] = ("d", "mean")
    oof = df.groupby("uid").agg(**agg)
    tags = [Path(f).name[4:-4] for f in files]
    mp = art / f"{tags[0]}.json"
    meta = json.loads(mp.read_text()) if mp.exists() else None    # site-fold OOFs only need the predictions
    return oof, tags, meta


def compositions(total: int, parts: int):
    """All non-negative integer vectors of length `parts` summing to `total` (stars and bars): C(total+parts-1, parts-1)
    of them, e.g. 43,758 for grid 8 x 11 arms -- NOT the 9^11 product that run 4a's report cell got stuck on."""
    if parts == 1:
        yield (total,)
        return
    for first in range(total + 1):
        for rest in compositions(total - first, parts - 1):
            yield (first,) + rest


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="artifacts_v3")
    ap.add_argument("--arm", nargs=2, action="append", metavar=("NAME", "PATTERN"), required=True)
    ap.add_argument("--site-arm", nargs=2, action="append", metavar=("NAME", "PATTERN"), default=[],
                    help="leave-pseudo-site-out OOF pattern per arm; when every arm has one, T is fitted on the site pool")
    ap.add_argument("--extra", nargs=2, action="append", metavar=("NAME", "GLOB"), default=[],
                    help="extra checkpoint tags (e.g. full-data models 'r2d_efficientnet_b0_fall_s*') appended to an arm")
    ap.add_argument("--lam", type=float, default=0.02)
    ap.add_argument("--pi", type=float, default=DEFAULT_PRIOR)
    ap.add_argument("--t-floor", type=float, default=1.0)
    ap.add_argument("--grid", type=int, default=10, help="simplex resolution for arm weights")
    ap.add_argument("--cache", default="cache_v4", help="for folds.csv (nested estimate) and sites.csv (per-site report)")
    ap.add_argument("--nested", action="store_true", help="print the nested-CV estimate of the whole combiner and select clip/hedge by it")
    ap.add_argument("--clip", type=float, nargs="*", default=[0.0, 0.01, 0.02, 0.03], help="candidate tail clips (0 = none)")
    ap.add_argument("--hedge-q", type=float, nargs="*", default=[0.0, 0.85, 0.95], help="candidate disagreement quantiles (0 = no hedge)")
    ap.add_argument("--hedge-lam", type=float, nargs="*", default=[0.15, 0.3])
    ap.add_argument("--typ", action="store_true", help="select a typicality-aware temperature (needs a 'd' column in the feat arm OOF) by the nested estimate")
    a = ap.parse_args()
    art = Path(a.out)
    arms, oofs = [], {}
    for name, pattern in a.arm:
        oof, tags, meta = arm_oof(art, pattern)
        oofs[name] = oof
        arms.append({"name": name, "arm": meta["spec"]["arm"], "spec": meta["spec"], "model_kw": {k: v for k, v in meta.get("model_kw", {}).items() if k != "weights_path"},
                     "norm": meta.get("norm", "p75"), "checkpoints": meta.get("checkpoints") or [f"artifacts/{t}.pt" for t in tags]})
        print(f"{name:<10} ckpts={len(tags):2d} AUROC={roc_auc_score(oof.y, oof.p):.4f} logloss={logloss(oof.y.values, oof.p.values):.4f}")
    for name, pat in a.extra:
        arm = next(x for x in arms if x["name"] == name)
        extra = sorted(Path(f).name[:-3] for f in glob.glob(str(art / f"{pat}.pt")))
        arm["checkpoints"] += [f"artifacts/{t}.pt" for t in extra]
        print(f"{name}: +{len(extra)} extra checkpoints ({pat})")
    n_max = max(len(o) for o in oofs.values())
    short = [x["name"] for x in arms if len(oofs[x["name"]]) < 0.98 * n_max]
    if short:
        print(f"dropping arms with incomplete OOF coverage (< 98% of {n_max} uids): {short}")
        arms = [x for x in arms if x["name"] not in short]; oofs = {k: v for k, v in oofs.items() if k not in short}
    uids = sorted(set.intersection(*[set(o.index) for o in oofs.values()]))
    y = oofs[arms[0]["name"]].loc[uids, "y"].values.astype(float)
    Z = np.stack([logit(oofs[arm["name"]].loc[uids, "p"].values) for arm in arms])
    W_grid_all = np.array(list(compositions(a.grid, len(arms))), float) / a.grid
    logits_all = W_grid_all @ Z
    losses_all = (np.maximum(logits_all, 0.0) - logits_all * y + np.log1p(np.exp(-np.abs(logits_all)))).mean(axis=1)
    w = W_grid_all[np.argmin(losses_all)]
    z = (w[:, None] * Z).sum(0)
    T_rand = fit_temperature(z, y)

    # ---- nested estimate of the full combiner + clip/hedge selection (weights & T refitted on 4 folds, applied to the 5th) ----
    def fit_wt(Zs, ys):
        W_grid = np.array(list(compositions(a.grid, Zs.shape[0])), float) / a.grid
        logits = W_grid @ Zs
        losses = (np.maximum(logits, 0.0) - logits * ys + np.log1p(np.exp(-np.abs(logits)))).mean(axis=1)
        bw = W_grid[np.argmin(losses)]
        zz = (bw[:, None] * Zs).sum(0)
        return bw, max(fit_temperature(zz, ys), a.t_floor)

    def apply(Zs, ww, TT, clip, hedge, typ=None, dd=None):
        zz = (ww[:, None] * Zs).sum(0)
        Ti = TT * (1 + typ[0] * np.maximum(0.0, dd - typ[1])) if (typ is not None and dd is not None) else TT
        pp = sigmoid(zz / Ti)
        lam_i = np.full(pp.shape, a.lam)
        if hedge is not None and Zs.shape[0] >= 2:
            spread = Zs.max(0) - Zs.min(0); lam_i = np.where(spread > hedge[0], hedge[1], lam_i)
        pp = (1 - lam_i) * pp + lam_i * a.pi
        return np.clip(pp, clip, 1 - clip) if clip > 0 else pp

    chosen_clip, chosen_hedge = 0.0, None
    if a.nested:
        folds = pd.read_csv(Path(a.cache) / "folds.csv", dtype={"uid": str}).set_index("uid").loc[uids, "fold"].values
        spread_all = Z.max(0) - Z.min(0)
        fold_models = {k: fit_wt(Z[:, folds != k], y[folds != k]) for k in np.unique(folds)}
        cands = []
        for clip in a.clip:
            for q in a.hedge_q:
                for hl in (a.hedge_lam if q > 0 else [0.0]):
                    hedge = None if q <= 0 else (float(np.quantile(spread_all, q)), hl)
                    pn = np.zeros(len(y))
                    for k in np.unique(folds):
                        va = folds == k
                        ww, TT = fold_models[k]
                        pn[va] = apply(Z[:, va], ww, TT, clip, hedge)
                    cands.append((logloss(y, pn), clip, q, hl, hedge))
        cands.sort(key=lambda c: c[0])
        base = next(c for c in cands if c[1] == 0.0 and c[2] == 0.0)
        print(f"nested combiner estimate (no clip, no hedge): {base[0]:.4f}")
        for c in cands[:6]:
            print(f"  nested {c[0]:.4f}  clip={c[1]}  hedge_q={c[2]} hedge_lam={c[3]}")
        bestc = cands[0]
        if bestc[0] < base[0] - 0.002:
            chosen_clip, chosen_hedge = bestc[1], bestc[4]
            print(f"selected clip={chosen_clip} hedge={chosen_hedge} (nested gain {base[0]-bestc[0]:.4f})")
        else:
            print("clip/hedge give no reliable nested gain; keeping plain combiner")
        # per-site report
        sp = Path(a.cache) / "sites.csv"
        if sp.exists():
            sites = pd.read_csv(sp, dtype={"uid": str}).set_index("uid").loc[uids, "site"].values
            pn = np.zeros(len(y))
            for k in np.unique(folds):
                va = folds == k
                ww, TT = fold_models[k]
                pn[va] = apply(Z[:, va], ww, TT, chosen_clip, chosen_hedge)
            print("per-site nested: site n prev mean_p logloss auroc")
            for st in pd.Series(sites).value_counts().index[:9]:
                m = sites == st
                au = roc_auc_score(y[m], pn[m]) if 0 < y[m].mean() < 1 else float("nan")
                print(f"  {st:<26} {m.sum():4d} {y[m].mean():.2f} {pn[m].mean():.2f} {logloss(y[m], pn[m]):.4f} {au:.3f}")
    chosen_typ = None
    if a.typ and a.nested and "feat" in oofs and "d" in oofs["feat"].columns:
        dd = oofs["feat"].loc[uids, "d"].values.astype(float)
        folds = pd.read_csv(Path(a.cache) / "folds.csv", dtype={"uid": str}).set_index("uid").loc[uids, "fold"].values
        def nested_ll(typ):
            pn = np.zeros(len(y))
            for k in np.unique(folds):
                va = folds == k
                ww, TT = fold_models[k]
                pn[va] = apply(Z[:, va], ww, TT, chosen_clip, chosen_hedge, typ, dd[va] if typ else None)
            return logloss(y, pn)
        base_t = nested_ll(None); cands_t = []
        for kappa in (0.25, 0.5, 1.0, 2.0):
            for q in (0.75, 0.9):
                d0 = float(np.quantile(dd, q)); cands_t.append((nested_ll((kappa, d0)), kappa, d0, q))
        cands_t.sort(); bt = cands_t[0]
        print(f"typicality: nested {base_t:.4f} -> best {bt[0]:.4f} (kappa={bt[1]}, d0=q{bt[3]}={bt[2]:.2f}); d median {np.median(dd):.2f}")
        if bt[0] < base_t - 0.003:
            chosen_typ = (bt[1], bt[2]); print(f"selected typicality temperature kappa={bt[1]} d0={bt[2]:.3f}")
        else:
            print("typicality temperature gives no reliable nested gain; not selected")
    T_site = None
    site = dict(a.site_arm)
    if site and all(x["name"] in site for x in arms):
        oofs_s = {n: arm_oof(art, site[n])[0] for n in site}
        uids_s = sorted(set.intersection(*[set(o.index) for o in oofs_s.values()]))
        y_s = oofs_s[arms[0]["name"]].loc[uids_s, "y"].values.astype(float)
        Z_s = np.stack([logit(oofs_s[x["name"]].loc[uids_s, "p"].values) for x in arms])
        z_s = (w[:, None] * Z_s).sum(0)
        T_site = fit_temperature(z_s, y_s)
        print(f"site-fold pool: n={len(y_s)} AUROC={roc_auc_score(y_s, sigmoid(z_s)):.4f} logloss(T=1)={logloss(y_s, sigmoid(z_s)):.4f} "
              f"logloss(T_site)={logloss(y_s, sigmoid(z_s / T_site)):.4f} T_site={T_site:.3f} (random-fold T={T_rand:.3f})")
    T = max(T_site if T_site is not None else T_rand, a.t_floor)
    final = apply(Z, w, T, chosen_clip, chosen_hedge, chosen_typ, (oofs["feat"].loc[uids, "d"].values.astype(float) if chosen_typ else None))
    print(f"weights={dict(zip([x['name'] for x in arms], np.round(w, 2)))}  T={T:.3f} (random {T_rand:.3f}, site {T_site}, floor {a.t_floor})  lam={a.lam}  pi={a.pi:.3f}")
    print(f"stack OOF: AUROC={roc_auc_score(y, final):.4f} logloss={logloss(y, final):.4f}  (equal-weight logloss={logloss(y, sigmoid(Z.mean(0))):.4f})")
    man = {"pi": a.pi, "T": float(T), "T_random": float(T_rand), "T_site": (float(T_site) if T_site is not None else None), "lam": a.lam,
           "clip": ([chosen_clip, 1 - chosen_clip] if chosen_clip > 0 else None),
           "hedge": ({"thresh": float(chosen_hedge[0]), "lam": float(chosen_hedge[1])} if chosen_hedge else None),
           "typ": ({"kappa": float(chosen_typ[0]), "d0": float(chosen_typ[1])} if chosen_typ else None), "weights": {x["name"]: float(wi) for x, wi in zip(arms, w)}, "arms": arms,
           "oof_logloss": logloss(y, final), "oof_auroc": float(roc_auc_score(y, final)), "n": int(len(y))}
    (art / "manifest.json").write_text(json.dumps(man, indent=1))
    pd.DataFrame({"uid": uids, "y": y, "p": final}).to_csv(art / "oof_stack.csv", index=False)
    print(f"wrote {art/'manifest.json'}")


In [ ]:
%%writefile scripts/gate_v3.py
"""Registration gate G1 + the run-3 LR arm: registered-ROI striatal features -> L2 logistic regression.
Reports OOF AUROC/log loss on the random folds AND leave-pseudo-site-out folds, plus per-site bias.
Writes <cache>/features_v3.npy, <cache>/oof_lr_v3.csv and (if --save) artifacts_v3/lr_v3.json.

python scripts/gate_v3.py --cache cache_v4 --template artifacts/template.npz [--save]
GATE: random-fold OOF AUROC >= 0.92 and registration fallback < 3%.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score

from datsc.features_v3 import FEATURE_NAMES, striatal_features
from datsc.lrmodel import LRModel
from datsc.register import Template


def oof_lr(X, y, folds, C=0.3):
    p = np.zeros(len(y))
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        m = LRModel(C=C).fit(X[tr], y[tr])
        p[va] = m.predict_proba(X[va])
    return p


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--template", default="artifacts/template.npz")
    ap.add_argument("--out", default="artifacts_v3")
    ap.add_argument("--save", action="store_true")
    ap.add_argument("--C", type=float, default=0.3)
    a = ap.parse_args()
    cache = Path(a.cache); T = Template.load(a.template)
    vols = np.load(cache / "vols.npy", mmap_mode="r"); meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
    fp = cache / "features_v3.npy"
    if fp.exists() and np.load(fp).shape[0] == len(meta):
        X = np.load(fp)
    else:
        X = np.stack([striatal_features(np.asarray(vols[i], np.float32), T) for i in range(len(meta))]); np.save(fp, X)
    y = meta["is_pathologic"].values.astype(float)
    folds = pd.read_csv(cache / "folds.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "fold"].values
    p = oof_lr(X, y, folds, a.C)
    auc, ll = roc_auc_score(y, p), log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))
    print(f"random folds : OOF AUROC={auc:.4f} logloss={ll:.4f}")
    if (cache / "folds_site.csv").exists():
        fs = pd.read_csv(cache / "folds_site.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "fold"].values
        ps = oof_lr(X, y, fs, a.C)
        print(f"site folds   : OOF AUROC={roc_auc_score(y, ps):.4f} logloss={log_loss(y, np.clip(ps,1e-6,1-1e-6)):.4f}")
        sites = pd.read_csv(cache / "sites.csv", dtype={"uid": str}).set_index("uid").loc[meta.uid, "site"].values
        for s in pd.Series(sites).value_counts().index[:8]:
            m = sites == s
            print(f"  site {s:<22} n={m.sum():4d} prev={y[m].mean():.2f} mean_p(site)={ps[m].mean():.2f} mean_p(rand)={p[m].mean():.2f} "
                  f"AUROC site={roc_auc_score(y[m], ps[m]) if 0<y[m].mean()<1 else float('nan'):.3f} rand={roc_auc_score(y[m], p[m]) if 0<y[m].mean()<1 else float('nan'):.3f}")
    fb = 1 - meta["reg_ok"].mean() if "reg_ok" in meta else float("nan")
    print(f"registration fallback: {100*fb:.1f}%  (normals {100*(1-meta.loc[y==0,'reg_ok'].mean()):.1f}%, abnormals {100*(1-meta.loc[y==1,'reg_ok'].mean()):.1f}%)" if "reg_ok" in meta else "")
    # univariate AUROCs for insight
    print("top single features:", sorted([(round(max(roc_auc_score(y, X[:, i]), 1 - roc_auc_score(y, X[:, i])), 3), n) for i, n in enumerate(FEATURE_NAMES)], reverse=True)[:6])
    print(f"GATE G1: {'PASS' if auc >= 0.92 and fb < 0.03 else 'FAIL'} (AUROC {auc:.3f} vs 0.92; fallback {100*fb:.1f}% vs 3%)")
    pd.DataFrame({"uid": meta.uid, "y": y, "p": p}).to_csv(cache / "oof_lr_v3.csv", index=False)
    if a.save:
        Path(a.out).mkdir(exist_ok=True)
        LRModel(C=a.C).fit(X, y).save(Path(a.out) / "lr_v3.json")
        pd.DataFrame({"uid": meta.uid, "y": y, "p": p}).to_csv(Path(a.out) / "oof_lrarm.csv", index=False)
        if (cache / "folds_site.csv").exists():
            pd.DataFrame({"uid": meta.uid, "y": y, "p": ps}).to_csv(Path(a.out) / "oof_lrarm_site.csv", index=False)
        import json
        (Path(a.out) / "lrarm.json").write_text(json.dumps({"tag": "lrarm", "spec": {"arm": "lr"}, "model_kw": {}, "norm": "ref",
                                                            "checkpoints": ["artifacts/lr_v3.json"], "val_auroc": float(auc), "val_logloss": float(ll)}))
        print(f"saved {Path(a.out)/'lr_v3.json'} + oof_lrarm.csv (+ _site) + lrarm.json")


In [ ]:
%%writefile scripts/generate_paper_figures.py
# -*- coding: utf-8 -*-
"""Generate high-quality, publication-grade TikZ figures for all papers."""
from pathlib import Path

FIG_DIR = Path("paper_v2/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------
# 1. Figure 1: DaT-SPECT Striatal Uptake Schematic
# -------------------------------------------------------------
FIG_SCHEMATIC = r"""\begin{tikzpicture}[scale=0.92, every node/.style={transform shape}]
  \tikzset{
    skull/.style={fill=gray!7, draw=gray!45, line width=1.1pt},
    striatum/.style={fill=orange!85!red, draw=red!70!black, line width=0.6pt},
    core/.style={fill=yellow!95!orange, draw=none}
  }

  % (a) Normal Scan
  \begin{scope}[local bounding box=normalscan]
    \draw[skull] (0,0) ellipse (2.05cm and 2.25cm);
    \draw[dashed, gray!30, thin] (0,-2.0) -- (0,2.0);
    
    % Left Striatum
    \filldraw[striatum] (-0.65,0.45) circle (0.34cm);
    \filldraw[striatum, rotate around={25:(-0.95,-0.25)}] (-0.95,-0.25) ellipse (0.50cm and 0.21cm);
    
    % Right Striatum
    \filldraw[striatum] (0.65,0.45) circle (0.34cm);
    \filldraw[striatum, rotate around={-25:(0.95,-0.25)}] (0.95,-0.25) ellipse (0.50cm and 0.21cm);
    
    % Caudate hot cores
    \filldraw[core] (-0.65,0.45) circle (0.17cm);
    \filldraw[core] (0.65,0.45) circle (0.17cm);
    
    % Labels (placed well away from center)
    \draw[-{Latex[length=1.8mm]}, thick, gray!60!black] (-1.8,1.2) node[above, font=\scriptsize\bfseries] {Caudate Head} -- (-0.85,0.6);
    \draw[-{Latex[length=1.8mm]}, thick, gray!60!black] (-1.8,-1.2) node[below, font=\scriptsize\bfseries] {Putamen Tail} -- (-1.1,-0.45);
    
    \node[font=\small\bfseries, align=center] at (0,-2.7) {(a) Normal Uptake\\{\footnotesize Symmetric bilateral ``commas''}};
  \end{scope}

  % (b) Pathological Scan (placed with 6.5cm clearance)
  \begin{scope}[xshift=6.5cm, local bounding box=abnormalscan]
    \draw[skull] (0,0) ellipse (2.05cm and 2.25cm);
    \draw[dashed, gray!30, thin] (0,-2.0) -- (0,2.0);
    
    % Left side: Partially preserved caudate, faint putamen
    \filldraw[striatum] (-0.65,0.45) circle (0.34cm);
    \filldraw[fill=orange!45!gray!30, draw=gray!40, dashed, line width=0.6pt, rotate around={25:(-0.90,-0.2)}] (-0.90,-0.2) ellipse (0.35cm and 0.17cm);
    \filldraw[core] (-0.65,0.45) circle (0.15cm);
    
    % Right side: Complete putaminal loss ("Period" sign)
    \filldraw[striatum] (0.65,0.45) circle (0.32cm);
    \draw[draw=red!70!black, dashed, line width=0.8pt, rotate around={-25:(0.95,-0.25)}] (0.95,-0.25) ellipse (0.50cm and 0.21cm);
    \filldraw[core] (0.65,0.45) circle (0.14cm);
    
    % Deficit callout pointing rightwards, clear of all drawings
    \draw[-{Latex[length=2.0mm]}, line width=1pt, red!75!black] (1.85,-1.2) node[below, font=\scriptsize\bfseries, text=red!75!black, align=center] {Severe Putaminal Deficit\\(Flattened to ``Period'')} -- (1.05,-0.35);
    
    \node[font=\small\bfseries, align=center] at (0,-2.7) {(b) Abnormal (\texttt{is\_pathologic} = 1)\\{\footnotesize Asymmetric loss of putaminal binding}};
  \end{scope}
\end{tikzpicture}"""

(FIG_DIR / "fig_schematic.tex").write_text(FIG_SCHEMATIC, encoding="utf-8")


# -------------------------------------------------------------
# 2. Figure 2: Preprocessing Pipeline (Clean, Modular Flow)
# -------------------------------------------------------------
FIG_PREPROCESSING = r"""\begin{tikzpicture}[
  font=\footnotesize\sffamily,
  node distance=8mm and 12mm,
  stage/.style={draw=blue!40!black, rounded corners=4pt, fill=blue!5, line width=0.8pt, align=center, minimum height=1.1cm, inner sep=6pt},
  decision/.style={draw=orange!60!black, fill=orange!10, rounded corners=2pt, line width=0.8pt, align=center, minimum height=1.0cm, inner sep=5pt},
  passnode/.style={draw=teal!60!black, fill=teal!10, rounded corners=4pt, line width=0.8pt, align=center, minimum height=1.1cm, inner sep=6pt},
  failnode/.style={draw=red!60!black, fill=red!8, rounded corners=4pt, line width=0.8pt, align=center, minimum height=1.1cm, inner sep=6pt},
  arrow/.style={-{Latex[length=2.2mm]}, thick, gray!70!black},
  lbl/.style={font=\scriptsize\bfseries, text=gray!80!black}
]
  % Stage 1: Harmonization
  \node[stage, minimum width=2.8cm] (raw) {\textbf{Multi-Site NIfTI}\\10 native hospitals\\(2.46--3.9\,mm spacing)};
  \node[stage, right=of raw, minimum width=3.0cm] (headbox) {\textbf{Fixed Head Box}\\$192\times192\times144$\,mm\\2\,mm isotropic (CoM)};
  
  % Stage 2: Registration & Fallback
  \node[stage, right=of headbox, minimum width=3.0cm] (reg) {\textbf{Affine Registration}\\to label-blind\\symmetric template};
  \node[decision, below=7mm of reg, minimum width=2.8cm] (gate) {\textbf{Quality Gate}\\Transform OK?};
  
  % Paths
  \node[passnode, right=of reg, minimum width=2.6cm] (regframe) {\textbf{Registered}\\\textbf{Frame}\\(canonical)};
  \node[failnode, right=of gate, minimum width=2.6cm] (comframe) {\textbf{CoM Frame}\\(safe fallback)};
  
  % Stage 3: Normalization & Derivation
  \node[stage, right=14mm of regframe, minimum width=3.0cm] (norm) {\textbf{Ref-Region Norm.}\\Non-striatal uptake $\to 1.0$};
  \node[stage, right=of norm, minimum width=3.2cm] (inputs) {\textbf{Derived Inputs}\\$\bullet$ 3-Slab Stacks (2.5D)\\$\bullet$ Striatal Crops (3D)\\$\bullet$ Radiomics (Feat)};

  % Connecting arrows
  \draw[arrow] (raw) -- (headbox);
  \draw[arrow] (headbox) -- (reg);
  \draw[arrow] (reg) -- (gate);
  
  \draw[arrow] (reg) -- node[above, lbl] {Pass} (regframe);
  \draw[arrow] (gate) -- node[above, lbl, text=red!70!black] {Fallback} (comframe);
  
  \draw[arrow] (regframe) -- (norm);
  \draw[arrow] (comframe.east) -| ++(0.7,0) |- (norm.south west);
  \draw[arrow] (norm) -- (inputs);

\end{tikzpicture}"""

(FIG_DIR / "fig_preprocessing.tex").write_text(FIG_PREPROCESSING, encoding="utf-8")


# -------------------------------------------------------------
# 3. Figure 3: Hemisphere Arm Architecture (Compact Horizontal)
# -------------------------------------------------------------
FIG_HEMI = r"""\begin{tikzpicture}[
  font=\scriptsize\sffamily,
  scale=0.92, every node/.style={transform shape},
  box/.style={draw=blue!40!black, rounded corners=3pt, fill=blue!6, align=center, inner sep=4pt, minimum height=0.85cm},
  enc/.style={draw=teal!60!black, rounded corners=4pt, fill=teal!12, align=center, line width=0.9pt, inner sep=5pt, minimum height=1.5cm},
  head/.style={draw=orange!60!black, rounded corners=3pt, fill=orange!10, align=center, inner sep=4pt, minimum height=0.85cm},
  op/.style={draw=gray!60!black, circle, fill=gray!12, inner sep=2pt, font=\small\bfseries},
  arrow/.style={-{Latex[length=2mm]}, thick, gray!70!black}
]
  % Top row: Input & Half Crops
  \node[box, fill=purple!8, minimum width=3.8cm] (input) at (0, 3.8) {\textbf{Striatal Crop} ($48\times48\times32$)\\(Registered / CoM frame)};
  \node[box, minimum width=2.4cm] (left1) at (-2.6, 2.3) {Left Half\\$24\times48\times32$};
  \node[box, minimum width=2.4cm] (right1) at (2.6, 2.3) {Right Half\\$24\times48\times32$};
  \node[box, fill=gray!10, minimum width=2.4cm] (mirror) at (2.6, 1.0) {Midline Flip $\mathcal{M}$\\(left orientation)};

  % Mid row: Shared 3D Encoder
  \node[enc, minimum width=3.2cm] (encL) at (-2.6, -0.6) {\textbf{Shared 3D ResNet}\\\textbullet\ Conv3D Stem\\\textbullet\ 4$\times$ Block3D ($24{\to}192$)\\\textbullet\ Global Average Pool};
  \node[enc, minimum width=3.2cm] (encR) at (2.6, -0.6) {\textbf{Shared 3D ResNet}\\\textbullet\ Conv3D Stem\\\textbullet\ 4$\times$ Block3D ($24{\to}192$)\\\textbullet\ Global Average Pool};
  \node[font=\tiny\itshape, text=teal!70!black] at (0, -0.6) {Identical\\Shared Weights};
  \draw[dashed, teal!60, thick] (encL.east) -- (encR.west);

  % Feature vectors
  \node[box, fill=teal!6, minimum width=2.0cm] (fL) at (-2.6, -2.1) {$f_L \in \mathbb{R}^{192}$};
  \node[box, fill=teal!6, minimum width=2.0cm] (fR) at (2.6, -2.1) {$f_R \in \mathbb{R}^{192}$};

  % Bottom row: Dual Symmetric Heads
  \node[head, fill=green!10, minimum width=3.2cm] (combine) at (0, -3.2) {\textbf{Bilateral \& Asymmetry}\\\textbf{Feature}: $[f_L+f_R,\, |f_L-f_R|]$};
  \node[head, minimum width=2.5cm] (pairhead) at (0, -4.5) {\textbf{Pair Head MLP}\\$384 \to 128 \to 1$};
  
  \node[head, minimum width=2.0cm] (sideL) at (-4.2, -3.2) {Side Head\\(shared)};
  \node[head, minimum width=2.0cm] (sideR) at (4.2, -3.2) {Side Head\\(shared)};
  \node[op] (maxop) at (4.2, -4.5) {$\max$};

  \node[op] (sumop) at (0, -5.7) {$+$};
  \node[box, fill=orange!20, minimum width=4.0cm, line width=1pt] (output) at (0, -6.8) {\textbf{Final Logit}: $z = z_{\text{pair}} + z_{\text{side}}$\\{\scriptsize Provably invariant to left-right scan flip}};

  % Arrows
  \draw[arrow] (input.south) -| (left1.north);
  \draw[arrow] (input.south) -| (right1.north);
  \draw[arrow] (left1) -- (encL);
  \draw[arrow] (right1) -- (mirror);
  \draw[arrow] (mirror) -- (encR);
  \draw[arrow] (encL) -- (fL);
  \draw[arrow] (encR) -- (fR);
  
  \draw[arrow] (fL.south) |- (combine.west);
  \draw[arrow] (fR.south) |- (combine.east);
  \draw[arrow] (combine) -- (pairhead);
  
  \draw[arrow] (fL.west) -| (sideL.north);
  \draw[arrow] (fR.east) -| (sideR.north);
  \draw[arrow] (sideL.south) |- node[pos=0.7,above,font=\tiny]{$z_L$} (maxop.west);
  \draw[arrow] (sideR.south) -- node[right,font=\tiny]{$z_R$} (maxop.north);

  \draw[arrow] (pairhead) -- node[left,font=\tiny]{$z_{\text{pair}}$} (sumop);
  \draw[arrow] (maxop.south) |- node[pos=0.7,above,font=\tiny]{$z_{\text{side}}$} (sumop.east);
  \draw[arrow] (sumop) -- (output);
\end{tikzpicture}"""

(FIG_DIR / "fig_hemi.tex").write_text(FIG_HEMI, encoding="utf-8")


# -------------------------------------------------------------
# 4. Figure 4: Multi-Arm Ensemble Combiner
# -------------------------------------------------------------
FIG_ENSEMBLE = r"""\begin{tikzpicture}[
  font=\footnotesize\sffamily,
  scale=0.95, every node/.style={transform shape},
  arm3d/.style={draw=teal!70!black, rounded corners=3pt, minimum width=4.8cm, minimum height=0.9cm, align=left, fill=teal!10, line width=0.8pt},
  arm25d/.style={draw=blue!70!black, rounded corners=3pt, minimum width=4.8cm, minimum height=0.9cm, align=left, fill=blue!8, line width=0.8pt},
  armfeat/.style={draw=orange!70!black, rounded corners=3pt, minimum width=4.8cm, minimum height=0.9cm, align=left, fill=orange!10, line width=0.8pt},
  comb/.style={draw=green!60!black, rounded corners=4pt, minimum width=4.2cm, minimum height=3.2cm, align=center, fill=green!8, line width=0.9pt},
  outbox/.style={draw=orange!70!black, rounded corners=3pt, minimum width=2.8cm, minimum height=1.1cm, align=center, fill=orange!18, line width=0.9pt},
  arrow/.style={-{Latex[length=2.2mm]}, thick, gray!70!black}
]
  % 5 Arms
  \node[arm3d] (a1) at (0, 2.6) {\textbf{Hemisphere 3D} (HemiNet3D)\hfill\textbf{50.0\%}\\{\scriptsize Shared bilateral ResNet encoder}};
  \node[arm3d] (a2) at (0, 1.3) {\textbf{Self-Supervised 3D} (VICReg)\hfill\textbf{12.5\%}\\{\scriptsize Pretrained 3D CNN on whole brain}};
  \node[armfeat] (a3) at (0, 0.0) {\textbf{Feature Arm} (Radiomics)\hfill\textbf{12.5\%}\\{\scriptsize 116 striatal descriptors + Kernel RFF}};
  \node[arm25d] (a4) at (0, -1.3) {\textbf{Coronal 2.5D} (ConvNeXt-Tiny)\hfill\textbf{12.5\%}\\{\scriptsize 3-slab coronal projection stacks}};
  \node[arm25d] (a5) at (0, -2.6) {\textbf{Axial 2.5D} (EfficientNetV2-S)\hfill\textbf{12.5\%}\\{\scriptsize 3-slab axial projection stacks}};

  % Combiner Box
  \node[comb] (comb) at (6.2, 0.0) {
    \textbf{\large Calibration Combiner}\\[4pt]
    \parbox{3.8cm}{\centering
      \textbullet\ Log-odds conversion: $z_i = \text{logit}(p_i)$\\[2pt]
      \textbullet\ Weighted pooling: $\bar{z} = \sum w_i z_i$\\[2pt]
      \textbullet\ Temperature scaling: $T=0.90$\\[2pt]
      \textbullet\ Bayesian prior mix: $\lambda=0.01$
    }
  };

  % Output Box
  \node[outbox] (out) at (10.8, 0.0) {\textbf{Final Prediction}\\$\hat{p} \in [0, 1]$\\{\scriptsize Well-calibrated}};

  % Connections
  \foreach \a in {a1, a2, a3, a4, a5} {
    \draw[arrow] (\a.east) -- (comb.west |- \a.east);
  }
  \draw[arrow] (comb) -- (out);
\end{tikzpicture}"""

(FIG_DIR / "fig_ensemble.tex").write_text(FIG_ENSEMBLE, encoding="utf-8")


# -------------------------------------------------------------
# 5. Figure 5: Publication-Grade Shift Gap Bar Chart
# -------------------------------------------------------------
FIG_SHIFTGAP = r"""\begin{tikzpicture}[
  font=\sffamily,
  scale=1.0,
  y=22cm, x=1.6cm,
  barlabel/.style={font=\footnotesize\bfseries},
  axislabel/.style={font=\scriptsize\bfseries, text=gray!80!black}
]
  % Y Axis and gridlines
  \draw[-{Latex[length=2.2mm]}, thick, gray!70!black] (0,0) -- (0,0.30) node[above, font=\footnotesize\bfseries] {Log Loss (lower is better)};
  \draw[-{Latex[length=2.2mm]}, thick, gray!70!black] (0,0) -- (4.8,0);

  \foreach \val in {0.05, 0.10, 0.15, 0.20, 0.25} {
    \draw[gray!25, dashed] (-0.1, \val) -- (4.5, \val);
    \node[anchor=east, font=\tiny, text=gray!60!black] at (-0.12, \val) {\val};
  }

  % Bar 1: Random-Fold OOF (0.2266)
  \draw[fill=blue!45!teal, draw=blue!60!black, rounded corners=2pt, line width=0.8pt] (0.5, 0) rectangle (1.3, 0.2266);
  \node[barlabel, text=blue!60!black] at (0.9, 0.237) {0.2266};
  \node[axislabel, align=center] at (0.9, -0.025) {Random-Fold\\Nested OOF};

  % Bar 2: Leave-Site-Out OOF (0.2402)
  \draw[fill=orange!55!yellow, draw=orange!70!black, rounded corners=2pt, line width=0.8pt] (1.8, 0) rectangle (2.6, 0.2402);
  \node[barlabel, text=orange!80!black] at (2.2, 0.251) {0.2402};
  \node[axislabel, align=center] at (2.2, -0.025) {Leave-Site-Out\\(Site-Fold OOF)};

  % Bar 3: Public Leaderboard (0.2473)
  \draw[fill=red!50!orange, draw=red!70!black, rounded corners=2pt, line width=0.8pt] (3.1, 0) rectangle (3.9, 0.2473);
  \node[barlabel, text=red!80!black] at (3.5, 0.258) {\textbf{0.2473}};
  \node[axislabel, align=center] at (3.5, -0.025) {Public Leaderboard\\(\textbf{Rank $\sim$16} / 450)};

  % Shift Gap Annotation Bracket (Between Bar 1 and Bar 2)
  \draw[thick, gray!60!black] (0.9, 0.260) -- (0.9, 0.268) -- (2.2, 0.268) -- (2.2, 0.260);
  \node[font=\scriptsize\bfseries, fill=white, inner sep=2pt, text=gray!80!black] at (1.55, 0.270) {Shift Gap $\Delta = +0.0136$};

  % Public Gap Annotation Bracket (Between Bar 1 and Bar 3)
  \draw[thick, red!70!black] (0.9, 0.282) -- (0.9, 0.290) -- (3.5, 0.290) -- (3.5, 0.282);
  \node[font=\scriptsize\bfseries, fill=white, inner sep=2pt, text=red!75!black] at (2.2, 0.292) {Real Test Gap $\Delta = +0.0207$};

\end{tikzpicture}"""

(FIG_DIR / "fig_shiftgap.tex").write_text(FIG_SHIFTGAP, encoding="utf-8")


# -------------------------------------------------------------
# 6. Single-Column Shift Gap Chart for ISBI 2027 (Compact)
# -------------------------------------------------------------
FIG_SHIFTGAP_ISBI = r"""\begin{tikzpicture}[
  font=\sffamily,
  scale=0.85,
  y=17cm, x=1.2cm,
  barlabel/.style={font=\tiny\bfseries},
  axislabel/.style={font=\tiny\bfseries, text=gray!80!black}
]
  \draw[-{Latex[length=1.8mm]}, thick, gray!70!black] (0,0) -- (0,0.30) node[above, font=\tiny\bfseries] {Log loss};
  \draw[-{Latex[length=1.8mm]}, thick, gray!70!black] (0,0) -- (4.2,0);

  \foreach \val in {0.10, 0.20, 0.30} {
    \draw[gray!25, dashed] (-0.08, \val) -- (4.0, \val);
    \node[anchor=east, font=\tiny, text=gray!60!black] at (-0.1, \val) {\val};
  }

  % Bar 1
  \draw[fill=blue!45!teal, draw=blue!60!black, rounded corners=1pt] (0.4, 0) rectangle (1.1, 0.2266);
  \node[barlabel, text=blue!70!black] at (0.75, 0.237) {0.2266};
  \node[axislabel, align=center] at (0.75, -0.03) {Random\\OOF};

  % Bar 2
  \draw[fill=orange!55!yellow, draw=orange!70!black, rounded corners=1pt] (1.5, 0) rectangle (2.2, 0.2402);
  \node[barlabel, text=orange!80!black] at (1.85, 0.251) {0.2402};
  \node[axislabel, align=center] at (1.85, -0.03) {Site-Fold\\OOF};

  % Bar 3
  \draw[fill=red!50!orange, draw=red!70!black, rounded corners=1pt] (2.6, 0) rectangle (3.3, 0.2473);
  \node[barlabel, text=red!80!black] at (2.95, 0.258) {0.2473};
  \node[axislabel, align=center] at (2.95, -0.03) {Public LB\\(Rank $\sim$16)};

  % Gap bracket
  \draw[thick, gray!60!black] (0.75, 0.268) -- (0.75, 0.274) -- (1.85, 0.274) -- (1.85, 0.268);
  \node[font=\tiny\bfseries, fill=white, inner sep=1pt] at (1.3, 0.276) {$\shiftgap = 0.0136$};

  \draw[thick, red!70!black] (0.75, 0.286) -- (0.75, 0.292) -- (2.95, 0.292) -- (2.95, 0.286);
  \node[font=\tiny\bfseries, fill=white, inner sep=1pt, text=red!70!black] at (1.85, 0.294) {Gap $+0.0207$};
\end{tikzpicture}"""

(FIG_DIR / "fig_shiftgap_isbi.tex").write_text(FIG_SHIFTGAP_ISBI, encoding="utf-8")

print("Successfully generated all publication figures in paper_v2/figures/")


In [ ]:
%%writefile scripts/join_parts.py
"""Reassemble submission_v3.zip from downloaded parts and verify the md5 printed by the split cell.
python scripts/join_parts.py data/parts_v3 <md5> [--out submission_v3.zip]"""
import glob, hashlib, sys, zipfile
from pathlib import Path
d, md5 = Path(sys.argv[1]), sys.argv[2]
out = Path(sys.argv[4] if len(sys.argv) > 4 else "submission_v3.zip")
parts = sorted(glob.glob(str(d / "submission_v3.zip.part*")))
with open(out, "wb") as w:
    for p in parts:
        w.write(open(p, "rb").read())
got = hashlib.md5(open(out, "rb").read()).hexdigest()
print(len(parts), "parts ->", out, out.stat().st_size, "bytes; md5", got, "OK" if got == md5 else "MISMATCH")
if got == md5:
    z = zipfile.ZipFile(out); print("testzip:", z.testzip(), "|", len(z.namelist()), "files")


In [ ]:
%%writefile scripts/local_e2e_v3.py
"""Run a submission zip locally exactly as the DrivenData container does, on the smoke-test set, and score it.
    python scripts/local_e2e_v3.py <submission.zip> <smoke_dir>   (smoke_dir holds niftis/ or *.nii.gz, submission_format.csv, test_labels.csv)"""
from __future__ import annotations

import json, os, re, shutil, subprocess, sys, tempfile, time, zipfile
from pathlib import Path

import numpy as np, pandas as pd
from sklearn.metrics import log_loss

zip_path, smoke = Path(sys.argv[1]).resolve(), Path(sys.argv[2]).resolve()
fmt_csv = next(smoke.rglob("submission_format.csv")); src = fmt_csv.parent
niftis = sorted(p for p in src.rglob("*") if p.name.endswith((".nii", ".nii.gz")))
work = Path(tempfile.mkdtemp(prefix="e2e_")); data = work / "data"; (data / "niftis").mkdir(parents=True)
for p in niftis:
    shutil.copy(p, data / "niftis" / p.name)
shutil.copy(fmt_csv, data / "submission_format.csv")
sim = work / "code_execution"; sim.mkdir()
with zipfile.ZipFile(zip_path) as z:
    z.extractall(sim)
assert (sim / "main.py").exists()
env = {**os.environ, "DATSC_DATA_ROOT": str(data), "DATSC_OUTPUT_CSV": "submission.csv"}
t0 = time.time(); r = subprocess.run([sys.executable, "main.py"], cwd=sim, env=env, capture_output=True, text=True); wall = time.time() - t0
print(r.stdout[-3000:]); print(r.stderr[-1500:]); print(f"exit={r.returncode} wall={wall:.0f}s")
assert r.returncode == 0
used = re.findall(r"arm (\S+): checkpoints used (\d+)/(\d+)", r.stdout)
print("checkpoints used per arm:", used)
assert used and all(int(u) == int(m) for _, u, m in used), "ARM CHECKPOINTS NOT ALL USED"
# Torch arms print "checkpoints used N/N"; the featc/LR arms print a different line and were
# invisible to the assertion above, so a silently-dead arm still passed the gate.
man = json.loads((sim / "artifacts" / "manifest.json").read_text())
spoke = {a for a, _, _ in used} | set(re.findall(r"arm (\S+): (?:feat model|LR) on \d+ volumes", r.stdout))
silent = [a["name"] for a in man["arms"] if a["name"] not in spoke]
assert not silent, f"ARMS IN MANIFEST BUT SILENT IN OUTPUT: {silent} (ran: {sorted(spoke)})"
for pat in (r"ladder: [^\n]+", r"arm \S+ failed: [^\n]+", r"checkpoint failed \([^\n]+", r"Traceback"):
    hit = re.findall(pat, r.stdout + r.stderr)
    assert not hit, f"DEGRADED RUN, ensemble is not what was gated: {hit[:3]}"
assert man.get("typ") is None, "manifest typ is set: inference would print test-set statistics (rules)"
fmt = pd.read_csv(fmt_csv, dtype={"uid": str}); sub = pd.read_csv(sim / "submission.csv", dtype={"uid": str})
assert list(sub.columns) == ["uid", "is_pathologic"] and list(sub.uid) == list(fmt.uid) and sub.is_pathologic.between(0, 1).all()
lab = pd.read_csv(next(src.rglob("test_labels.csv")), dtype={"uid": str}).set_index("uid").loc[sub.uid]
y = lab.is_pathologic.values.astype(float); p = sub.is_pathologic.values
print(f"FORMAT OK {sub.shape} | smoke-test log loss = {log_loss(y, p):.4f} (prior-only {log_loss(y, np.full(len(y), 0.548)):.4f}) | acc@0.5 = {((p > 0.5) == (y > 0.5)).mean():.3f} | {wall:.0f}s")
print("work dir:", work)


In [ ]:
%%writefile scripts/make_folds.py
"""cache/meta.csv + cache/*.npz -> cache/folds.csv (uid, is_pathologic, group, fold)."""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd

from datsc.folds import assign_folds, embed, group_near_duplicates

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--thresh", type=float, default=0.98)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    cache = Path(a.cache)
    meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
    meta = meta[meta["ok"]].reset_index(drop=True)
    crops = np.stack([np.load(cache / f"{u}.npz")["crop"] for u in meta["uid"]])
    groups = group_near_duplicates(embed(crops), a.thresh)
    n_dup = len(groups) - len(np.unique(groups))
    print(f"near-duplicate pairs collapsed into groups: {n_dup}")
    meta["group"] = groups
    meta["fold"] = assign_folds(meta["is_pathologic"].values.astype(int), groups, 5, a.seed)
    meta[["uid", "is_pathologic", "group", "fold", "native_spacing_mm", "fallback"]].to_csv(cache / "folds.csv", index=False)
    print(meta.groupby("fold")["is_pathologic"].agg(["count", "mean"]))


In [ ]:
%%writefile scripts/make_forharsh_pdf.py
"""Render forharsh.md into a styled forharsh.pdf (reportlab). python scripts/make_forharsh_pdf.py"""
from __future__ import annotations

import re
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import (KeepTogether, PageBreak, Paragraph, Preformatted, SimpleDocTemplate, Spacer, Table, TableStyle)

ROOT = Path(__file__).resolve().parent.parent
SRC, OUT = ROOT / "forharsh.md", ROOT / "forharsh.pdf"
INK, ACCENT, ACCENT2, SOFT, CODEBG, MUTED = colors.HexColor("#1f2933"), colors.HexColor("#0b6e4f"), colors.HexColor("#c8553d"), colors.HexColor("#e6f2ee"), colors.HexColor("#f4f6f8"), colors.HexColor("#5b6773")
W, H = A4

ss = getSampleStyleSheet()
S = {
    "title": ParagraphStyle("t", parent=ss["Title"], fontName="Helvetica-Bold", fontSize=22, leading=26, textColor=colors.white, alignment=TA_LEFT, spaceAfter=0),
    "sub": ParagraphStyle("s", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.5, leading=12.5, textColor=colors.white),
    "h2": ParagraphStyle("h2", parent=ss["Heading2"], fontName="Helvetica-Bold", fontSize=12.5, leading=15, textColor=ACCENT, spaceBefore=10, spaceAfter=4),
    "body": ParagraphStyle("b", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.2, leading=12.2, textColor=INK, spaceAfter=3),
    "step": ParagraphStyle("st", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.2, leading=12.2, textColor=INK, leftIndent=16, spaceAfter=3),
    "bullet": ParagraphStyle("bu", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.2, leading=12.2, textColor=INK, leftIndent=12, bulletIndent=2, spaceAfter=2),
    "code": ParagraphStyle("c", parent=ss["Code"], fontName="Courier", fontSize=7.9, leading=9.8, textColor=INK, backColor=CODEBG, borderColor=colors.HexColor("#d5dbe0"), borderWidth=0.6, borderPadding=5, leftIndent=6, spaceBefore=3, spaceAfter=7),
    "cell": ParagraphStyle("ce", parent=ss["BodyText"], fontName="Helvetica", fontSize=8.4, leading=10.6, textColor=INK),
    "cellh": ParagraphStyle("ch", parent=ss["BodyText"], fontName="Helvetica-Bold", fontSize=8.4, leading=10.6, textColor=colors.white),
    "kicker": ParagraphStyle("k", parent=ss["BodyText"], fontName="Helvetica-Bold", fontSize=8, leading=10, textColor=ACCENT2, spaceAfter=1),
}


def inline(t: str) -> str:
    t = t.replace("⋯", "...").replace("→", "-&gt;").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("-&amp;gt;", "&rarr;")
    t = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", t)
    t = re.sub(r"`([^`]+)`", r'<font face="Courier" size="8" color="#0b6e4f">\1</font>', t)
    return t


def wrap_code(line: str, width: int = 98) -> str:
    """Soft-wrap long command lines at spaces; continuation lines are indented (the .md keeps the full line)."""
    out, cur = [], ""
    for tok in line.split(" "):
        if cur and len(cur) + 1 + len(tok) > width:
            out.append(cur); cur = "    " + tok
        else:
            cur = (cur + " " + tok) if cur else tok
    out.append(cur)
    return "\n".join(out)


def header_block(title: str, sub: str):
    t = Table([[Paragraph(title, S["title"])], [Paragraph(sub, S["sub"])]], colWidths=[W - 30 * mm])
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), ACCENT), ("LEFTPADDING", (0, 0), (-1, -1), 12), ("RIGHTPADDING", (0, 0), (-1, -1), 12),
                           ("TOPPADDING", (0, 0), (-1, 0), 10), ("BOTTOMPADDING", (0, -1), (-1, -1), 10)]))
    return t


def section(title: str):
    bar = Table([[Paragraph(inline(title), S["h2"])]], colWidths=[W - 30 * mm])
    bar.setStyle(TableStyle([("LINEBELOW", (0, 0), (-1, -1), 1.2, ACCENT), ("LEFTPADDING", (0, 0), (-1, -1), 0), ("BOTTOMPADDING", (0, 0), (-1, -1), 2)]))
    return bar


def md_table(rows):
    cells = [r.strip().strip("|").split("|") for r in rows if not set(r.replace("|", "").strip()) <= set("-: ")]
    n = max(len(r) for r in cells)
    data = [[Paragraph(inline(c.strip()), S["cellh"] if i == 0 else S["cell"]) for c in (r + [""] * (n - len(r)))] for i, r in enumerate(cells)]
    widths = [(W - 30 * mm) * f for f in ([0.22, 0.48, 0.30] if n == 3 else [1 / n] * n)]
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), ACCENT), ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, SOFT]),
                           ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#c9d3dc")), ("VALIGN", (0, 0), (-1, -1), "TOP"),
                           ("LEFTPADDING", (0, 0), (-1, -1), 5), ("RIGHTPADDING", (0, 0), (-1, -1), 5), ("TOPPADDING", (0, 0), (-1, -1), 4), ("BOTTOMPADDING", (0, 0), (-1, -1), 4)]))
    return t


def footer(canvas, doc):
    canvas.saveState(); canvas.setFont("Helvetica", 7.5); canvas.setFillColor(MUTED)
    canvas.drawString(15 * mm, 9 * mm, "forharsh — DaT Parkinson's Challenge playbook until Sept 16 2026 · repo H:\\augsepthacks\\parkinsonnn")
    canvas.drawRightString(W - 15 * mm, 9 * mm, f"page {doc.page}")
    canvas.setStrokeColor(colors.HexColor("#c9d3dc")); canvas.line(15 * mm, 12 * mm, W - 15 * mm, 12 * mm); canvas.restoreState()


lines = SRC.read_text(encoding="utf-8").split("\n")
flow, buf, tbl, in_code = [], [], [], False
title = lines[0][2:] if lines[0].startswith("# ") else "forharsh"
# subtitle = first paragraph after the title
intro = next((l for l in lines[1:] if l.strip()), "")
flow.append(header_block(title.split(" — ")[0] if " — " in title else title, inline(title.split(" — ", 1)[1] if " — " in title else "")))
flow.append(Spacer(1, 6))
flow.append(Paragraph(inline(intro), S["body"])); flow.append(Spacer(1, 4))
skip_intro = True
for line in lines[1:]:
    if skip_intro:
        if line.strip() == intro.strip():
            skip_intro = False
        continue
    if line.strip().startswith("```"):
        if in_code:
            ind = min((len(x) - len(x.lstrip()) for x in buf if x.strip()), default=0)
            flow.append(Preformatted("\n".join(wrap_code(x[ind:]) for x in buf), S["code"])); buf = []
        in_code = not in_code; continue
    if in_code:
        buf.append(line); continue
    if line.startswith("|"):
        tbl.append(line); continue
    if tbl:
        flow.append(md_table(tbl)); flow.append(Spacer(1, 6)); tbl = []
    if line.startswith("## "):
        flow.append(section(line[3:]))
    elif re.match(r"^\d+\.\s", line):
        num, rest = line.split(".", 1)
        flow.append(Paragraph(f'<font color="#c8553d"><b>{num}.</b></font> {inline(rest.strip())}', S["step"]))
    elif line.startswith("- "):
        flow.append(Paragraph(inline(line[2:]), S["bullet"], bulletText="•"))
    elif line.strip():
        flow.append(Paragraph(inline(line), S["body"]))
    else:
        flow.append(Spacer(1, 3))
if tbl:
    flow.append(md_table(tbl))
doc = SimpleDocTemplate(str(OUT), pagesize=A4, leftMargin=15 * mm, rightMargin=15 * mm, topMargin=14 * mm, bottomMargin=16 * mm, title="forharsh", author="Claude + Harshdip")
doc.build(flow, onFirstPage=footer, onLaterPages=footer)
print("wrote", OUT, OUT.stat().st_size, "bytes")


In [ ]:
%%writefile scripts/make_lessons_pdf.py
"""Render a markdown file into a styled PDF (reportlab).

    python scripts/make_lessons_pdf.py                       # LESSONS.md  -> LESSONS.pdf
    python scripts/make_lessons_pdf.py RUNBOOK.md RUNBOOK.pdf
"""
from __future__ import annotations

import re
import sys
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import (HRFlowable, Paragraph, Preformatted, SimpleDocTemplate, Spacer, Table, TableStyle)

ROOT = Path(__file__).resolve().parent.parent
SRC = ROOT / (sys.argv[1] if len(sys.argv) > 1 else "LESSONS.md")
OUT = ROOT / (sys.argv[2] if len(sys.argv) > 2 else SRC.stem + ".pdf")
INK, ACCENT, ACCENT2, SOFT, CODEBG, MUTED = (colors.HexColor("#1f2933"), colors.HexColor("#24476b"), colors.HexColor("#b0562f"),
                                             colors.HexColor("#eaf0f6"), colors.HexColor("#f4f6f8"), colors.HexColor("#5b6773"))
W, H = A4

ss = getSampleStyleSheet()
S = {
    "title": ParagraphStyle("t", parent=ss["Title"], fontName="Helvetica-Bold", fontSize=21, leading=25, textColor=colors.white, alignment=TA_LEFT, spaceAfter=0),
    "sub": ParagraphStyle("s", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.5, leading=13, textColor=colors.white),
    "h2": ParagraphStyle("h2", parent=ss["Heading2"], fontName="Helvetica-Bold", fontSize=13, leading=16, textColor=ACCENT, spaceBefore=12, spaceAfter=5),
    "h3": ParagraphStyle("h3", parent=ss["Heading3"], fontName="Helvetica-Bold", fontSize=10.4, leading=13, textColor=ACCENT2, spaceBefore=9, spaceAfter=3),
    "body": ParagraphStyle("b", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.3, leading=12.6, textColor=INK, spaceAfter=4),
    "step": ParagraphStyle("st", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.3, leading=12.6, textColor=INK, leftIndent=16, spaceAfter=3),
    "bullet": ParagraphStyle("bu", parent=ss["BodyText"], fontName="Helvetica", fontSize=9.3, leading=12.6, textColor=INK, leftIndent=13, bulletIndent=3, spaceAfter=2),
    "code": ParagraphStyle("c", parent=ss["Code"], fontName="Courier", fontSize=7.9, leading=9.8, textColor=INK, backColor=CODEBG,
                           borderColor=colors.HexColor("#d5dbe0"), borderWidth=0.6, borderPadding=5, leftIndent=6, spaceBefore=3, spaceAfter=7),
    "cell": ParagraphStyle("ce", parent=ss["BodyText"], fontName="Helvetica", fontSize=8.3, leading=10.5, textColor=INK),
    "cellh": ParagraphStyle("ch", parent=ss["BodyText"], fontName="Helvetica-Bold", fontSize=8.3, leading=10.5, textColor=colors.white),
}


def _esc(s: str) -> str:
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def inline(t: str) -> str:
    """Code spans are stashed before bold/italic run, so a literal * inside `oof_*.csv` cannot open a tag."""
    spans: list[str] = []

    def stash(m):
        spans.append(m.group(1))
        return f"\x00{len(spans) - 1}\x00"

    t = re.sub(r"`([^`]+)`", stash, t)
    t = _esc(t)
    t = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", t)
    t = re.sub(r"(?<!\*)\*(?!\s)([^*]+?)(?<!\s)\*(?!\*)", r"<i>\1</i>", t)
    return re.sub(r"\x00(\d+)\x00", lambda m: f'<font face="Courier" size="8" color="#24476b">{_esc(spans[int(m.group(1))])}</font>', t)


def header_block(title: str, sub: str):
    t = Table([[Paragraph(title, S["title"])], [Paragraph(sub, S["sub"])]], colWidths=[W - 30 * mm])
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), ACCENT), ("LEFTPADDING", (0, 0), (-1, -1), 12), ("RIGHTPADDING", (0, 0), (-1, -1), 12),
                           ("TOPPADDING", (0, 0), (-1, 0), 11), ("BOTTOMPADDING", (0, -1), (-1, -1), 11)]))
    return t


def section(title: str):
    bar = Table([[Paragraph(inline(title), S["h2"])]], colWidths=[W - 30 * mm])
    bar.setStyle(TableStyle([("LINEBELOW", (0, 0), (-1, -1), 1.1, ACCENT), ("LEFTPADDING", (0, 0), (-1, -1), 0), ("BOTTOMPADDING", (0, 0), (-1, -1), 3)]))
    return bar


def md_table(rows):
    cells = [r.strip().strip("|").split("|") for r in rows if not set(r.replace("|", "").strip()) <= set("-: ")]
    n = max(len(r) for r in cells)
    data = [[Paragraph(inline(c.strip()), S["cellh"] if i == 0 else S["cell"]) for c in (r + [""] * (n - len(r)))] for i, r in enumerate(cells)]
    widths = [(W - 30 * mm) * f for f in ([0.09, 0.55, 0.20, 0.16] if n == 4 else [1 / n] * n)]
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), ACCENT), ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, SOFT]),
                           ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#c9d3dc")), ("VALIGN", (0, 0), (-1, -1), "TOP"),
                           ("LEFTPADDING", (0, 0), (-1, -1), 5), ("RIGHTPADDING", (0, 0), (-1, -1), 5),
                           ("TOPPADDING", (0, 0), (-1, -1), 4), ("BOTTOMPADDING", (0, 0), (-1, -1), 4)]))
    return t


FOOT = ""


def footer(canvas, doc):
    canvas.saveState(); canvas.setFont("Helvetica", 7.5); canvas.setFillColor(MUTED)
    canvas.drawString(15 * mm, 9 * mm, FOOT[:110])
    canvas.drawRightString(W - 15 * mm, 9 * mm, f"page {doc.page}")
    canvas.setStrokeColor(colors.HexColor("#c9d3dc")); canvas.line(15 * mm, 12 * mm, W - 15 * mm, 12 * mm); canvas.restoreState()


lines = SRC.read_text(encoding="utf-8").split("\n")
title = lines[0][2:] if lines[0].startswith("# ") else SRC.stem
intro = next((l for l in lines[1:] if l.strip()), "")
FOOT = re.sub(r"[*`]", "", title)
flow = [header_block(title, inline(intro)), Spacer(1, 8)]

buf, tbl, in_code, seen_intro = [], [], False, False
for line in lines[1:]:
    if not seen_intro:
        if line.strip() == intro.strip():
            seen_intro = True
        continue
    if line.strip().startswith("```"):
        if in_code:
            flow.append(Preformatted("\n".join(buf), S["code"])); buf = []
        in_code = not in_code; continue
    if in_code:
        buf.append(line); continue
    if line.startswith("|"):
        tbl.append(line); continue
    if tbl:
        flow.append(md_table(tbl)); flow.append(Spacer(1, 7)); tbl = []
    if line.strip() == "---":
        flow.append(Spacer(1, 5)); flow.append(HRFlowable(width="100%", thickness=0.6, color=colors.HexColor("#c9d3dc"))); flow.append(Spacer(1, 3))
    elif line.startswith("### "):
        flow.append(Paragraph(inline(line[4:]), S["h3"]))
    elif line.startswith("## "):
        flow.append(section(line[3:]))
    elif re.match(r"^\d+\.\s", line):
        num, rest = line.split(".", 1)
        flow.append(Paragraph(f'<font color="#b0562f"><b>{num}.</b></font> {inline(rest.strip())}', S["step"]))
    elif line.startswith("- "):
        flow.append(Paragraph(inline(line[2:]), S["bullet"], bulletText="•"))
    elif line.strip():
        flow.append(Paragraph(inline(line), S["body"]))
    else:
        flow.append(Spacer(1, 3))
if tbl:
    flow.append(md_table(tbl))

doc = SimpleDocTemplate(str(OUT), pagesize=A4, leftMargin=15 * mm, rightMargin=15 * mm, topMargin=14 * mm, bottomMargin=16 * mm,
                        title="What I learned running an ML competition", author="Harshdip Saha")
doc.build(flow, onFirstPage=footer, onLaterPages=footer)
print("wrote", OUT, OUT.stat().st_size, "bytes")


In [ ]:
%%writefile scripts/make_prior_submission.py
"""Build a prior-only submission.zip (main.py + datsc + tests_synth.py + calib.json with lam=1)."""
from __future__ import annotations

import json
import sys
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
from datsc.config import DEFAULT_PRIOR

if __name__ == "__main__":
    out = ROOT / "submission_prior.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ROOT / "main.py", "main.py")
        z.write(ROOT / "tests_synth.py", "tests_synth.py")
        for p in sorted((ROOT / "datsc").glob("*.py")):
            z.write(p, f"datsc/{p.name}")
        z.writestr("artifacts/calib.json", json.dumps({"w": 0.0, "T": 1.0, "lam": 1.0, "pi": DEFAULT_PRIOR}))
    print("wrote", out)


In [ ]:
%%writefile scripts/make_soft_targets.py
"""Run 8: the soft-target table = Master v2's NESTED out-of-fold predictions (equal weights over the 5 pillars, temperature refit per
fold with floor 0.9, prior mix 0.02) for every training scan -> kaggle/run8_distill/soft_master_v2.csv (uid,y,p,loss,fold). No scan data inside.
python scripts/make_soft_targets.py [--src data/master_pool/artifacts_v3 ...] [--out kaggle/run8_distill/soft_master_v2.csv] [--T-floor 0.9]"""
import argparse, glob, os, sys
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
src = open(Path(__file__).resolve().parent / "fit_stack_v3.py", encoding="utf-8").read().split('if __name__ == "__main__":')[0]
ns = {"__name__": "fs", "__file__": str((Path(__file__).resolve().parent / "fit_stack_v3.py"))}; exec(src, ns)
logloss, sigmoid, logit, fit_temperature = ns["logloss"], ns["sigmoid"], ns["logit"], ns["fit_temperature"]
PILLARS = {"a3d_hemi": "r3d_hemi48_f?_s?", "a3d_ssl": "r3d_ssl_f?_s?", "featc": "featcarm", "a2d_cnx_cor": "r2d_convnext_tiny_cor_f?_s?", "a2d_v2s": "r2d_efficientnet_v2_s_f?_s[01]"}

def load_arm(srcs, pat):
    fs = sorted(set(f for d in srcs for f in glob.glob(str(Path(d) / f"oof_{pat}.csv"))))
    if not fs:
        return None
    df = pd.concat([pd.read_csv(f, dtype={"uid": str}) for f in fs])
    return df.groupby("uid").agg(y=("y", "first"), p=("p", "mean"))

def nested_predictions(srcs, cache="cache_v4", T_floor=0.9, lam=0.02, pi=0.548, arms=PILLARS):
    oofs = {n: load_arm(srcs, p) for n, p in arms.items()}
    missing = [n for n, o in oofs.items() if o is None or len(o) < 1300]
    assert not missing, f"missing/incomplete OOF for {missing}"
    uids = sorted(set.intersection(*[set(o.index) for o in oofs.values()]))
    y = oofs[next(iter(arms))].loc[uids, "y"].values.astype(float)
    Z = np.stack([logit(oofs[n].loc[uids, "p"].values) for n in arms])
    folds = pd.read_csv(Path(cache) / "folds.csv", dtype={"uid": str}).set_index("uid").loc[uids, "fold"].values
    w = np.ones(len(arms)) / len(arms); pn = np.zeros(len(y))
    for k in np.unique(folds):
        tr, va = folds != k, folds == k
        T = max(fit_temperature((w[:, None] * Z[:, tr]).sum(0), y[tr]), T_floor)
        pn[va] = (1 - lam) * sigmoid((w[:, None] * Z[:, va]).sum(0) / T) + lam * pi
    loss = -(y * np.log(pn) + (1 - y) * np.log(1 - pn))
    return pd.DataFrame({"uid": uids, "y": y, "p": pn, "loss": loss, "fold": folds})

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--src", action="append", default=None); ap.add_argument("--out", default="kaggle/run8_distill/soft_master_v2.csv")
    ap.add_argument("--cache", default="cache_v4"); ap.add_argument("--T-floor", type=float, default=0.9); a = ap.parse_args()
    srcs = a.src or ["data/master_pool/artifacts_v3"]
    df = nested_predictions(srcs, a.cache, a.T_floor); Path(a.out).parent.mkdir(parents=True, exist_ok=True); df.to_csv(a.out, index=False)
    o = df.sort_values("loss", ascending=False)
    print(f"wrote {a.out}: {len(df)} scans; nested log loss {df.loss.mean():.4f}; top-100 scans carry {o.loss.iloc[:100].sum()/df.loss.sum()*100:.1f}% of the loss; "
          f"{int((df.loss > 1).sum())} scans > 1 nat; confident mistakes {int(((df.p > .8) & (df.y == 0)).sum() + ((df.p < .2) & (df.y == 1)).sum())}")


In [ ]:
%%writefile scripts/pack_submission.py
"""Build submission.zip: main.py, tests_synth.py, datsc/*.py, artifacts/{lr.json,calib.json,cnn_f*_s*.pt}."""
from __future__ import annotations

import argparse
import sys
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--artifacts", default="artifacts")
    ap.add_argument("--out", default="submission.zip")
    a = ap.parse_args()
    art = Path(a.artifacts)
    need = [art / "lr.json", art / "calib.json"]
    weights = sorted(art.glob("cnn_f*_s*.pt"))   # hotspot + _com checkpoints (dual-crop ensemble)
    for p in need:
        assert p.exists(), f"missing {p}"
    with zipfile.ZipFile(ROOT / a.out, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ROOT / "main.py", "main.py")
        z.write(ROOT / "tests_synth.py", "tests_synth.py")
        for p in sorted((ROOT / "datsc").glob("*.py")):
            z.write(p, f"datsc/{p.name}")
        for p in need + weights:
            z.write(p, f"artifacts/{p.name}")
    print(f"wrote {a.out} with {len(weights)} checkpoints")


In [ ]:
%%writefile scripts/pack_v3.py
"""Build the run-3 submission zip: main_v3.py -> main.py, tests_synth.py, datsc/*.py, artifacts/manifest.json + every
checkpoint the manifest references. python scripts/pack_v3.py [--artifacts artifacts_v3] [--out submission_v3.zip]"""
from __future__ import annotations

import argparse
import json
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--artifacts", default="artifacts_v3")
    ap.add_argument("--out", default="submission_v3.zip")
    a = ap.parse_args()
    art = Path(a.artifacts)
    man = json.loads((art / "manifest.json").read_text())
    cks = [ck for arm in man["arms"] for ck in arm["checkpoints"]]
    with zipfile.ZipFile(ROOT / a.out, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ROOT / "main_v3.py", "main.py")
        z.write(ROOT / "tests_synth.py", "tests_synth.py")
        for p in sorted((ROOT / "datsc").glob("*.py")):
            z.write(p, f"datsc/{p.name}")
        z.write(art / "manifest.json", "artifacts/manifest.json")
        for extra in ("template.npz", "lr_v3.json"):
            if (art / extra).exists() and f"artifacts/{extra}" not in cks:
                z.write(art / extra, f"artifacts/{extra}")
        for ck in cks:
            src = art / Path(ck).name
            assert src.exists(), src
            z.write(src, ck)
    size = (ROOT / a.out).stat().st_size / 1e6
    print(f"wrote {a.out}: {len(cks)} checkpoints across {len(man['arms'])} arms, {size:.0f} MB, "
          f"OOF logloss {man.get('oof_logloss', float('nan')):.4f}")


In [ ]:
%%writefile scripts/pseudo_sites.py
"""Pseudo-site labels from acquisition fingerprints (native spacing x native shape x intensity regime),
a leave-pseudo-site-out fold assignment for shift-aware validation/calibration, and reuse of the run-2
random folds so run-3 OOF predictions stay comparable/stackable with run-2 ones.

python scripts/pseudo_sites.py --cache cache_v3 [--run2-folds cache/folds.csv]
writes cache_v3/sites.csv (uid, site, site_id), cache_v3/folds.csv (copied run-2 folds), cache_v3/folds_site.csv
"""
from __future__ import annotations

import argparse
import shutil
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd


def site_key(row) -> str:
    sp = f"{row.sp_x:.2f}"
    shape = f"{int(row.shape_x)}x{int(row.shape_y)}x{int(row.shape_z)}"
    regime = "ultralow" if row.p99 < 20 else ("low" if row.p99 < 2000 else "high")
    return f"{sp}|{shape}|{regime}"


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache_v3")
    ap.add_argument("--run2-folds", default="cache/folds.csv")
    ap.add_argument("--min-site", type=int, default=25, help="smaller fingerprints are merged into 'other'")
    ap.add_argument("--n-site-folds", type=int, default=5)
    a = ap.parse_args()
    cache = Path(a.cache)
    meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
    meta["site"] = meta.apply(site_key, axis=1)
    counts = meta["site"].value_counts()
    small = counts[counts < a.min_site].index
    meta.loc[meta["site"].isin(small), "site"] = "other"
    tab = meta.groupby("site").agg(n=("uid", "size"), abnormal=("is_pathologic", "mean")).sort_values("n", ascending=False)
    print(tab.to_string())
    sites = {s: i for i, s in enumerate(tab.index)}
    meta["site_id"] = meta["site"].map(sites)
    meta[["uid", "site", "site_id"]].to_csv(cache / "sites.csv", index=False)

    # leave-pseudo-site-out folds: greedy balance sites into n folds by size ('other' spread by hashing)
    order = tab.drop(index="other", errors="ignore").sort_values("n", ascending=False).index.tolist()
    load = np.zeros(a.n_site_folds); site_fold = {}
    for s in order:
        k = int(np.argmin(load)); site_fold[s] = k; load[k] += tab.loc[s, "n"]
    fold = meta["site"].map(site_fold)
    rng = np.random.RandomState(0)
    other = meta["site"] == "other"
    fold[other] = rng.randint(0, a.n_site_folds, other.sum())
    fs = meta[["uid", "is_pathologic"]].copy(); fs["fold"] = fold.astype(int).values
    fs.to_csv(cache / "folds_site.csv", index=False)
    print("\nsite folds:\n", fs.groupby("fold").agg(n=("uid", "size"), abnormal=("is_pathologic", "mean")).to_string())

    r2 = Path(a.run2_folds)
    if r2.exists():
        f2 = pd.read_csv(r2, dtype={"uid": str})
        assert set(f2["uid"]) == set(meta["uid"]), "run-2 folds do not cover the same uids"
        shutil.copy(r2, cache / "folds.csv")
        print(f"\ncopied run-2 folds {r2} -> {cache/'folds.csv'}")
    else:
        print(f"\nWARNING: {r2} not found; build folds with scripts/make_folds.py")


In [ ]:
%%writefile scripts/qc_grid.py
"""PNG grids of cached crops for eyeballing. Usage: python scripts/qc_grid.py --cache cache --out qc"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def grid(cache: Path, uids, title, out_png: Path, ncol=6):
    n = len(uids)
    if n == 0:
        return
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.2 * ncol, 2.4 * nrow))
    for ax, uid in zip(np.ravel(axes), list(uids) + [None] * (nrow * ncol - n)):
        ax.axis("off")
        if uid is None:
            continue
        z = np.load(cache / f"{uid}.npz")
        crop = z["crop"].astype(np.float32)
        ax.imshow(crop[:, :, crop.shape[2] // 2].T, cmap="hot", origin="lower", vmin=0, vmax=max(crop.max(), 1e-3))
        ax.set_title(uid, fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_png, dpi=90)
    plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--out", default="qc")
    a = ap.parse_args()
    cache, out = Path(a.cache), Path(a.out)
    out.mkdir(exist_ok=True)
    meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
    ok = meta[meta["ok"]]
    rng = np.random.default_rng(0)
    grid(cache, rng.choice(ok["uid"], min(30, len(ok)), replace=False), "random", out / "random.png")
    grid(cache, ok[ok["n_levels"] < 12]["uid"], "ultra-low grey levels (<12)", out / "ultralow.png")
    grid(cache, ok[ok["z_extent_mm"] < 120]["uid"], "short z extent (<120mm)", out / "short_z.png")
    grid(cache, ok[ok["fallback"]]["uid"], "anchor fallback", out / "fallback.png")
    print(meta[["ok", "fallback"]].agg(["sum", "mean"]))
    print("ultra-low:", int((ok["n_levels"] < 12).sum()), "short-z:", int((ok["z_extent_mm"] < 120).sum()))


In [ ]:
%%writefile scripts/register_cache.py
"""cache_v3 (CoM boxes, p75 units) -> cache_v4 (registered to the study template, reference-region units).

Uses exactly datsc.register.register_and_normalise, i.e. the same step main.py applies to a test scan after
prepare_volume's CoM box. Output layout == cache_v3 plus the registration stats appended to stats.npy and meta.csv.

python scripts/register_cache.py --src cache_v3 --out cache_v4 --template artifacts/template.npz [--workers 10]
"""
from __future__ import annotations

import argparse
import os
import shutil
import sys
import time
from multiprocessing import get_context
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd

from datsc.register import N_REG_STATS, REG_STAT_NAMES, Template, register_and_normalise
from datsc.volume import VOL_SHAPE

_T = None


def _init(path):
    global _T
    _T = Template.load(path)


def _one(args):
    i, box = args
    try:
        vol, st, _ = register_and_normalise(box, _T)
        return i, vol.astype(np.float16), st, ""
    except Exception as e:  # noqa: BLE001
        return i, box.astype(np.float16), np.zeros(N_REG_STATS, np.float32), repr(e)[:120]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="cache_v3")
    ap.add_argument("--out", default="cache_v4")
    ap.add_argument("--template", default="artifacts/template.npz")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 2))
    a = ap.parse_args()
    src, out = Path(a.src), Path(a.out); out.mkdir(parents=True, exist_ok=True)
    vols = np.load(src / "vols.npy", mmap_mode="r"); stats = np.load(src / "stats.npy"); meta = pd.read_csv(src / "meta.csv", dtype={"uid": str})
    N = len(meta)
    vout = np.lib.format.open_memmap(out / "vols.npy", mode="w+", dtype=np.float16, shape=(N,) + VOL_SHAPE)
    rstats = np.zeros((N, N_REG_STATS), np.float32); errs = [""] * N
    t0 = time.time()
    with get_context("spawn").Pool(a.workers, initializer=_init, initargs=(a.template,)) as pool:
        for k, (i, vol, st, err) in enumerate(pool.imap_unordered(_one, ((i, np.asarray(vols[i], np.float32)) for i in range(N)), chunksize=4)):
            vout[i] = vol; rstats[i] = st; errs[i] = err
            if (k + 1) % 200 == 0 or k + 1 == N:
                print(f"{k+1}/{N} in {time.time()-t0:.0f}s", flush=True)
    vout.flush()
    # stats: cache_v3 may predate the n_levels_native column; pad with nan so the layout matches prepare_volume(template=...)
    if stats.shape[1] == 15:
        stats = np.concatenate([stats, np.full((N, 1), np.nan, np.float32)], axis=1)
    np.save(out / "stats.npy", np.concatenate([stats, rstats], axis=1).astype(np.float32))
    shutil.copy(src / "scalars.npy", out / "scalars.npy")
    for k, name in enumerate(REG_STAT_NAMES):
        meta[name] = rstats[:, k]
    meta["reg_err"] = errs
    meta.to_csv(out / "meta.csv", index=False)
    for f in ("folds.csv", "folds_run2.csv", "folds_site.csv", "sites.csv"):
        if (src / f).exists():
            shutil.copy(src / f, out / f)
    ok = rstats[:, 2] > 0.5
    print(f"registered {int(ok.sum())}/{N} ({100*ok.mean():.1f}%) in {time.time()-t0:.0f}s; fallback {int((~ok).sum())}; errors {sum(1 for e in errs if e)}")
    print(f"ncc_reg p2/10/50/90 = {np.percentile(rstats[:,1],[2,10,50,90]).round(3)} | scale p10/50/90 = {np.percentile(rstats[:,3],[10,50,90]).round(3)} | "
          f"ref_mean p10/50/90 = {np.percentile(rstats[:,6],[10,50,90]).round(3)}")
    lab = meta["is_pathologic"].values
    print(f"fallback rate: normals {100*(~ok[lab==0]).mean():.1f}%  abnormals {100*(~ok[lab==1]).mean():.1f}%")


In [ ]:
%%writefile scripts/reoof_v3.py
"""Recompute OOF predictions for every CV checkpoint in --art with a given TTA (so stacking weights are fitted on
exactly what main.py produces). Updates oof_<tag>.csv, and val_logloss/val_auroc + spec.tta_z in <tag>.json.

python scripts/reoof_v3.py --art artifacts_v3 --cache cache_v4 --tta-z -1 0 1 [--only "r2d_*"]
"""
from __future__ import annotations

import argparse
import glob
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from datsc.data_v3 import VolDataset, VolumeCache
from datsc.models_v3 import build_model
from datsc.train_v3 import InputSpec, auroc_np, logloss_np, predict

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--art", default="artifacts_v3")
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--tta-z", type=int, nargs="+", default=[-1, 0, 1])
    ap.add_argument("--tta-affine", type=int, default=None, help="override the spec's affine-TTA count (None = keep the checkpoint's)")
    ap.add_argument("--only", default="*")
    ap.add_argument("--batch-size", type=int, default=32)
    a = ap.parse_args()
    art, cache = Path(a.art), VolumeCache(a.cache)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    for jp in sorted(glob.glob(str(art / f"{a.only}.json"))):
        meta = json.loads(Path(jp).read_text())
        if "spec" not in meta or meta.get("fold", -1) < 0 or meta["spec"].get("arm") not in ("2d", "3d", "tri"):
            continue
        tag = meta["tag"]; ck = art / f"{tag}.pt"
        if not ck.exists():
            continue
        spec_d = dict(meta["spec"]); spec_d["tta_z"] = a.tta_z if spec_d["arm"] in ("2d", "tri") else [0]
        if a.tta_affine is not None:
            spec_d["tta_affine"] = a.tta_affine
        from datsc.train_b import make_spec
        spec = make_spec(spec_d)
        folds = pd.read_csv(Path(a.cache) / meta.get("folds_csv", "folds.csv"), dtype={"uid": str})
        va = folds[(folds["fold"] == meta["fold"]) & folds["uid"].isin(cache.index)]
        dl = DataLoader(VolDataset(cache, va["uid"], va["is_pathologic"], meta.get("norm", "ref")), batch_size=a.batch_size, num_workers=2)
        model = build_model(spec.arm, **meta["model_kw"]).to(device)
        model.load_state_dict(torch.load(ck, map_location=device, weights_only=True)); model.eval()
        p, uids = predict(model, dl, spec, device, tta_flip=True)
        y = va.set_index("uid").loc[uids, "is_pathologic"].values.astype(float)
        old_ll = meta.get("val_logloss")
        meta["val_logloss"], meta["val_auroc"], meta["spec"] = logloss_np(y, p), auroc_np(y, p), spec.to_dict()
        pd.DataFrame({"uid": uids, "y": y, "p": p}).to_csv(art / f"oof_{tag}.csv", index=False)
        Path(jp).write_text(json.dumps(meta))
        print(f"{tag}: logloss {old_ll if old_ll is None else round(old_ll, 4)} -> {meta['val_logloss']:.4f}  auroc {meta['val_auroc']:.4f}  tta_z={spec.tta_z} tta_affine={spec.tta_affine}", flush=True)


In [ ]:
%%writefile scripts/repack_v3.py
"""Rebuild a submission zip with the repo's current code but the artifacts (manifest + checkpoints + template) of an
existing zip. Used to repair a Kaggle-built zip when only the inference code needs to change.
    python scripts/repack_v3.py <in.zip> <out.zip>"""
from __future__ import annotations

import sys, zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
src, dst = Path(sys.argv[1]), Path(sys.argv[2])
with zipfile.ZipFile(src) as zi, zipfile.ZipFile(dst, "w", zipfile.ZIP_DEFLATED) as zo:
    zo.write(ROOT / "main_v3.py", "main.py")
    zo.write(ROOT / "tests_synth.py", "tests_synth.py")
    for p in sorted((ROOT / "datsc").glob("*.py")):
        zo.write(p, f"datsc/{p.name}")
    n = 0
    for info in zi.infolist():
        if info.filename.startswith("artifacts/"):
            zo.writestr(info, zi.read(info.filename)); n += 1
    code = [i.filename for i in zi.infolist() if not i.filename.startswith("artifacts/")]
print(f"{dst}: fresh code ({len(code)} files replaced) + {n} artifact files from {src.name}; {dst.stat().st_size/1e6:.0f} MB")


In [ ]:
%%writefile scripts/run4_assemble.py
"""One-shot assembly after a Kaggle run: nested stack over every arm with OOF files, drop zero-weight arms, pack the zip
with every referenced checkpoint, run the container-style local check on the smoke set.

python scripts/run4_assemble.py --src data/kaggle_output_v4 --extra-dir artifacts/featarm_v4 --smoke <smoke_dir> --zip data/submission_v4.zip
  --src        the downloaded version output (scripts/watch_kaggle_output.py); needs <src>/artifacts_v3 (.pt) and/or <src>/out (json, oof)
  --extra-dir  extra artefact dirs copied in first (e.g. the locally fitted feature arm)
"""
from __future__ import annotations

import argparse
import glob
import json
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))


def sh(cmd: str) -> str:
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=ROOT)
    print(r.stdout[-6000:]); print(r.stderr[-1500:])
    assert r.returncode == 0, f"failed: {cmd}"
    return r.stdout


def stack_cmd(art: Path, arms, site, extras, grid: int) -> str:
    cmd = f'python scripts/fit_stack_v3.py --out "{art}" --lam 0.02 --t-floor 1.0 --nested --grid {grid} --cache cache_v4'
    cmd += "".join(f' --arm {n} "{p}"' for n, p in arms)
    cmd += "".join(f' --site-arm {n} "{q}"' for n, q in site)
    cmd += "".join(f' --extra {n} "{g}"' for n, g in extras)
    return cmd


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True)
    ap.add_argument("--extra-dir", action="append", default=[])
    ap.add_argument("--smoke", default=None)
    ap.add_argument("--zip", default="data/submission_v4.zip")
    ap.add_argument("--grid", type=int, default=8)
    ap.add_argument("--min-weight", type=float, default=0.05, help="arms below this weight are dropped and the stack re-fitted")
    a = ap.parse_args()
    src = Path(a.src); art = src / "assembled"; art.mkdir(exist_ok=True)
    # 1) gather artefacts: .pt/.npz/.json/.csv from artifacts_v3 and out (+ extra dirs) into one folder
    for d in [Path(x) for x in a.extra_dir] + [src / "artifacts_v3", src / "out"]:
        if d.exists():
            for f in d.iterdir():
                if f.is_file() and f.suffix in (".pt", ".npz", ".json", ".csv") and not (art / f.name).exists():
                    shutil.copy(f, art / f.name)
    tpl = ROOT / "artifacts" / "template.npz"
    if tpl.exists() and not (art / "template.npz").exists():
        shutil.copy(tpl, art / "template.npz")
    print("artefacts:", len(list(art.glob("*.pt"))), "pt |", len(list(art.glob("oof_*.csv"))), "oof |", sorted(p.name for p in art.glob("*.npz")))
    # 2) arms from the generator's ARMS/EXTRAS, filtered to what exists
    ns = {"__file__": str(ROOT / "kaggle" / "gen_run3_inject.py"), "__name__": "gen"}
    exec(compile((ROOT / "kaggle" / "gen_run3_inject.py").read_text(encoding="utf-8"), "gen", "exec"), ns)
    arms = [(n, p) for n, p, _ in ns["ARMS"] if glob.glob(str(art / f"oof_{p}.csv"))]
    site = [(n, q) for n, _, q in ns["ARMS"] if q and glob.glob(str(art / f"oof_{q}.csv"))]
    extras = [(n, g) for n, g in ns["EXTRAS"] if glob.glob(str(art / f"{g}.pt")) and n in dict(arms)]
    print("arms:", [n for n, _ in arms], "| site arms:", [n for n, _ in site], "| extras:", extras)
    out = sh(stack_cmd(art, arms, site, extras, a.grid))
    man = json.loads((art / "manifest.json").read_text())
    w = man["weights"]; keep = [n for n, _ in arms if w.get(n, 0) >= a.min_weight]
    print("weights:", w, "| nested:", [l for l in out.splitlines() if "nested combiner estimate" in l])
    if len(keep) < len(arms):
        print(f"re-fitting without zero-weight arms: dropped {[n for n, _ in arms if n not in keep]}")
        arms2 = [(n, p) for n, p in arms if n in keep]; site2 = [(n, q) for n, q in site if n in keep]; extras2 = [(n, g) for n, g in extras if n in keep]
        out2 = sh(stack_cmd(art, arms2, site2, extras2, a.grid))
        man = json.loads((art / "manifest.json").read_text())
    print("final weights:", man["weights"], "T", round(man["T"], 3), "OOF", round(man["oof_logloss"], 4), "clip", man.get("clip"), "hedge", man.get("hedge"))
    # 3) pack + verify
    sh(f'python scripts/pack_v3.py --artifacts "{art}" --out "{a.zip}"')
    if a.smoke:
        sh(f'python scripts/local_e2e_v3.py "{a.zip}" "{a.smoke}"')
    print("DONE:", a.zip)


In [ ]:
%%writefile scripts/shift_diagnostic.py
"""Leave-one-spacing-cluster-out on the LR arm (cheap proxy for site shift).
Rule (precommitted): worse cluster inflates log loss by <50% -> lam=0.02, else lam=0.05."""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd

from datsc.calibrate import logloss
from datsc.lrmodel import LRModel, feature_matrix

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--C", type=float, default=0.3)
    a = ap.parse_args()
    f = pd.read_csv(Path(a.cache) / "folds.csv", dtype={"uid": str})
    X = feature_matrix(Path(a.cache), f["uid"]); y = f["is_pathologic"].values.astype(int)
    base = logloss(y, pd.read_csv(Path(a.cache) / "oof_lr.csv", dtype={"uid": str})["p_lr"].values)
    worst = 0.0
    for lo, hi, name in ((2.43, 2.52, "2.46mm"), (3.85, 3.95, "3.90mm")):   # 2.43 excludes the separate 2.40mm cluster
        held = (f["native_spacing_mm"] >= lo) & (f["native_spacing_mm"] < hi)
        p = LRModel(C=a.C).fit(X[~held], y[~held]).predict_proba(X[held])
        ll = logloss(y[held], p); infl = ll / base - 1
        worst = max(worst, infl)
        print(f"hold out {name}: n={int(held.sum())} logloss={ll:.4f} (random-CV={base:.4f}, inflation={infl:+.0%})")
    lam = 0.02 if worst < 0.5 else 0.05
    print(f"RULE -> lam={lam}")


In [ ]:
%%writefile scripts/ssl_pretrain_c.py
"""Approach C: VICReg pretraining of the striatal-crop 3D encoder on the training scans (labels unused), then an optional
linear-probe report. Skips the pretraining when --out already exists (resume-friendly).

python scripts/ssl_pretrain_c.py --cache cache_v4 --out artifacts_v3/ssl_c_encoder.pt --epochs 200 [--probe] [--max-steps N]
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np
import pandas as pd

from datsc.ssl_c import extract_features, linear_probe_oof, pretrain

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--out", default="artifacts_v3/ssl_c_encoder.pt")
    ap.add_argument("--epochs", type=int, default=200)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--widths", type=int, nargs=4, default=[24, 48, 96, 192])
    ap.add_argument("--center3d", type=float, nargs=3, default=None, help="default: template striatal_center from artifacts/template.npz")
    ap.add_argument("--jitter", type=int, default=2)
    ap.add_argument("--aug", default="base", choices=["base", "strong"])
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--max-steps", type=int, default=None, help="smoke tests: stop after N optimisation steps")
    ap.add_argument("--probe", action="store_true", help="after pretraining: linear-probe OOF on frozen features (random + site folds)")
    ap.add_argument("--force", action="store_true")
    a = ap.parse_args()
    out = Path(a.out)
    center = a.center3d
    if center is None:
        tpl = next((p for p in (Path("artifacts/template.npz"), Path(a.cache) / "template.npz", Path("artifacts_v3/template.npz")) if p.exists()), None)
        center = [float(v) for v in np.load(tpl)["striatal_center"]] if tpl else [47.5, 57.3, 33.5]
    if out.exists() and not a.force:
        print(f"[ssl_c] {out} exists -- skipping pretraining (use --force to redo)")
    else:
        pretrain(a.cache, out, epochs=a.epochs, batch_size=a.batch_size, lr=a.lr, widths=tuple(a.widths), center=tuple(center),
                 jitter=a.jitter, aug=a.aug, num_workers=a.workers, max_steps=a.max_steps)
    if a.probe:
        from sklearn.metrics import log_loss, roc_auc_score
        cache = Path(a.cache); meta = pd.read_csv(cache / "meta.csv", dtype={"uid": str})
        y = meta["is_pathologic"].values.astype(float)
        feats = extract_features(out, cache, meta["uid"], num_workers=0)
        rep = {}
        for name, fc in (("random", "folds.csv"), ("site", "folds_site.csv")):
            if not (cache / fc).exists():
                continue
            folds = pd.read_csv(cache / fc, dtype={"uid": str}).set_index("uid").loc[meta.uid, "fold"].values
            p = linear_probe_oof(feats, y, folds)
            rep[name] = {"auroc": float(roc_auc_score(y, p)), "logloss": float(log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)))}
            print(f"[ssl_c] linear probe ({name} folds): AUROC={rep[name]['auroc']:.4f} logloss={rep[name]['logloss']:.4f}", flush=True)
        out.with_suffix(".probe.json").write_text(json.dumps(rep))


In [ ]:
%%writefile scripts/sweep_calibration.py
"""Sweep the combiner's temperature floor and prior mix over the nested pool, next to the tail risk each lam buys.

python scripts/sweep_calibration.py [--src data/run8/artifacts_v3 --src data/master_pool/artifacts_v3]
                                    [--arms a3d_hemi@all,a3d_ssl@all,...] [--T 0.8,0.9,1.0] [--lam 0,0.005,0.01,0.02]

Why this script exists: the nested pool is built from random folds and therefore contains no site shift, so its
log loss is monotone decreasing in lam and it will always recommend lam -> 0. That recommendation is only safe if
the hidden test holds no scans the ensemble is confidently wrong about. lam is the insurance against exactly those:
a maximally confident wrong scan costs -log(lam*pi) nats, against -log(EPS) = 9.21 at lam = 0 (datsc/calibrate.py:11,48).
The WORST column prices that insurance, and BREAK-EVEN says how many such scans make the cheaper lam the wrong choice.
Read both columns before moving lam. The smoke set is never fitted on (ADR 0005).
"""
import argparse, sys
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT / "scripts"))

EPS = 1e-4  # datsc/calibrate.py:11 — the floor the shipped combiner clips to


def worst_case_nats(lam, pi, q_lo=0.0, q_hi=1.0):
    """Loss of a single confidently wrong scan at this lam.

    Defaults give the theoretical bound (the model predicting 0 or 1 exactly). Pass the pool's observed
    extremes instead for the realistic figure: this ensemble never reaches 0 or 1, so the theoretical bound
    overstates what lam=0 actually risks (4.0 nats of extra exposure against a real ~3.0).
    """
    p_when_truth_is_1 = max((1 - lam) * q_lo + lam * pi, EPS)
    p_when_truth_is_0 = min((1 - lam) * q_hi + lam * pi, 1 - EPS)
    return -np.log(p_when_truth_is_1), -np.log(1 - p_when_truth_is_0)


def main():
    import build_master_v3 as bm  # deferred: pulls in scipy/sklearn via fit_stack_v3, so importing this module stays cheap

    ap = argparse.ArgumentParser()
    ap.add_argument("--src", action="append", default=None)
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--arms", default=None, help="comma-separated pool keys; default = pillars@all + every run-8 arm present")
    ap.add_argument("--T", default="0.80,0.85,0.90,0.95,1.00")
    ap.add_argument("--lam", default="0,0.0025,0.005,0.0075,0.01,0.015,0.02,0.03")
    ap.add_argument("--mode", default="equal", choices=["equal", "shrunk", "fitted"])
    ap.add_argument("--n-test", type=int, default=2000, help="assumed hidden test size, for the break-even count")
    a = ap.parse_args()

    srcs = a.src or ["data/run8/artifacts_v3", "data/master_pool/artifacts_v3"]
    Ts = [float(x) for x in a.T.split(",")]
    lams = [float(x) for x in a.lam.split(",")]

    pool = bm.Pool(srcs, a.cache, Ts[0], lams[0])
    if a.arms:
        names = [n.strip() for n in a.arms.split(",")]
        missing = [n for n in names if n not in pool.oofs]
        if missing:
            raise SystemExit(f"not in pool: {missing}\navailable: {sorted(pool.oofs)}")
    else:
        names = [n + "@all" for n in bm.PILLARS if n + "@all" in pool.oofs] + [n for n in bm.NEW if n in pool.oofs]
    print(f"pool: {len(pool.uids)} scans, prevalence {pool.y.mean():.4f}; arms ({len(names)}): {', '.join(names)}")

    grid = {}
    print(f"\nNESTED LOG LOSS ({a.mode} weights)")
    print("   T    " + "".join(f"lam={l:<9.4f}" for l in lams))
    for T in Ts:
        pool.T_floor = T
        row = f"{T:>5.2f}  "
        for lam in lams:
            pool.lam = lam
            grid[(T, lam)] = pool.nested(names, a.mode)
            row += f"{grid[(T, lam)]:<13.5f}"
        print(row)

    (bT, bl), bll = min(grid.items(), key=lambda kv: kv[1])
    print(f"\nnested optimum: T={bT:.2f} lam={bl:.4f} -> {bll:.5f}   (NO SHIFT IN THIS POOL — see the header)")

    # Price the tail against what this ensemble actually predicts, not against 0 and 1 which it never reaches.
    Zs = pool.Z(names); weq = np.ones(len(names)) / len(names)
    q_obs = 1.0 / (1.0 + np.exp(-(weq[:, None] * Zs).sum(0) / bT))
    q_lo, q_hi = float(q_obs.min()), float(q_obs.max())

    print(f"\nTAIL RISK AT THE NESTED-OPTIMAL T={bT:.2f}")
    print(f"observed combined-probability range: [{q_lo:.6f}, {q_hi:.6f}] "
          f"(theoretical bound would use [0, 1] and overstate lam=0's exposure by ~1 nat)")
    print(f"{'lam':>8} {'nested':>10} {'vs lam=.01':>11} {'WORST y=1':>10} {'WORST y=0':>10} {'BREAK-EVEN vs lam=.01':>23}")
    pool.T_floor = bT
    ref = grid[(bT, 0.01)] if (bT, 0.01) in grid else None
    w1_ref, _ = worst_case_nats(0.01, pool.pi, q_lo, q_hi)
    for lam in lams:
        ll = grid[(bT, lam)]
        w1, w0 = worst_case_nats(lam, pool.pi, q_lo, q_hi)
        cell = ""
        if ref is not None and lam != 0.01:
            gain, extra = ref - ll, w1 - w1_ref
            cell = "safer" if gain < 0 or extra <= 0 else f"{gain * a.n_test / extra:.0f} scans of {a.n_test}"
        print(f"{lam:>8.4f} {ll:>10.5f} {(ref - ll) if ref is not None else float('nan'):>+11.5f} {w1:>10.2f} {w0:>10.2f} {cell:>23}")
    print(f"\nBREAK-EVEN = how many confidently-wrong scans in {a.n_test} erase that lam's nested gain over lam=0.01.")
    print("If you would not bet against that many, keep lam=0.01.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/template_landmarks.py
"""Add histogram-standardisation landmarks (reference-region percentiles of the study template) to template.npz.
python scripts/template_landmarks.py [--template artifacts/template.npz] [--also artifacts_v3/template.npz data/ckpt_upload_run3/template.npz]
"""
from __future__ import annotations

import argparse
import shutil
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np

from datsc.register import HIST_PCTS, hist_landmarks

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--template", default="artifacts/template.npz")
    ap.add_argument("--also", nargs="*", default=["artifacts_v3/template.npz"])
    a = ap.parse_args()
    z = dict(np.load(a.template))
    lm = hist_landmarks(z["box"], z["ref_mask"].astype(bool))     # reference region (brain minus striatum), never the striatum itself
    z["hist"] = lm
    np.savez(a.template, **z)
    print("landmarks", dict(zip(HIST_PCTS, np.round(lm, 4))))
    for dst in a.also:
        Path(dst).parent.mkdir(parents=True, exist_ok=True); shutil.copy(a.template, dst); print("copied ->", dst)


In [ ]:
%%writefile scripts/train_a.py
"""Run-6a training CLI: the striatal-crop 3D family (approach A). Same trainer as scripts/train_v3.py (datsc.train_v3.train_run),
plus the SE / stochastic-depth architecture switch and a tag scheme that keeps the variants apart in the checkpoint dataset.

python scripts/train_a.py --fold 0 --seed 0 --crop3d 48 48 32 --center3d 47.5 57.3 33.5 --epochs 80                 # base arch, 80 epochs
python scripts/train_a.py --fold 0 --seed 0 --crop3d 48 48 32 --center3d 47.5 57.3 33.5 --arch se --widths 32 64 128 256
python scripts/train_a.py --fold 0 --seed 0 --crop3d 64 64 40 --center3d 47.5 57.3 33.5 --size 64 64 40 --epochs 60
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from datsc.train_v3 import InputSpec, train_run


def default_tag(a) -> str:
    crop = "x".join(map(str, a.crop3d)) if a.crop3d else "x".join(map(str, a.size))
    if a.arch == "se":
        base = f"r3d_se{a.crop3d[0] if a.crop3d else a.size[0]}" + (f"b{a.blocks}" if a.blocks > 1 else "")
    elif a.epochs != 50:
        base = f"r3d_crop{a.crop3d[0]}e{a.epochs}" if a.crop3d else f"r3d_{crop}e{a.epochs}"
    else:
        base = f"r3d_crop{crop}" if a.crop3d else f"r3d_{crop}"
    if a.dann > 0 or a.dro > 0:      # the standard config keeps the run-5 tag family so a finished copy is reused by the runner
        base = "r3d_cropinv" if (crop == "48x48x32" and a.epochs == 50 and a.arch == "base") else base + "inv"
    tag = f"{base}_f{a.fold if a.fold >= 0 else 'all'}_s{a.seed}"
    if a.folds_csv != "folds.csv":
        tag += "_" + Path(a.folds_csv).stem.replace("folds_", "")
    return tag


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--fold", type=int, required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--out", default="artifacts_v3")
    ap.add_argument("--folds-csv", default="folds.csv")
    ap.add_argument("--tag", default=None)
    ap.add_argument("--epochs", type=int, default=50)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--weight-decay", type=float, default=1e-2)
    ap.add_argument("--warmup", type=int, default=3)
    ap.add_argument("--mixup", type=float, default=0.2)
    ap.add_argument("--ema", type=float, default=0.995)
    ap.add_argument("--norm", default="ref", choices=["ref", "p75", "median", "zscore", "p99", "hist"])
    ap.add_argument("--aug", default="base", choices=["base", "strong"])
    ap.add_argument("--dann", type=float, default=0.0)
    ap.add_argument("--dro", type=float, default=0.0)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--size", type=int, nargs=3, default=[48, 48, 32], help="model input size (== crop3d for no resize)")
    ap.add_argument("--crop3d", type=int, nargs=3, default=[48, 48, 32])
    ap.add_argument("--center3d", type=float, nargs=3, default=None, help="template striatal_center x y z (voxels)")
    ap.add_argument("--arch", default="base", choices=["base", "se"], help="base = run-4a ResNet3D; se = SE + stochastic-depth family (datsc.models_a)")
    ap.add_argument("--widths", type=int, nargs=4, default=None, help="default: base 24 48 96 192, se 32 64 128 256")
    ap.add_argument("--blocks", type=int, default=1, help="se: residual blocks per stage")
    ap.add_argument("--drop-path", type=float, default=0.1, help="se: max stochastic-depth rate (linear over blocks)")
    ap.add_argument("--no-se", action="store_true", help="se arch without the channel gates")
    ap.add_argument("--tta-affine", type=int, default=0)
    a = ap.parse_args()
    widths = tuple(a.widths) if a.widths else ((32, 64, 128, 256) if a.arch == "se" else (24, 48, 96, 192))
    spec = InputSpec("3d", size=a.size, crop3d=a.crop3d, center3d=a.center3d, tta_affine=a.tta_affine)
    if a.arch == "se":
        model_kw = dict(arch="se", widths=widths, blocks=a.blocks, se=not a.no_se, drop_path=a.drop_path, n_scalars=0)
    else:
        model_kw = dict(widths=widths, stem_stride=1, n_scalars=0)
    tag = a.tag or default_tag(a)
    train_run(a.cache, a.fold, a.seed, a.out, spec, model_kw, tag, epochs=a.epochs, batch_size=a.batch_size, lr=a.lr,
              weight_decay=a.weight_decay, warmup_epochs=a.warmup, mixup=a.mixup, ema_decay=a.ema or None, norm=a.norm,
              folds_csv=a.folds_csv, num_workers=a.workers, aug=a.aug, dann=a.dann, dro=a.dro)


In [ ]:
%%writefile scripts/train_b.py
"""Run-6b training CLI: tri-planar late-fusion member (shared ImageNet backbone over axial / coronal / sagittal-L / sagittal-R
slab stacks of the registered box). Same protocol as run 3 (fixed epochs, EMA-final, mixup, base augmentation), plus
slab-centre jitter and optional site-invariant training / hist normalisation.

python scripts/train_b.py --fold 0 --seed 0 --backbone convnext_tiny --weights pretrained/convnext_tiny_in1k.pt [--norm hist]
Tags: r2t_<backbone>[_hist]_f<f>_s<s>[_site]   (r2t = registered, 2.5D, tri-planar)
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
import numpy as np

from datsc.data_b import hemisphere_centres
from datsc.train_b import DEFAULT_VIEWS, InputSpecB
from datsc.train_v3 import train_run


def template_centres(path: str):
    """(z_center, y_center, x_left, x_right) from the study template; None -> InputSpecB defaults."""
    p = Path(path)
    if not p.exists():
        return None
    z = np.load(p); c = z["striatal_center"]
    xl, xr = hemisphere_centres(z["striatal_mask"].astype(bool), float(c[0]))
    return float(c[2]), float(c[1]), xl, xr


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--fold", type=int, required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--out", default="artifacts_v3")
    ap.add_argument("--folds-csv", default="folds.csv")
    ap.add_argument("--tag", default=None)
    ap.add_argument("--template", default="artifacts/template.npz")
    ap.add_argument("--epochs", type=int, default=24)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--backbone-lr", type=float, default=1e-4)
    ap.add_argument("--weight-decay", type=float, default=1e-2)
    ap.add_argument("--warmup", type=int, default=3)
    ap.add_argument("--mixup", type=float, default=0.2)
    ap.add_argument("--ema", type=float, default=0.99)
    ap.add_argument("--norm", default="ref", choices=["ref", "hist", "p75"])
    ap.add_argument("--aug", default="base", choices=["base", "strong"])
    ap.add_argument("--z-jitter", type=int, default=1)
    ap.add_argument("--dann", type=float, default=0.0)
    ap.add_argument("--dro", type=float, default=0.0)
    ap.add_argument("--no-aug", action="store_true")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--backbone", default="convnext_tiny")
    ap.add_argument("--weights", default=None)
    ap.add_argument("--n-slabs", type=int, default=3)
    ap.add_argument("--slab-mm", type=float, default=8.0)
    ap.add_argument("--hw", type=int, default=160)
    ap.add_argument("--views", nargs="+", default=list(DEFAULT_VIEWS))
    ap.add_argument("--fusion", default="attn", choices=["attn", "concat"])
    ap.add_argument("--z-center", type=float, default=None)
    ap.add_argument("--y-center", type=float, default=None)
    ap.add_argument("--x-left", type=float, default=None)
    ap.add_argument("--x-right", type=float, default=None)
    ap.add_argument("--tta-z", type=int, nargs="+", default=[-1, 0, 1])
    ap.add_argument("--tta-affine", type=int, default=0)
    ap.add_argument("--crop", type=int, default=None)
    ap.add_argument("--dropout", type=float, default=0.3)
    a = ap.parse_args()
    tc = template_centres(a.template)
    zc, yc, xl, xr = (a.z_center, a.y_center, a.x_left, a.x_right)
    if tc is not None:
        zc = zc if zc is not None else tc[0]; yc = yc if yc is not None else tc[1]
        xl = xl if xl is not None else tc[2]; xr = xr if xr is not None else tc[3]
    spec = InputSpecB(n_slabs=a.n_slabs, slab_mm=a.slab_mm, hw=a.hw, views=a.views, tta_z=a.tta_z, crop=a.crop, tta_affine=a.tta_affine,
                      **{k: v for k, v in dict(z_center=zc, y_center=yc, x_left=xl, x_right=xr).items() if v is not None})
    model_kw = dict(backbone=a.backbone, weights_path=a.weights, n_in=a.n_slabs, n_views=len(a.views), fusion=a.fusion,
                    dropout=a.dropout, n_scalars=0)
    tag = a.tag or f"r2t_{a.backbone}{'_hist' if a.norm == 'hist' else ''}_f{a.fold if a.fold >= 0 else 'all'}_s{a.seed}"
    if a.folds_csv != "folds.csv" and not a.tag:
        tag += "_" + Path(a.folds_csv).stem.replace("folds_", "")
    train_run(a.cache, a.fold, a.seed, a.out, spec, model_kw, tag, epochs=a.epochs, batch_size=a.batch_size, lr=a.lr,
              backbone_lr=a.backbone_lr, weight_decay=a.weight_decay, warmup_epochs=a.warmup, mixup=a.mixup,
              ema_decay=a.ema or None, norm=a.norm, folds_csv=a.folds_csv, num_workers=a.workers, augment=not a.no_aug,
              aug=a.aug, z_jitter=a.z_jitter, dann=a.dann, dro=a.dro)


In [ ]:
%%writefile scripts/train_cnn.py
"""python scripts/train_cnn.py --fold 0 --seed 0 [--control] [--epochs 40] [--cache cache] [--out artifacts]"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from datsc.train import train_fold

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--fold", type=int, required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--out", default="artifacts")
    ap.add_argument("--control", action="store_true", help="train on plain CoM crops (baseline)")
    ap.add_argument("--workers", type=int, default=2)
    a = ap.parse_args()
    train_fold(a.cache, a.fold, a.seed, a.out, epochs=a.epochs, batch_size=a.batch_size,
               num_workers=a.workers, control=a.control)


In [ ]:
%%writefile scripts/train_v3.py
"""Run-3 training CLI.

python scripts/train_v3.py --arm 3d --fold 0 --seed 0 [--size 96 96 72] [--widths 24 48 96 192]
python scripts/train_v3.py --arm 2d --backbone efficientnet_b0 --weights pretrained/efficientnet_b0_in1k.pt --fold 0 --seed 0 --z-center 40.5
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from datsc.train_v3 import InputSpec, train_run

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--arm", choices=["3d", "2d"], required=True)
    ap.add_argument("--fold", type=int, required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--cache", default="cache_v4")
    ap.add_argument("--out", default="artifacts_v3")
    ap.add_argument("--folds-csv", default="folds.csv")
    ap.add_argument("--tag", default=None)
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--backbone-lr", type=float, default=None)
    ap.add_argument("--weight-decay", type=float, default=1e-2)
    ap.add_argument("--warmup", type=int, default=3)
    ap.add_argument("--mixup", type=float, default=0.0)
    ap.add_argument("--ema", type=float, default=0.995)
    ap.add_argument("--select", default="final", choices=["final", "best"])
    ap.add_argument("--norm", default="ref", choices=["ref", "p75", "median", "zscore", "p99", "hist"])
    ap.add_argument("--n-scalars", type=int, default=0)
    ap.add_argument("--no-aug", action="store_true")
    ap.add_argument("--aug", default="base", choices=["base", "strong"], help="augmentation preset (datsc.augment.PRESETS)")
    ap.add_argument("--z-jitter", type=int, default=0, help="2D: random slab-centre shift in voxels during training")
    ap.add_argument("--dann", type=float, default=0.0, help="site-adversarial (gradient-reversal) head weight; 0 = off")
    ap.add_argument("--dro", type=float, default=0.0, help="group-DRO step size over pseudo-sites; 0 = off")
    ap.add_argument("--init", default=None, help="warm start: state dict (or {'encoder': ...}) whose stem/layers/backbone tensors are loaded")
    ap.add_argument("--workers", type=int, default=2)
    # 3d
    ap.add_argument("--size", type=int, nargs=3, default=[96, 96, 72])
    ap.add_argument("--widths", type=int, nargs=4, default=[24, 48, 96, 192])
    ap.add_argument("--stem-stride", type=int, default=1)
    ap.add_argument("--arch", default=None, choices=[None, "base", "se", "hemi"], help="3D architecture: base ResNet3D, se (run 6a), hemi (run 7 hemisphere member)")
    ap.add_argument("--loss", default="bce", choices=["bce", "bootstrap"], help="bootstrap = soft bootstrapped targets (label-noise robust)")
    ap.add_argument("--soft-csv", default=None, help="run 8: table uid,p[,loss] of the ensemble's nested OOF probability per training scan")
    ap.add_argument("--soft-lam", type=float, default=0.0, help="run 8: soft-target mix, y* = (1-lam) y + lam p (0 = hard labels)")
    ap.add_argument("--mistake-w", type=float, default=0.0, help="run 8: per-sample weight 1 + k * nested loss, normalised to mean 1 (0 = off)")
    ap.add_argument("--crop3d", type=int, nargs=3, default=None, help="3D: crop (voxels) around --center3d before the resize")
    ap.add_argument("--center3d", type=float, nargs=3, default=None, help="3D: crop centre (voxels; template striatal_center x y z)")
    # 2d
    ap.add_argument("--backbone", default="efficientnet_b0")
    ap.add_argument("--weights", default=None)
    ap.add_argument("--n-slabs", type=int, default=3)
    ap.add_argument("--slab-mm", type=float, default=8.0)
    ap.add_argument("--slab-mode", default="mean", choices=["mean", "max"])
    ap.add_argument("--hw", type=int, default=192)
    ap.add_argument("--z-center", type=float, default=None, help="voxel centre of the slab stack along the slab axis (template striatal_center[2] axial / [1] coronal)")
    ap.add_argument("--plane", default="axial", choices=["axial", "coronal"])
    ap.add_argument("--tta-z", type=int, nargs="+", default=[0], help="slab-centre offsets (voxels) averaged at test time, e.g. -1 0 1")
    ap.add_argument("--crop", type=int, default=None, help="central crop (voxels) of the slab image before resizing, e.g. 72")
    ap.add_argument("--tta-affine", type=int, default=0, help="test-time: K reproducible small affine perturbations of the box (all arms)")
    a = ap.parse_args()
    if a.arm == "3d":
        spec = InputSpec("3d", size=a.size, crop3d=a.crop3d, center3d=a.center3d, tta_affine=a.tta_affine)
        model_kw = dict(widths=tuple(a.widths), stem_stride=a.stem_stride, n_scalars=a.n_scalars)
        if a.arch and a.arch != "base":
            model_kw["arch"] = a.arch
        base = f"r3d_crop{'x'.join(map(str, a.crop3d))}" if a.crop3d else f"r3d_{'x'.join(map(str, a.size))}"
        if a.arch == "hemi":
            base = f"r3d_hemi{a.crop3d[0] if a.crop3d else a.size[0]}"
        tag = a.tag or f"{base}_f{a.fold if a.fold >= 0 else 'all'}_s{a.seed}"
    else:
        spec = InputSpec("2d", n_slabs=a.n_slabs, slab_mm=a.slab_mm, hw=a.hw, slab_mode=a.slab_mode, z_center=a.z_center,
                         plane=a.plane, tta_z=a.tta_z, crop=a.crop, tta_affine=a.tta_affine)
        model_kw = dict(backbone=a.backbone, weights_path=a.weights, n_in=a.n_slabs, n_scalars=a.n_scalars)
        tag = a.tag or f"r2d_{a.backbone}{'_cor' if a.plane == 'coronal' else ''}_f{a.fold if a.fold >= 0 else 'all'}_s{a.seed}"
    if a.folds_csv != "folds.csv" and not a.tag:            # explicit --tag is used verbatim (run 4a: '_site_site' double suffix)
        tag += "_" + Path(a.folds_csv).stem.replace("folds_", "")
    train_run(a.cache, a.fold, a.seed, a.out, spec, model_kw, tag, epochs=a.epochs, batch_size=a.batch_size, lr=a.lr,
              backbone_lr=a.backbone_lr, weight_decay=a.weight_decay, warmup_epochs=a.warmup, mixup=a.mixup,
              ema_decay=a.ema or None, norm=a.norm, folds_csv=a.folds_csv, num_workers=a.workers, augment=not a.no_aug,
              select=a.select, aug=a.aug, z_jitter=a.z_jitter, dann=a.dann, dro=a.dro, init_path=a.init, loss=a.loss,
              soft_csv=a.soft_csv, soft_lam=a.soft_lam, mistake_w=a.mistake_w)


In [ ]:
%%writefile scripts/update_paper_tex_files.py
# -*- coding: utf-8 -*-
"""Update LaTeX files with clean modular figures and correct bibliography tags."""
import re
from pathlib import Path

# -------------------------------------------------------------------
# 1. Update isbi2027.tex
# -------------------------------------------------------------------
isbi_path = Path("paper_v2/isbi2027.tex")
isbi_text = isbi_path.read_text(encoding="utf-8")

# Replace inline tikz with input
isbi_text = re.sub(
    r"\\begin\{tikzpicture\}\[y=13cm,x=0\.9cm\].*?\\end\{tikzpicture\}",
    r"\\input{figures/fig_shiftgap_isbi.tex}",
    isbi_text,
    flags=re.DOTALL
)
isbi_path.write_text(isbi_text, encoding="utf-8")
print("Updated isbi2027.tex")

# -------------------------------------------------------------------
# 2. Update ejnmmi_submission.tex
# -------------------------------------------------------------------
ejnmmi_path = Path("paper_v2/ejnmmi_submission.tex")
ejnmmi_text = ejnmmi_path.read_text(encoding="utf-8")

ejnmmi_text = re.sub(
    r"\\begin\{tikzpicture\}\[y=20cm\].*?\\end\{tikzpicture\}",
    r"\\input{figures/fig_shiftgap.tex}",
    ejnmmi_text,
    flags=re.DOTALL
)
ejnmmi_path.write_text(ejnmmi_text, encoding="utf-8")
print("Updated ejnmmi_submission.tex")

# -------------------------------------------------------------------
# 3. Update main.tex
# -------------------------------------------------------------------
main_path = Path("paper_v2/main.tex")
main_text = main_path.read_text(encoding="utf-8")

# Replace Fig 1
main_text = re.sub(
    r"\\begin\{tikzpicture\}\[scale=1\].*?\\end\{tikzpicture\}",
    r"\\input{figures/fig_schematic.tex}",
    main_text,
    flags=re.DOTALL
)

# Replace Fig 2 (resizebox wrapping tikz)
main_text = re.sub(
    r"\\resizebox\{\\textwidth\}\{!\}%?\s*\\begin\{tikzpicture\}\[\s*node distance=8mm and 10mm.*?\\end\{tikzpicture\}%?\s*\}",
    r"\\resizebox{\\textwidth}{!}{\\input{figures/fig_preprocessing.tex}}",
    main_text,
    flags=re.DOTALL
)

# Replace Fig 3 (Hemi arm - was 0.86\textheight!)
main_text = re.sub(
    r"\\resizebox\{!\}\{0\.86\\textheight\}%?\s*\\begin\{tikzpicture\}\[\s*font=\\scriptsize.*?\\end\{tikzpicture\}%?\s*\}",
    r"\\centering\\input{figures/fig_hemi.tex}",
    main_text,
    flags=re.DOTALL
)

# Replace Fig 4 (Ensemble)
main_text = re.sub(
    r"\\resizebox\{\\textwidth\}\{!\}%?\s*\\begin\{tikzpicture\}\[\s*node distance=7mm and 18mm.*?\\end\{tikzpicture\}%?\s*\}",
    r"\\centering\\input{figures/fig_ensemble.tex}",
    main_text,
    flags=re.DOTALL
)

# Replace Fig 5 (Shift gap)
main_text = re.sub(
    r"\\begin\{tikzpicture\}\[y=20cm\].*?\\end\{tikzpicture\}",
    r"\\input{figures/fig_shiftgap.tex}",
    main_text,
    flags=re.DOTALL
)

# Fix natbib bibitems in main.tex
main_text = main_text.replace(
    r"\bibitem{ensemblefusion2022}",
    r"\bibitem[{Diagnostics}(2022)]{ensemblefusion2022}"
)
main_text = main_text.replace(
    r"\bibitem{radiomics2022}",
    r"\bibitem[{EJNMMI}(2022)]{radiomics2022}"
)
main_text = main_text.replace(
    r"\bibitem{trainingprotocol2026}",
    r"\bibitem[{Electronics}(2026)]{trainingprotocol2026}"
)

main_path.write_text(main_text, encoding="utf-8")
print("Updated main.tex")

# -------------------------------------------------------------------
# 4. Update tmlr_submission.tex
# -------------------------------------------------------------------
tmlr_path = Path("paper_v2/tmlr_submission.tex")
tmlr_text = tmlr_path.read_text(encoding="utf-8")

# Replace Fig 1
tmlr_text = re.sub(
    r"\\begin\{tikzpicture\}\[scale=1\].*?\\end\{tikzpicture\}",
    r"\\input{figures/fig_schematic.tex}",
    tmlr_text,
    flags=re.DOTALL
)

# Replace Fig 2
tmlr_text = re.sub(
    r"\\resizebox\{\\textwidth\}\{!\}%?\s*\\begin\{tikzpicture\}\[\s*node distance=8mm and 10mm.*?\\end\{tikzpicture\}%?\s*\}",
    r"\\resizebox{\\textwidth}{!}{\\input{figures/fig_preprocessing.tex}}",
    tmlr_text,
    flags=re.DOTALL
)

# Replace Fig 3
tmlr_text = re.sub(
    r"\\resizebox\{!\}\{0\.86\\textheight\}%?\s*\\begin\{tikzpicture\}\[\s*font=\\scriptsize.*?\\end\{tikzpicture\}%?\s*\}",
    r"\\centering\\input{figures/fig_hemi.tex}",
    tmlr_text,
    flags=re.DOTALL
)

# Replace Fig 4
tmlr_text = re.sub(
    r"\\resizebox\{\\textwidth\}\{!\}%?\s*\\begin\{tikzpicture\}\[\s*node distance=7mm and 18mm.*?\\end\{tikzpicture\}%?\s*\}",
    r"\\centering\\input{figures/fig_ensemble.tex}",
    tmlr_text,
    flags=re.DOTALL
)

# Replace Fig 5
tmlr_text = re.sub(
    r"\\begin\{tikzpicture\}\[y=20cm\].*?\\end\{tikzpicture\}",
    r"\\input{figures/fig_shiftgap.tex}",
    tmlr_text,
    flags=re.DOTALL
)

tmlr_path.write_text(tmlr_text, encoding="utf-8")
print("Updated tmlr_submission.tex")


In [ ]:
%%writefile scripts/watch_kaggle_output.py
"""Poll the Kaggle commit and pull its output to this machine the moment it finishes.

Idempotent; meant to be fired every 5 min by Windows Task Scheduler (see kaggle/register_watcher.ps1), so it keeps
working when no Claude session is open. Writes data/kaggle_output/watch.log and, on success, DONE.json.
    python scripts/watch_kaggle_output.py [--kernel harshdipsaha/dat-parkinsons-train] [--out data/kaggle_output] [--once]
"""
from __future__ import annotations

import argparse, glob, hashlib, json, os, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
TASK = "DaT-KaggleOutputWatch"


def kaggle_exe() -> str:
    for c in (shutil.which("kaggle"), str(Path(sys.executable).parent / "Scripts" / "kaggle.exe"),
              str(Path(sys.executable).parent / "kaggle.exe")):
        if c and Path(c).exists():
            return c
    return "kaggle"


def log(out: Path, msg: str):
    line = f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} UTC  {msg}"
    with open(out / "watch.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(line)


def run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)


def status(kernel: str) -> str:
    r = run([kaggle_exe(), "kernels", "status", kernel])
    txt = (r.stdout + r.stderr).strip()
    for s in ("COMPLETE", "ERROR", "CANCEL", "RUNNING", "QUEUED"):
        if s in txt.upper():
            return s
    return "UNKNOWN: " + txt[-200:]


def verify(out: Path):
    zips = [p for p in out.rglob("submission_v3.zip") if p.is_file()]
    parts = sorted(p for p in out.rglob("submission_v3.zip.part*") if p.is_file())
    info = {"zip": None, "parts": len(parts), "md5": None, "testzip": None, "n_files": None, "size": None}
    if not zips and parts:
        tgt = out / "submission_v3.zip"
        with open(tgt, "wb") as w:
            for p in parts:
                w.write(p.read_bytes())
        zips = [tgt]
        info["joined"] = True
    if zips:
        z = max(zips, key=lambda p: p.stat().st_size)
        info["zip"] = str(z); info["size"] = z.stat().st_size
        info["md5"] = hashlib.md5(z.read_bytes()).hexdigest()
        try:
            zf = zipfile.ZipFile(z); info["testzip"] = zf.testzip(); info["n_files"] = len(zf.namelist())
            info["has_main"] = "main.py" in zf.namelist(); info["has_manifest"] = any(n.endswith("manifest.json") for n in zf.namelist())
        except zipfile.BadZipFile as e:
            info["testzip"] = f"BAD ZIP: {e}"
    md5s = list(out.rglob("*.md5")) + list(out.rglob("md5.txt"))
    if md5s:
        info["md5_expected_file"] = str(md5s[0]); info["md5_expected"] = md5s[0].read_text().split()[0] if md5s[0].read_text().split() else None
    return info


def fetch(kernel: str, out: Path) -> bool:
    """Download every output file of the latest version (the API pages 200 at a time; loop until no token)."""
    for attempt in range(3):
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi(); api.authenticate()
            token, n = None, 0
            while True:
                files, token = api.kernels_output(kernel, str(out), force=True, quiet=True, page_token=token, page_size=200)
                n += len(files)
                log(out, f"page fetched: {len(files)} files (total {n}); next token: {bool(token)}")
                if not token:
                    break
            return True
        except Exception as e:                     # noqa: BLE001
            log(out, f"api fetch attempt {attempt + 1} failed: {type(e).__name__}: {str(e)[-500:]}")
            r = run([kaggle_exe(), "kernels", "output", kernel, "-p", str(out), "--force", "--page-size", "200"])
            log(out, f"cli fallback rc={r.returncode} {(r.stdout + r.stderr).strip()[-400:]}")
            if r.returncode == 0:
                return True
            time.sleep(60)
    return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--kernel", default="harshdipsaha/dat-parkinsons-train")
    ap.add_argument("--out", default=str(ROOT / "data" / "kaggle_output"))
    ap.add_argument("--once", action="store_true", help="just report status and exit (no download)")
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    done, lock = out / "DONE.json", out / ".lock"
    if done.exists():
        log(out, "already fetched (DONE.json present); nothing to do")
        return 0
    if lock.exists() and time.time() - lock.stat().st_mtime < 3 * 3600:
        log(out, "another fetch in progress (lock < 3 h old); skipping")
        return 0
    st = status(a.kernel)
    log(out, f"status={st}")
    if a.once or st in ("RUNNING", "QUEUED") or st.startswith("UNKNOWN"):
        return 0
    lock.write_text(str(os.getpid()))
    try:
        ok = fetch(a.kernel, out)
        files = sorted(str(p.relative_to(out)) for p in out.rglob("*") if p.is_file() and p.name not in ("watch.log", ".lock"))
        log(out, f"downloaded {len(files)} files: {files[:40]}")
        info = verify(out) if ok else {}
        log(out, f"verify: {json.dumps(info)}")
        good = ok and info.get("zip") and info.get("testzip") is None and info.get("has_main")
        rec = {"kernel": a.kernel, "status": st, "fetched_utc": f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S}", "ok": bool(good), **info, "files": files}
        if ok:
            done.write_text(json.dumps(rec, indent=2))
            log(out, "DONE " + ("zip verified" if good else "downloaded but zip missing/invalid -- check watch.log"))
            if os.name == "nt":
                run(["schtasks", "/Delete", "/TN", TASK, "/F"])
        return 0 if good else 1
    finally:
        lock.unlink(missing_ok=True)


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile main_v3.py
"""DrivenData submission entrypoint (run 3). Prior CSV first, then a manifest-driven multi-arm ensemble
written stage-wise under a wall-clock ladder so any failure still leaves a valid submission.csv."""
import os
for _k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_k, "4")
import sys
import time
from pathlib import Path

T0 = time.time()
ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from datsc import config, infer_v3 as infer

ARTIFACTS = Path(os.environ.get("DATSC_ARTIFACTS", ROOT / "artifacts"))


def selftest():
    import tempfile
    from tests_synth import make_synthetic_volume, write_nifti
    from datsc.volume import VOL_SHAPE, prepare_volume
    from datsc.register import load_template
    tpl = load_template(ARTIFACTS / "template.npz")
    with tempfile.TemporaryDirectory() as td:
        for i, kw in enumerate(({}, {"levels": 8, "shape": (128, 128, 25), "spacing": (3.9,) * 3},
                                {"shape": (256, 256, 256), "spacing": (2.46,) * 3})):
            vol, sp = make_synthetic_volume(seed=i, **kw)
            r = prepare_volume(write_nifti(Path(td) / f"s{i}.nii.gz", vol, sp), template=tpl)
            assert r["vol"].shape == VOL_SHAPE and np.isfinite(r["scalars"]).all() and np.isfinite(r["stats"]).all()
    print(f"selftest: template {'used' if tpl is not None else 'absent'}")
    if (ARTIFACTS / "manifest.json").exists():
        m = infer.load_manifest(ARTIFACTS / "manifest.json")
        for a in m["arms"]:
            for ck in a["checkpoints"]:
                assert (ARTIFACTS / Path(ck).name).exists(), ck
    print("SELFTEST OK")


def main():
    fmt = pd.read_csv(config.RUNTIME_SUBMISSION_FORMAT, dtype={"uid": str})
    uids = fmt["uid"].astype(str).tolist()
    man = infer.load_manifest(ARTIFACTS / "manifest.json") if (ARTIFACTS / "manifest.json").exists() else \
        {"pi": config.DEFAULT_PRIOR, "T": 1.0, "lam": 0.0, "weights": {}, "arms": []}
    pi = float(man.get("pi", config.DEFAULT_PRIOR))
    infer.write_submission(uids, np.full(len(uids), pi), config.RUNTIME_OUTPUT_CSV)
    print(f"stage0 prior written for {len(uids)} rows at {time.time()-T0:.0f}s", flush=True)

    n_workers = max(1, min(20, (os.cpu_count() or 4) - 2))
    tpl = ARTIFACTS / "template.npz"
    print(f"template: {'yes' if tpl.exists() else 'NO (CoM frame)'}; workers {n_workers}", flush=True)
    vols, sc, st, ok = infer.preprocess_all(uids, config.RUNTIME_NIFTI_DIR, n_workers, deadline_s=infer.LADDER_STOP_S, t0=T0,
                                            template_path=tpl if tpl.exists() else None)
    arm_probs = {}
    typ = None
    feat_cfg = next((a for a in man["arms"] if a.get("arm") == "feat"), None)
    if man.get("typ") and feat_cfg is not None:
        try:
            d = infer.typicality_all(feat_cfg, ARTIFACTS, vols, st, ok)
            typ = {**man["typ"], "d": d}
            print(f"typicality: median {np.nanmedian(d):.2f}, {int(np.nansum(d > float(man['typ']['d0'])))} scans above d0={man['typ']['d0']:.2f}", flush=True)
        except Exception as e:  # noqa: BLE001
            print(f"typicality failed: {type(e).__name__}", flush=True)
    for a in man["arms"]:
        if time.time() - T0 > infer.LADDER_STOP_S:
            print("ladder: skipping remaining arms"); break
        try:
            arm_probs[a["name"]] = infer.run_arm(a, ARTIFACTS, vols, sc, st, ok, T0)
        except Exception as e:  # noqa: BLE001
            print(f"arm {a['name']} failed: {type(e).__name__}", flush=True); arm_probs[a["name"]] = None
        p = infer.combine(arm_probs, man["weights"], float(man["T"]), float(man["lam"]), pi, clip=man.get("clip"), hedge=man.get("hedge"), typ=typ)
        if p is not None:
            infer.write_submission(uids, np.where(ok & np.isfinite(p), p, pi), config.RUNTIME_OUTPUT_CSV)
            print(f"stage written after arm {a['name']} at {time.time()-T0:.0f}s", flush=True)
    print(f"done in {time.time()-T0:.0f}s; failed volumes: {int((~ok).sum())}", flush=True)


if __name__ == "__main__":
    if "--selftest" in sys.argv:
        selftest()
    else:
        main()
        sys.stdout.flush()
        os._exit(0)


In [ ]:
%%writefile tests_synth.py
"""Synthetic DaT-like volumes (no pytest deps) - used by tests/conftest.py and by main.py --selftest."""
import numpy as np
import nibabel as nib


def make_synthetic_volume(shape=(96, 96, 60), spacing=(2.46, 2.46, 2.46), levels=None,
                          seed=0, striatum_amp=4.0, asym=0.0, scale=1000.0):
    """Synthetic DaT-like head: ellipsoid background + two bright striatal blobs.

    asym in [0,1]: fraction by which the RIGHT blob is dimmed (simulates pathology).
    levels: if set, quantise to this many integer grey levels (ultra-low-count regime).
    Returns (array[x,y,z] float32, spacing).
    """
    rng = np.random.default_rng(seed)
    x, y, z = np.meshgrid(*[np.arange(s) for s in shape], indexing="ij")
    c = np.array(shape) / 2.0
    r = np.array(shape) / 2.6
    ell = (((x - c[0]) / r[0]) ** 2 + ((y - c[1]) / r[1]) ** 2 + ((z - c[2]) / r[2]) ** 2) <= 1.0
    vol = np.where(ell, 1.0, 0.0) + rng.normal(0, 0.08, size=shape)
    sig_vox = 6.0 / np.array(spacing)  # 6 mm blobs
    off_vox = 9.0 / spacing[0]         # blobs 9 mm left/right of midline
    for side, amp in ((-1, striatum_amp), (+1, striatum_amp * (1.0 - asym))):
        cx = c[0] + side * off_vox
        blob = np.exp(-(((x - cx) / sig_vox[0]) ** 2 + ((y - c[1]) / sig_vox[1]) ** 2
                        + ((z - c[2]) / sig_vox[2]) ** 2) / 2.0)
        vol += amp * blob
    vol = np.clip(vol, 0, None)
    if levels is not None:
        vol = np.round(vol / vol.max() * (levels - 1))
    else:
        vol = vol * scale
    return vol.astype(np.float32), tuple(float(s) for s in spacing)


def write_nifti(path, vol, spacing):
    affine = np.diag(list(spacing) + [1.0])
    nib.save(nib.Nifti1Image(vol, affine), str(path))
    return path


In [ ]:
!touch scripts/__init__.py
!python main_v3.py --selftest

In [ ]:
# cache_v4 (registered, reference-normalised 2 mm boxes) + template + ImageNet weights come from the uploaded dataset
import os, shutil, glob, numpy as np
src = os.path.dirname(next(iter(glob.glob(CACHE_DS + '/**/vols.npy', recursive=True)), ''))
assert src, 'cache dataset not mounted -- attach dat-parkinsons-cache-v4'
os.makedirs('cache_v4', exist_ok=True); os.makedirs('pretrained', exist_ok=True); os.makedirs('artifacts', exist_ok=True)
for f in os.listdir(src):
    dst = os.path.join('cache_v4', f)
    if not os.path.exists(dst):
        if f.endswith('.npy') and os.path.getsize(os.path.join(src, f)) > 50_000_000:
            os.symlink(os.path.join(src, f), dst)     # big arrays stay on the read-only mount (memmap)
        else:
            shutil.copy(os.path.join(src, f), dst)
for f in glob.glob(CACHE_DS + '/**/*.pt', recursive=True):
    if not os.path.exists('pretrained/' + os.path.basename(f)): shutil.copy(f, 'pretrained/')
tpl = next(iter(glob.glob(CACHE_DS + '/**/template.npz', recursive=True)))
shutil.copy(tpl, 'artifacts/template.npz'); shutil.copy(tpl, 'artifacts_v3/template.npz')
for f in glob.glob('/kaggle/input/datasets/harshdipsaha/dat-run3-checkpoints*/template.npz'):   # run 5: template with hist landmarks
    shutil.copy(f, 'artifacts/template.npz'); shutil.copy(f, 'artifacts_v3/template.npz'); print('template from the checkpoint dataset (landmarks):', 'hist' in np.load(f).files)
for pat in ('lr_v3.json', 'oof_lrarm.csv', 'oof_lrarm_site.csv', 'lrarm.json', 'feat_v4.npz', 'oof_featarm.csv', 'oof_featarm_site.csv', 'featarm.json'):      # registered-ROI LR arm (fitted locally)
    for f in glob.glob(CACHE_DS + '/**/' + pat, recursive=True): shutil.copy(f, 'artifacts_v3/')
print('cache_v4:', sorted(os.listdir('cache_v4'))); print('pretrained:', sorted(os.listdir('pretrained')))
# RESUME: if a previous version's output (or any dataset) mounted under /kaggle/input carries artifacts_v3/, copy it back so the
# runners skip finished jobs (chained commits = persistence without credentials)
os.makedirs('artifacts_v3', exist_ok=True); n_res = 0
for src_art in glob.glob('/kaggle/input/**/artifacts_v3', recursive=True) + glob.glob('/kaggle/input/**/out', recursive=True) + glob.glob('/kaggle/input/datasets/harshdipsaha/dat-run3-checkpoints*'):
    for f in glob.glob(src_art + '/*'):
        if os.path.isfile(f) and f.endswith(('.pt', '.json', '.csv', '.npz')) and not os.path.exists('artifacts_v3/' + os.path.basename(f)):
            shutil.copy(f, 'artifacts_v3/'); n_res += 1
print('resumed artefacts copied from /kaggle/input:', n_res)
if n_res == 0:      # fallback: pull the private checkpoint dataset with the API (needs the KAGGLE_USERNAME/KAGGLE_KEY secrets + internet)
    try:
        from kaggle_secrets import UserSecretsClient
        _u = UserSecretsClient(); os.environ['KAGGLE_USERNAME'] = _u.get_secret('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = _u.get_secret('KAGGLE_KEY')
        import subprocess as _sp
        _sp.run('pip -q install -U kaggle', shell=True, capture_output=True)
        _r = _sp.run('kaggle datasets download harshdipsaha/dat-run3-checkpoints -p /kaggle/working/ckpt_ds --unzip --force', shell=True, capture_output=True, text=True)
        print('checkpoint dataset download:', (_r.stdout + _r.stderr).strip()[-300:])
        for f in glob.glob('/kaggle/working/ckpt_ds/**/*', recursive=True):
            if os.path.isfile(f) and f.endswith(('.pt', '.json', '.csv', '.npz')) and not os.path.exists('artifacts_v3/' + os.path.basename(f)):
                shutil.copy(f, 'artifacts_v3/'); n_res += 1
        print('resumed artefacts from the checkpoint dataset download:', n_res)
    except Exception as _e:
        print('checkpoint dataset download skipped:', type(_e).__name__, str(_e)[:200])
!ls -la cache_v4 | head -20

In [ ]:
# timing check: 2 epochs of each arm on fold 0 (expect ~20-40 s/epoch on a T4)
!CUDA_VISIBLE_DEVICES=0 python scripts/train_v3.py --arm 3d --fold 0 --seed 99 --epochs 2 --out artifacts_timing --cache cache_v4 --size 64 64 48 --widths 24 48 96 192 2>&1 | grep -E "params|ep0|done"
!CUDA_VISIBLE_DEVICES=1 python scripts/train_v3.py --arm 2d --backbone efficientnet_b0 --weights pretrained/efficientnet_b0_in1k.pt --fold 0 --seed 99 --epochs 2 --out artifacts_timing --cache cache_v4 --batch-size 32 --z-center 33.5 2>&1 | grep -E "params|ep0|done"
!rm -rf artifacts_timing

In [ ]:
# soft-target table (run 8): Master v2's NESTED out-of-fold probability + loss per training scan, embedded (no scan data), -> cache_v4/soft_master_v2.csv
import base64, gzip, os, pandas as pd
_b = '...'
os.makedirs('cache_v4', exist_ok=True); open('cache_v4/soft_master_v2.csv', 'wb').write(gzip.decompress(base64.b64decode(_b)))
_t = pd.read_csv('cache_v4/soft_master_v2.csv', dtype={'uid': str}); print(len(_t), 'scans; nested loss', round(_t.loss.mean(), 4), '; top-100 share', round(_t.loss.sort_values(ascending=False).iloc[:100].sum() / _t.loss.sum(), 3))


In [ ]:
# === RESUMABLE 2-GPU JOB RUNNER (run 8) === (skips jobs whose checkpoint already exists in artifacts_v3/)
import subprocess, time, os, glob, json
LABEL = '(run 8)'
JOBS = [
 {
  "name": "r3d_hemisoft",
  "seeds": [
   0,
   1,
   2,
   3,
   4
  ],
  "folds_csv": "folds.csv",
  "tag": "r3d_hemisoft_f{f}_s{s}",
  "args": "--arm 3d --size 48 48 32 --crop3d 48 48 32 --center3d {XC} {YC} {ZC} --widths 24 48 96 192 --epochs 50 --batch-size 16 --lr 5e-4 --mixup 0.2 --ema 0.995 --arch hemi --soft-csv cache_v4/soft_master_v2.csv --soft-lam 0.4"
 },
 {
  "name": "r3d_crophistsoft",
  "seeds": [
   0,
   1,
   2
  ],
  "folds_csv": "folds.csv",
  "tag": "r3d_crophistsoft_f{f}_s{s}",
  "args": "--arm 3d --size 48 48 32 --crop3d 48 48 32 --center3d {XC} {YC} {ZC} --widths 24 48 96 192 --epochs 50 --batch-size 16 --lr 5e-4 --mixup 0.2 --ema 0.995 --norm hist --soft-csv cache_v4/soft_master_v2.csv --soft-lam 0.4"
 }
]
import numpy as np
_sc = np.load('artifacts/template.npz')['striatal_center']; XC, YC, ZC = float(_sc[0]), float(_sc[1]), float(_sc[2]); print('template striatal x/y/z (vox):', XC, YC, ZC)
def expand(jobs):
    out = []
    for j in jobs:
        for s in j['seeds']:
            for f in j.get('folds', range(5)):
                tag = j.get('tag', '').format(f=f if f >= 0 else 'all', s=s)
                args = (j['args'].replace('{ZC}', f"{ZC:.1f}").replace('{YC}', f"{YC:.1f}").replace('{XC}', f"{XC:.1f}")
                        + f" --folds-csv {j['folds_csv']}" + (f" --tag {tag}" if tag else ''))
                out.append((j['name'], f, s, args, tag))
    return out
def tag_for(name, f, s, args, tag=''):
    if tag:
        return tag
    if '--arm 3d' in args:
        size = 'x'.join(args.split('--size')[1].split()[:3]); tag = f"r3d_{size}_f{f if f >= 0 else 'all'}_s{s}"
    else:
        bb = args.split('--backbone')[1].split()[0]; tag = f"r2d_{bb}{'_cor' if '--plane coronal' in args else ''}_f{f if f >= 0 else 'all'}_s{s}"
    return tag + ('_site' if 'folds_site' in args else '')
# --- persistence: push artifacts_v3 to the private dataset harshdipsaha/dat-run3-checkpoints after every 4 finished jobs (needs Kaggle Secrets) ---
import threading, shutil as _sh
def _creds():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient(); return c.get_secret('KAGGLE_USERNAME'), c.get_secret('KAGGLE_KEY')
    except Exception as e:
        return None, None
_KU, _KK = _creds(); _UPLOAD_OK = bool(_KU and _KK)
print('checkpoint upload:', 'enabled' if _UPLOAD_OK else 'disabled (no KAGGLE_USERNAME/KAGGLE_KEY secrets)')
_up_lock = threading.Lock()
def push_checkpoints(note):
    if not _UPLOAD_OK or not _up_lock.acquire(blocking=False):
        return
    try:
        d = '/kaggle/working/ckpt_upload'; os.makedirs(d, exist_ok=True)
        for f in glob.glob('artifacts_v3/*'):
            if os.path.isfile(f) and not os.path.exists(os.path.join(d, os.path.basename(f))): _sh.copy(f, d)
        json.dump({'title': 'dat-run3-checkpoints', 'id': 'harshdipsaha/dat-run3-checkpoints', 'licenses': [{'name': 'other'}]}, open(os.path.join(d, 'dataset-metadata.json'), 'w'))
        env = {**os.environ, 'KAGGLE_USERNAME': _KU, 'KAGGLE_KEY': _KK}
        subprocess.run(['python', '-m', 'pip', 'install', '-q', '-U', 'kaggle'], capture_output=True, timeout=600)   # fine-grained tokens need client >= 2.x
        r = subprocess.run(['kaggle', 'datasets', 'version', '-p', d, '-m', note, '-q', '--dir-mode', 'skip'], env=env, capture_output=True, text=True, timeout=1800)
        print(f"[upload] {note}: rc={r.returncode} {(r.stdout or r.stderr).strip()[-160:]}", flush=True)
    except Exception as e:
        print('[upload] failed:', type(e).__name__, str(e)[:120], flush=True)
    finally:
        _up_lock.release()
queue = [(n, f, s, a, t) for n, f, s, a, t in expand(JOBS) if not os.path.exists(f"artifacts_v3/{tag_for(n,f,s,a,t)}.pt")]
print(f"{len(queue)} jobs to run, {len(expand(JOBS)) - len(queue)} already done")
running = {}; t0 = time.time(); done = []
while queue or running:
    for gpu in range(max(1, __import__('torch').cuda.device_count())):
        if gpu not in running and queue:
            n, f, s, a, t = queue.pop(0); tag = tag_for(n, f, s, a, t)
            log = open(f"log_{tag}.txt", "w")
            cmd = f"python scripts/train_v3.py --fold {f} --seed {s} --out artifacts_v3 --cache cache_v4 {a}"
            running[gpu] = (tag, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}))
            print(f"[gpu{gpu}] start {tag} at {time.time()-t0:.0f}s", flush=True)
    for gpu in list(running):
        tag, p = running[gpu]
        if p.poll() is not None:
            done.append((tag, p.returncode)); del running[gpu]
            last = subprocess.run(f"tail -n 1 log_{tag}.txt", shell=True, capture_output=True, text=True).stdout.strip()
            print(f"[gpu{gpu}] {tag} exit={p.returncode} at {time.time()-t0:.0f}s :: {last[-120:]}", flush=True)
            if len(done) % 4 == 0:
                threading.Thread(target=push_checkpoints, args=(f"{LABEL} {len(done)} jobs done",), daemon=True).start()
    time.sleep(10)
print(f"all done in {(time.time()-t0)/60:.0f} min; failures: {[t for t,r in done if r]}")
push_checkpoints(f"{LABEL} final ({len(done)} jobs)")

In [ ]:
# re-OOF every CV checkpoint with inference-time TTA (2D: z-shift -1/0/+1 x flip; 3D: flip) so the stack is fitted on what main.py produces
try:
    # re-OOF every CV checkpoint with inference-time TTA (2D: z-shift -1/0/+1 x flip; 3D: flip) so the stack is fitted on what main.py produces
    !python scripts/reoof_v3.py --art artifacts_v3 --cache cache_v4 --tta-z -1 0 1 2>&1 | grep -v Warning
except Exception as _e:
    import traceback; traceback.print_exc(); print('CELL FAILED (kept going):', _e)

In [ ]:
# super-stack: score every arm family of runs 3-8 present in artifacts_v3/, keep the 10 best by OOF log loss, nested combiner
try:
    # super-stack: score every arm family of runs 3-8 present in artifacts_v3/, keep the 10 best by OOF log loss, nested combiner
    import glob, json, os, subprocess, pandas as pd, numpy as np
    from sklearn.metrics import log_loss, roc_auc_score
    ARMS = [('a3d_hemisoft', 'r3d_hemisoft_f?_s?', 'r3d_hemisoft_f?_s?_site'), ('a3d_hemimw', 'r3d_hemimw_f?_s?', None), ('a3d_crophistsoft', 'r3d_crophistsoft_f?_s?', None), ('a3d_hemi', 'r3d_hemi48_f?_s?', 'r3d_hemi48_f?_s?_site'), ('a3d_hemiinv', 'r3d_hemiinv_f?_s?', None), ('a3d_cropboot', 'r3d_cropboot_f?_s?', None), ('a3d_crop', 'r3d_crop48x48x32_f?_s?', 'r3d_crop48x48x32_f?_s?_site'), ('a3d_cropinv', 'r3d_cropinv_f?_s?', 'r3d_cropinv_f?_s?_site'), ('a3d_crophist', 'r3d_crophist_f?_s?', None), ('a3d_crop80', 'r3d_crop48e80_f?_s?', None), ('a3d_se48', 'r3d_se48_f?_s?', None), ('a3d_crop64', 'r3d_crop64e60_f?_s?', None), ('a3d_ssl', 'r3d_ssl_f?_s?', 'r3d_ssl_f?_s?_site'), ('a2d_v2s', 'r2d_efficientnet_v2_s_f?_s?', None), ('a2d_cnx', 'r2d_convnext_tiny_f?_s?', None), ('a2d_cnx_cor', 'r2d_convnext_tiny_cor_f?_s?', None), ('a2d_cor', 'r2d_efficientnet_b0_cor_f?_s?', None), ('a2d_b0', 'r2d_efficientnet_b0_f?_s?', 'r2d_efficientnet_b0_f?_s?_site'), ('a2t_cnx', 'r2t_convnext_tiny_f?_s?', None), ('a2t_cnx_hist', 'r2t_convnext_tiny_hist_f?_s?', None), ('a2t_v2s', 'r2t_efficientnet_v2_s_f?_s?', None), ('featc', 'featcarm', 'featcarm_site'), ('feat', 'featarm', 'featarm_site'), ('lr', 'lrarm', 'lrarm_site')]; EXTRAS = [('a3d_hemisoft', 'r3d_hemisoft_fall_s*'), ('a3d_hemi', 'r3d_hemi48_fall_s*'), ('a3d_crop', 'r3d_crop48x48x32_fall_s*'), ('a3d_crop80', 'r3d_crop48e80_fall_s*'), ('a3d_ssl', 'r3d_ssl_fall_s*'), ('a2d_v2s', 'r2d_efficientnet_v2_s_fall_s*'), ('a2d_b0', 'r2d_efficientnet_b0_fall_s*')]
    rows = [{'run': os.path.basename(f)[:-5], **{k: json.load(open(f)).get(k) for k in ('val_logloss', 'val_auroc')}} for f in sorted(glob.glob('artifacts_v3/r*_f*_s*.json'))]
    df = pd.DataFrame(rows); df['arm'] = df.run.str.replace(r'_f(\\d|all)_s\\d(_site)?$', '', regex=True); print(df.groupby('arm')[['val_logloss', 'val_auroc']].agg(['mean', 'count']).round(4))
    scored = []
    for n, p, q in ARMS:
        fs = glob.glob(f'artifacts_v3/oof_{p}.csv')
        if not fs: continue
        o = pd.concat([pd.read_csv(f, dtype={'uid': str}) for f in fs]).groupby('uid').agg(y=('y', 'first'), p=('p', 'mean'))
        if len(o) < 1300: print('skip (incomplete OOF)', n, len(o)); continue
        scored.append((log_loss(o.y, np.clip(o.p, 1e-6, 1 - 1e-6)), n, p, q, len(fs)))
    scored.sort(); print('arms by OOF log loss:'); [print(f'  {ll:.4f} {n:<12} ckpts={k}') for ll, n, p, q, k in scored]
    keep = scored[:10]
    cmd = 'python scripts/fit_stack_v3.py --out artifacts_v3 ' + ' '.join(f'--arm {n} "{p}"' for _, n, p, q, _ in keep)
    cmd += ' ' + ' '.join(f'--site-arm {n} "{q}"' for _, n, p, q, _ in keep if q and glob.glob(f'artifacts_v3/oof_{q}.csv'))
    cmd += ' ' + ' '.join(f'--extra {n} "{g}"' for n, g in EXTRAS if glob.glob(f'artifacts_v3/{g}.pt') and n in [k[1] for k in keep]) + ' --lam 0.02 --t-floor 0.9 --nested --typ --grid 8 --cache cache_v4'
    print(cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print(r.stdout[-5000:]); print(r.stderr[-800:])
    !python scripts/pack_v3.py --artifacts artifacts_v3 --out submission_v3.zip
    !cp submission_v3.zip artifacts_v3/manifest.json artifacts_v3/oof_stack.csv out/ && cp artifacts_v3/*.json artifacts_v3/oof_*.csv cache_v4/folds*.csv cache_v4/sites.csv out/ 2>/dev/null; du -sh out submission_v3.zip
except Exception as _e:
    import traceback; traceback.print_exc(); print('CELL FAILED (kept going):', _e)

In [ ]:
# GATE (run 8): nested estimate (equal weights, T refit per fold with floor 0.9, lam 0.02) of the Master v2 pillars vs pillars + each run-8 arm
try:
    # GATE (run 8): nested estimate (equal weights, T refit per fold with floor 0.9, lam 0.02) of the Master v2 pillars vs pillars + each run-8 arm
    import glob, numpy as np, pandas as pd
    from scipy.optimize import minimize_scalar
    PIL = {'a3d_hemi': 'r3d_hemi48_f?_s?', 'a3d_ssl': 'r3d_ssl_f?_s?', 'featc': 'featcarm', 'a2d_cnx_cor': 'r2d_convnext_tiny_cor_f?_s?', 'a2d_v2s': 'r2d_efficientnet_v2_s_f?_s[01]'}
    NEW = {'a3d_hemisoft': 'r3d_hemisoft_f?_s?', 'a3d_hemimw': 'r3d_hemimw_f?_s?', 'a3d_crophistsoft': 'r3d_crophistsoft_f?_s?'}
    def load(p):
        fs = [f for f in glob.glob(f'artifacts_v3/oof_{p}.csv') if '_site' not in f and '_fall' not in f]
        if not fs: return None
        o = pd.concat([pd.read_csv(f, dtype={'uid': str}) for f in fs]).groupby('uid').agg(y=('y', 'first'), p=('p', 'mean')); return o if len(o) >= 1300 else None
    oofs = {n: o for n, o in ((n, load(p)) for n, p in {**PIL, **NEW}.items()) if o is not None}
    uids = sorted(set.intersection(*[set(o.index) for o in oofs.values()])); y = next(iter(oofs.values())).loc[uids, 'y'].values.astype(float)
    folds = pd.read_csv('cache_v4/folds.csv', dtype={'uid': str}).set_index('uid').loc[uids, 'fold'].values
    lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))); sg = lambda z: 1 / (1 + np.exp(-z))
    ll = lambda yy, pp: float(-np.mean(yy * np.log(np.clip(pp, 1e-7, 1)) + (1 - yy) * np.log(np.clip(1 - pp, 1e-7, 1))))
    def nested(names):
        Z = np.stack([lg(oofs[n].loc[uids, 'p'].values) for n in names]).mean(0); pn = np.zeros(len(y))
        for k in np.unique(folds):
            tr, va = folds != k, folds == k
            T = max(minimize_scalar(lambda t: ll(y[tr], sg(Z[tr] / t)), bounds=(0.3, 5), method='bounded').x, 0.9)
            pn[va] = 0.98 * sg(Z[va] / T) + 0.02 * 0.548
        return ll(y, pn)
    pil = [n for n in PIL if n in oofs]; base = nested(pil); print(f'pillars ({len(pil)} arms): nested {base:.4f}')
    for n in NEW:
        if n in oofs: print(f'  + {n:<16} nested {nested(pil + [n]):.4f}  ({nested(pil + [n]) - base:+.4f});  arm alone {ll(y, oofs[n].loc[uids, "p"].values):.4f}')
    new = [n for n in NEW if n in oofs]
    if len(new) > 1: print(f'  + all run-8 arms   nested {nested(pil + new):.4f}  ({nested(pil + new) - base:+.4f})')
    print('GATE: an arm is in if the gain is <= -0.003 (laptop: python scripts/build_master_v3.py --src data/run8/artifacts_v3 --src data/master_pool/artifacts_v3)')
    
except Exception as _e:
    import traceback; traceback.print_exc(); print('CELL FAILED (kept going):', _e)

In [ ]:
# === END-TO-END CHECK of submission_v3.zip exactly as the DrivenData container runs it (+ smoke-set score per arm) ===
try:
    # === END-TO-END CHECK of submission_v3.zip exactly as the DrivenData container runs it (+ smoke-set score per arm) ===
    import json, os, re, shutil, subprocess, sys, time, zipfile
    from pathlib import Path
    import numpy as np, pandas as pd
    from sklearn.metrics import log_loss
    
    DS = Path(RAW)
    def build_data_root(dst, nifti_paths, fmt_df):
        dst = Path(dst); shutil.rmtree(dst, ignore_errors=True); (dst / "niftis").mkdir(parents=True)
        for p in nifti_paths:
            os.symlink(p, dst / "niftis" / p.name)
        fmt_df.to_csv(dst / "submission_format.csv", index=False); return dst
    def run_main(zip_path, data_root, tag):
        SIM = Path(f"/kaggle/working/code_execution_{tag}"); shutil.rmtree(SIM, ignore_errors=True); SIM.mkdir()
        with zipfile.ZipFile(zip_path) as z: z.extractall(SIM)
        assert (SIM / "main.py").exists()
        env = {**os.environ, "DATSC_DATA_ROOT": str(data_root), "DATSC_OUTPUT_CSV": "submission.csv"}
        t0 = time.time(); r = subprocess.run(["python", "main.py"], cwd=SIM, env=env, capture_output=True, text=True); wall = time.time() - t0
        print(r.stdout[-2500:]); print(r.stderr[-1500:]); print(f"exit={r.returncode} wall={wall:.0f}s"); assert r.returncode == 0
        return SIM, wall, r.stdout
    smoke_fmt = next(p for p in DS.rglob("submission_format.csv")); SRC = smoke_fmt.parent
    niftis = sorted(p for p in SRC.rglob("*") if p.name.endswith((".nii", ".nii.gz")))
    fmt = pd.read_csv(smoke_fmt, dtype={"uid": str})
    ROOT_S = build_data_root("/kaggle/working/sim_data_smoke", niftis, fmt)
    SIM, wall, out = run_main("submission_v3.zip", ROOT_S, "smoke_v3")
    used = re.findall(r"arm (\S+): checkpoints used (\d+)/(\d+)", out)
    assert used and all(int(u) == int(m) for _, u, m in used), f"ARM CHECKPOINTS NOT ALL USED: {used}"
    print("all arms used every checkpoint:", [(a, u) for a, u, _ in used])
    sub = pd.read_csv(SIM / "submission.csv", dtype={"uid": str})
    assert list(sub.columns) == ["uid", "is_pathologic"] and list(sub.uid) == list(fmt.uid) and sub.is_pathologic.between(0, 1).all()
    lab = pd.read_csv(next(SRC.rglob("test_labels.csv")), dtype={"uid": str}).set_index("uid").loc[sub.uid]
    y = lab.is_pathologic.values.astype(float); p = sub.is_pathologic.values
    print(f"FORMAT OK {sub.shape} | smoke-test log loss = {log_loss(y, p):.4f} (prior-only {log_loss(y, np.full(len(y), 0.548)):.4f}) | acc@0.5 = {((p>0.5)==(y>0.5)).mean():.3f} | {wall:.0f}s of 360s")
    # --- the smoke factor: log loss of EVERY arm on the 20 labelled smoke scans (out-of-distribution check only; nothing is fitted on it) ---
    # ladder so far: run 2 smoke 0.2567 -> public 0.3391, run 3 0.2397 -> 0.2817, run 4a 0.2176 -> 0.2667
    sys.path.insert(0, str(SIM))
    from datsc import infer_v3 as infer
    man = json.loads((SIM / "artifacts" / "manifest.json").read_text())
    uids = list(fmt.uid.astype(str)); t0 = time.time()
    vols, sc, st, ok = infer.preprocess_all(uids, ROOT_S / "niftis", max(1, min(20, (os.cpu_count() or 4) - 2)), deadline_s=infer.LADDER_STOP_S, t0=t0, template_path=SIM / "artifacts" / "template.npz")
    arm_probs, rows = {}, []
    for a in man["arms"]:
        pa = infer.run_arm(a, SIM / "artifacts", vols, sc, st, ok, t0)
        arm_probs[a["name"]] = pa
        if pa is not None and np.isfinite(pa).all():
            rows.append((a["name"], man["weights"].get(a["name"], 0.0), len(a["checkpoints"]), log_loss(y, np.clip(pa, 1e-6, 1 - 1e-6))))
    pst = infer.combine(arm_probs, man["weights"], float(man["T"]), float(man["lam"]), float(man.get("pi", 0.548)), clip=man.get("clip"), hedge=man.get("hedge"))
    print("smoke-set log loss per arm (20 scans; OOD check, not a fitting target):")
    for n, w, k, l in sorted(rows, key=lambda r: r[3]):
        print(f"  {n:<12} weight {w:.3f}  ckpts {k:3d}  smoke logloss {l:.4f}")
    print(f"  {'STACK':<12} smoke logloss {log_loss(y, np.clip(pst, 1e-6, 1 - 1e-6)):.4f}   (main.py above: {log_loss(y, p):.4f})")
    json.dump({"per_arm": [{"arm": n, "weight": w, "ckpts": k, "smoke_logloss": l} for n, w, k, l in rows], "stack": float(log_loss(y, np.clip(pst, 1e-6, 1 - 1e-6)))}, open("out/smoke_per_arm.json", "w"), indent=1)
    # 3-hour budget: 150 training volumes on this box, extrapolated to 2,000 on the 24-vCPU/A100 box
    from datsc.config import find_data_dir, find_nifti_path
    d = find_data_dir(); labels = pd.read_csv(d.labels_csv, dtype={"uid": str}).sample(n=150, random_state=1)
    ROOT_T = build_data_root("/kaggle/working/sim_data_timing", [find_nifti_path(d.niftis_dir, u) for u in labels.uid], pd.DataFrame({"uid": labels.uid.values, "is_pathologic": 0.5}))
    SIM_T, wall_t, out_t = run_main("submission_v3.zip", ROOT_T, "timing_v3")
    m = re.search(r"preprocessed (\d+)/(\d+) volumes in (\d+)s", out_t); prep = float(m.group(3))
    nw = max(1, min(20, (os.cpu_count() or 4) - 2)); cpu_s = prep * nw / 150
    print(f"EXTRAPOLATED 2,000 volumes: prep ~{cpu_s*2000/20/60:.0f} min + models ~{(wall_t-prep)*2000/150/60:.0f} min (T4->A100 is faster) | limit 180 min")
    sub_t = pd.read_csv(SIM_T / "submission.csv", dtype={"uid": str}); lab_t = labels.set_index("uid").loc[sub_t.uid]
    print(f"in-sample log loss on 150 training volumes = {log_loss(lab_t.is_pathologic.values.astype(float), sub_t.is_pathologic.values):.4f}")
    
except Exception as _e:
    import traceback; traceback.print_exc(); print('CELL FAILED (kept going):', _e)

In [ ]:
# === SPLIT submission_v3.zip into <95 MB parts for browser download (Kaggle's in-editor download caps ~100 MB) ===
try:
    # === SPLIT submission_v3.zip into <95 MB parts for browser download (Kaggle's in-editor download caps ~100 MB) ===
    import os, subprocess, hashlib
    os.makedirs('parts', exist_ok=True); subprocess.run('rm -f parts/*', shell=True)
    subprocess.run('split -b 95m -d -a 2 submission_v3.zip parts/submission_v3.zip.part', shell=True, check=True)
    md5 = hashlib.md5(open('submission_v3.zip', 'rb').read()).hexdigest()
    open('parts/MD5', 'w').write(md5 + '  submission_v3.zip\n')
    print('md5', md5, 'size', os.path.getsize('submission_v3.zip'))
    print(sorted(os.listdir('parts')))
    
except Exception as _e:
    import traceback; traceback.print_exc(); print('CELL FAILED (kept going):', _e)